# ARC-AGI-3 — arcnav submission (SungsuHyun)

Our solver: a tool-using LLM agent (Qwen3.6-27B-FP8 served by vLLM inside the notebook) with a python sandbox, a navigation helper (avatar/move learning, coarse map, BFS routing, targets, action gauge), solver synthesis (`propose_solver`: the model writes a persistent `solve()` that the harness verifies and runs without further model calls) and failure policies. Sources are embedded from `arcnav/` by `scripts/build_arcnav_notebook.py`; do not edit cells here.

Commit = 2-game offline smoke (15 min); the competition rerun plays every gateway game.

In [ ]:
import json, os, subprocess, sys, time
T0 = time.time()
RERUN = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))
WORK = '/kaggle/working'
print('competition rerun:', RERUN)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--find-links', '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels', 'arc-agi'], check=True)
# arcnav package (embedded sources)
SOURCES = json.loads("{\"__init__.py\": \"\\\"\\\"\\\"arcnav \\u2014 our ARC-AGI-3 solver: a tool-using LLM harness built around\\nnavigation helpers, solver synthesis and programmatic safety policies.\\n\\nModules: frame (observation encoding), nav (navigation helper), sandbox\\n(python tool), llm (OpenAI-compatible tool-calling client), prompts, solver\\n(persistent solver policies), agent (per-game loop), runner (local/Kaggle play).\\n\\\"\\\"\\\"\\n__version__ = \\\"0.1.0\\\"\\n\", \"agent.py\": \"\\\"\\\"\\\"Per-game loop: environment stepping, model turns, python tool, solver auto-run.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport time\\nfrom pathlib import Path\\nfrom typing import Any, Optional\\n\\nfrom arcengine import GameAction, GameState\\n\\nfrom . import solver as solver_policy\\nfrom .frame import Frame, masked_ascii, summarize_diff, grid_to_png_b64\\nfrom .llm import ChatClient, ContextLengthError\\nfrom .nav import NavHelper\\nfrom . import rules as rule_induction\\nfrom .autopilot import run_two_body, run_reach\\nfrom . import goals as goal_inference\\nfrom .prompts import PROPOSE_TOOL, PYTHON_TOOL, system_prompt, turn_header\\nfrom .sandbox import Sandbox\\n\\nMODEL_TO_ENGINE = {\\\"UP\\\": \\\"ACTION1\\\", \\\"DOWN\\\": \\\"ACTION2\\\", \\\"LEFT\\\": \\\"ACTION3\\\", \\\"RIGHT\\\": \\\"ACTION4\\\", \\\"SPACE\\\": \\\"ACTION5\\\", \\\"MOUSE\\\": \\\"ACTION6\\\", \\\"ACTION7\\\": \\\"ACTION7\\\"}\\nENGINE_TO_MODEL = {v: k for k, v in MODEL_TO_ENGINE.items()}\\nMAX_ACTIONS_PER_CALL = 24   # blind 50-action batches walked straight into game over on s5i5\\nPROBE_SWEEP_AFTER_TURNS = 8   # iter4: after this many model turns on a level without completing it, the harness probes every untried action once\\nMAX_ACTIONS_PER_TOOL_RUN = 30   # also caps loops of single-action calls inside one python tool run\\nPROPOSAL_ATTEMPTS = 0   # >0: after a level completion, action() is blocked and propose_solver is forced via tool_choice for this many turns.\\n                        # Set E (forced, 3 turns): 0.75, level>=2 0/4 \\u2014 forced solvers burned 50\\u201380 actions per level; disabled by default.\\n\\n\\nclass _T:  # transition view for the host-side NavHelper\\n    def __init__(self, action, before_frame, after_frame):\\n        self.action, self.before_frame, self.after_frame = action, before_frame, after_frame\\n\\n\\nclass GameSession:\\n    def __init__(self, env, game_id: str, client: Optional[ChatClient], *, log_dir: Path, max_minutes: float = 20.0,\\n                 max_actions: int = 3000, max_model_turns: int = 400, keep_full_turns: int = 3, tool_timeout: int = 30,\\n                 context_tokens: int = 32768, verbose: bool = True, deadline: Optional[float] = None, think_first_turns: int = 0,\\n                 tool_choice_required: bool = False, oracle_rules: str = \\\"\\\", image_context: bool = False):\\n        self.env, self.game_id, self.client = env, game_id, client\\n        self.log_dir = Path(log_dir); self.log_dir.mkdir(parents=True, exist_ok=True)\\n        self.max_minutes, self.max_actions, self.max_model_turns = max_minutes, max_actions, max_model_turns\\n        self.keep_full_turns, self.tool_timeout, self.context_tokens, self.verbose = keep_full_turns, tool_timeout, context_tokens, verbose\\n        self.deadline = deadline   # absolute epoch seconds (global run cap), optional\\n        self.image_context = image_context   # attach the current board as an image to the latest user message (VLM)\\n        self.oracle_rules = oracle_rules   # D1 diagnostic: ground-truth rules injected into every turn (never in submissions)\\n        self.tool_choice_required = tool_choice_required   # force a tool call every turn (models with flaky tool formatting)\\n        self.think_first_turns = think_first_turns   # iter3: chain-of-thought ON for the first N model turns of every level\\n        self.level_turn_start = 0\\n        self.transitions: list[dict] = []          # payloads for the sandbox\\n        self.host_transitions: list[_T] = []       # Frame views for the host NavHelper\\n        self.attempt_start_index = 0               # index into host_transitions where the current level attempt began\\n        self.cycle_hits_this_level = 0\\n        self.frame: Optional[Frame] = None\\n        self.level, self.levels_total, self.state = 1, 0, \\\"NOT_PLAYED\\\"\\n        self.valid_actions: list[str] = []\\n        self.actions_used = self.level_actions = self.resets = self.step_no = 0\\n        self.level_action_log: list[int] = []\\n        self.solver: Optional[dict] = None\\n        self.model_turns = self.solver_turns = 0\\n        self.level_just_completed = False\\n        self.last_outcome: list[str] = []\\n        self.notes = \\\"\\\"\\n        self.checklist: dict = {\\\"goal\\\": \\\"\\\", \\\"roles\\\": {}, \\\"plan\\\": \\\"\\\", \\\"tried\\\": []}   # model-owned fields; harness adds facts\\n        self.checklist_level = 1\\n        self.rules_text = \\\"\\\"\\n        self.goal_hypotheses: list[dict] = []\\n        self.autopilot_tries: dict[int, int] = {}   # level -> attempts of the rule-based autopilot\\n        self._run_actions = 0\\n        self.game_overs_this_level = 0\\n        self.rejections_in_row = 0\\n        self.zero_action_turns = 0   # consecutive model turns that executed nothing\\n        self.probe_sweeps: dict[int, str] = {}   # iter4: level -> transition table from the harness probe sweep\\n        self.level_recaps: list[str] = []   # harness-written summaries of how each completed level was won\\n        self.recent_codes: list[str] = []   # repetition guard: identical tool code is not executed twice in a row\\n        self.proposal_required = 0   # >0: action() blocked until propose_solver is called (set after a level completion)\\n        self.messages: list[dict] = []\\n        self.t0 = time.time()\\n        self.transcript = open(self.log_dir / f\\\"{game_id}.log\\\", \\\"a\\\")\\n        self.events = open(self.log_dir / f\\\"{game_id}.jsonl\\\", \\\"a\\\")\\n        self.sandbox = Sandbox(action_handler=self._on_action, solver_handler=self._on_solver, state_provider=self._state, timeout=tool_timeout)\\n\\n    # ------------------------------------------------------------ environment\\n    def _apply(self, raw) -> None:\\n        self.state = str(raw.state).split(\\\".\\\")[-1]\\n        self.levels_total = int(raw.win_levels or 0)\\n        self.level = int(raw.levels_completed) + 1\\n        self.valid_actions = [ENGINE_TO_MODEL.get(f\\\"ACTION{a}\\\", f\\\"ACTION{a}\\\") for a in (raw.available_actions or [])]\\n        grids = [g.tolist() if hasattr(g, \\\"tolist\\\") else g for g in raw.frame]\\n        if grids:\\n            self.frame = Frame(grids[-1], step=self.step_no, level=self.level)\\n\\n    def reset(self) -> None:\\n        raw = self.env.step(GameAction.RESET)\\n        self.resets += 1; self._apply(raw)\\n        self.level_actions = 0\\n\\n    def _step_engine(self, act: dict):\\n        name = act[\\\"action\\\"]\\n        ga = GameAction.from_name(MODEL_TO_ENGINE[name]) if hasattr(GameAction, \\\"from_name\\\") else GameAction[MODEL_TO_ENGINE[name]]\\n        data = {\\\"x\\\": int(act[\\\"col\\\"]), \\\"y\\\": int(act[\\\"row\\\"])} if name == \\\"MOUSE\\\" else {}\\n        ga.set_data(data) if data else None\\n        return self.env.step(ga, data=ga.action_data.model_dump())\\n\\n    def execute(self, actions: list[dict]) -> dict:\\n        \\\"\\\"\\\"Run actions until one completes a level or ends the game. Records transitions.\\\"\\\"\\\"\\n        results, executed, changed_any, level_completed, game_over, stopped = [], 0, False, False, False, None\\n        if len(actions) > MAX_ACTIONS_PER_CALL:\\n            actions = actions[:MAX_ACTIONS_PER_CALL]; stopped = f\\\"only the first {MAX_ACTIONS_PER_CALL} actions of a call are executed; look at the board, then continue\\\"\\n        for act in actions:\\n            if act[\\\"action\\\"] not in self.valid_actions and self.valid_actions:\\n                stopped = f\\\"{act['action']} is not a valid action now ({self.valid_actions})\\\"; break\\n            if act[\\\"action\\\"] == \\\"MOUSE\\\" and (\\\"row\\\" not in act or \\\"col\\\" not in act):\\n                stopped = \\\"MOUSE needs row and col\\\"; break\\n            if act[\\\"action\\\"] == \\\"MOUSE\\\" and not (0 <= int(act[\\\"row\\\"]) <= 63 and 0 <= int(act[\\\"col\\\"]) <= 63):\\n                stopped = f\\\"MOUSE coordinates out of range (row={act['row']}, col={act['col']}; valid 0..63)\\\"; break\\n            if self.actions_used >= self.max_actions:\\n                stopped = \\\"action budget exhausted\\\"; break\\n            before = self.frame; prev_level = self.level\\n            raw = self._step_engine(act)\\n            self.step_no += 1; self.actions_used += 1; self.level_actions += 1; executed += 1\\n            self._apply(raw)\\n            after = self.frame\\n            label = act[\\\"action\\\"] if act[\\\"action\\\"] != \\\"MOUSE\\\" else {\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": act[\\\"row\\\"], \\\"col\\\": act[\\\"col\\\"]}\\n            self.transitions.append({\\\"action\\\": label, \\\"before\\\": before.to_payload(), \\\"after\\\": after.to_payload()})\\n            self.host_transitions.append(_T(label, before, after))   # dict label for MOUSE keeps row/col (rule induction needs them)\\n            ch = before.ascii != after.ascii; changed_any |= ch\\n            results.append({\\\"action\\\": label, \\\"changed\\\": ch})\\n            if self.level > prev_level or self.state == \\\"WIN\\\":\\n                level_completed = True; self.level_action_log.append(self.level_actions); self.level_actions = 0; self.game_overs_this_level = 0; self.cycle_hits_this_level = 0\\n                self.attempt_start_index = len(self.host_transitions)   # cycle guard compares states within the current attempt only\\n                self.level_turn_start = self.model_turns\\n                try:\\n                    self.level_recaps.append(self._level_recap(prev_level))\\n                except Exception as e:  # never break play on a recap error\\n                    self._log(f\\\"recap failed: {e!r}\\\")\\n                try:\\n                    cur = [t for t in self.host_transitions if t.before_frame.level == prev_level]\\n                    first_ascii = cur[0].before_frame.ascii if cur else None; start = 0\\n                    for i, t in enumerate(cur):   # last attempt = after the last reset on that level\\n                        if i > 0 and t.before_frame.ascii == first_ascii:\\n                            start = i\\n                    hyps = goal_inference.infer(cur[start:], prev_level, movement_keys=any(a in self.valid_actions for a in (\\\"UP\\\", \\\"DOWN\\\", \\\"LEFT\\\", \\\"RIGHT\\\")))\\n                    if hyps:\\n                        self.goal_hypotheses = hyps + [h for h in self.goal_hypotheses if h.get(\\\"level\\\") != prev_level]\\n                        self._log(\\\"goal hypotheses: \\\" + \\\" | \\\".join(h[\\\"text\\\"] for h in hyps[:3]))\\n                        self._event(kind=\\\"goals\\\", level=prev_level, hypotheses=hyps)\\n                except Exception as e:\\n                    self._log(f\\\"goal inference failed: {e!r}\\\")\\n                self._log(f\\\"*** level {prev_level} completed after {self.level_action_log[-1]} actions (total {self.actions_used})\\\")\\n                stopped = \\\"level completed\\\" if self.state != \\\"WIN\\\" else \\\"game won\\\"; break\\n            if self.state == \\\"GAME_OVER\\\":\\n                game_over = True; self.game_overs_this_level += 1; self._log(\\\"*** game over -> reset\\\"); self.reset(); stopped = \\\"game over (reset done, level restarted)\\\"\\n                self.attempt_start_index = len(self.host_transitions); self.cycle_hits_this_level = 0\\n                self.level_turn_start = self.model_turns   # re-open the thinking window: the level restarts, re-plan with reasoning\\n                break\\n        return {\\\"executed_count\\\": executed, \\\"board_changed\\\": changed_any, \\\"level_completed\\\": level_completed, \\\"game_over\\\": game_over,\\n                \\\"stopped_reason\\\": stopped, \\\"results\\\": results[-12:]}\\n\\n    # ------------------------------------------------------------ sandbox hooks\\n    def _state(self) -> dict:\\n        return {\\\"frame\\\": self.frame.to_payload() if self.frame else None, \\\"valid_actions\\\": self.valid_actions, \\\"level\\\": self.level,\\n                \\\"levels_total\\\": self.levels_total, \\\"transitions\\\": self.transitions[-400:], \\\"notes\\\": self.notes, \\\"level_recaps\\\": self.level_recaps,\\n                \\\"checklist\\\": self.checklist, \\\"rules_text\\\": self.rules_text, \\\"goal_hypotheses\\\": self.goal_hypotheses}\\n\\n    def _on_action(self, actions: list[dict]) -> tuple[dict, dict]:\\n        if self.proposal_required > 0:\\n            return {\\\"executed_count\\\": 0, \\\"board_changed\\\": False, \\\"level_completed\\\": False, \\\"game_over\\\": False,\\n                    \\\"stopped_reason\\\": \\\"blocked: a level was just completed, so call propose_solver(code) first (inspection is fine, actions are not)\\\",\\n                    \\\"results\\\": []}, self._state()\\n        left = MAX_ACTIONS_PER_TOOL_RUN - self._run_actions\\n        if left <= 0:\\n            return {\\\"executed_count\\\": 0, \\\"board_changed\\\": False, \\\"level_completed\\\": False, \\\"game_over\\\": False,\\n                    \\\"stopped_reason\\\": f\\\"this tool run already executed {MAX_ACTIONS_PER_TOOL_RUN} actions: finish the call, read the board and the outcome, then continue in the next turn\\\",\\n                    \\\"results\\\": []}, self._state()\\n        res = self.execute(actions[:left])\\n        self._run_actions += res[\\\"executed_count\\\"]\\n        if res[\\\"level_completed\\\"]:\\n            self.level_just_completed = True\\n            if self.solver and self.solver[\\\"status\\\"] == \\\"active\\\":\\n                self.solver[\\\"no_progress_actions\\\"] = 0\\n            elif self.state != \\\"WIN\\\":\\n                self.proposal_required = PROPOSAL_ATTEMPTS\\n        return res, self._state()\\n\\n    def _on_solver(self, code: str, report: dict) -> dict:\\n        if report.get(\\\"ok\\\"):\\n            self.solver = solver_policy.new_solver(code, report)\\n            self._log(f\\\"solver stored ({self.solver['status']}):\\\\n\\\" + code)\\n            note = (\\\"Solver stored and VERIFIED; the harness will run solve() automatically from the next turn.\\\" if self.solver[\\\"status\\\"] == \\\"active\\\" else\\n                    \\\"Solver stored as a DRAFT (no predict() or accuracy < 0.8): the harness will show its suggestion each turn but will not run it by itself.\\\")\\n            return {**report, \\\"stored\\\": True, \\\"note\\\": note}\\n        self.solver = {\\\"status\\\": \\\"rejected\\\", \\\"reason\\\": report.get(\\\"reason\\\"), \\\"report\\\": report, \\\"code\\\": code}\\n        self._log(f\\\"solver rejected: {report.get('reason')}\\\")\\n        return {**report, \\\"stored\\\": False}\\n\\n    # ------------------------------------------------------------ model turns\\n    def _log(self, text: str) -> None:\\n        line = f\\\"[{time.time() - self.t0:7.1f}s a={self.actions_used} L{self.level}] {text}\\\"\\n        self.transcript.write(line + \\\"\\\\n\\\"); self.transcript.flush()\\n        if self.verbose:\\n            print(f\\\"{self.game_id} {line[:220]}\\\", flush=True)\\n\\n    def _event(self, **kw) -> None:\\n        self.events.write(json.dumps({\\\"t\\\": round(time.time() - self.t0, 1), **kw}, default=str) + \\\"\\\\n\\\"); self.events.flush()\\n\\n    def _nav_lines(self) -> list[str]:\\n        try:\\n            cur = [t for t in self.host_transitions if t.before_frame.level == t.after_frame.level == self.level]\\n            nav = NavHelper(cur, self.frame)\\n            if nav.moves:\\n                lines = [\\\"Navigation helper summary:\\\", nav.summary()]\\n                ready = []\\n                g = nav.gauge(); left = g.get(\\\"actions_left\\\") if g else None\\n                for t in [t for t in nav.targets() if t.get(\\\"path_len\\\")][:4]:\\n                    path = nav.path_to(t[\\\"row\\\"], t[\\\"col\\\"])\\n                    if path:\\n                        tag = f\\\"colour {t.get('color')}, {'visited' if t.get('visited') else 'unvisited'}\\\"\\n                        if left is not None and len(path) >= left:\\n                            tag += f\\\", TOO LONG for the gauge ({len(path)} >= {left} left)\\\"\\n                        ready.append(f\\\"  nav.path_to({t['row']}, {t['col']}) -> {path}  # {tag}\\\")\\n                fr = nav.frontier()\\n                if fr and (path := nav.path_to(fr[0], fr[1])):\\n                    ready.append(f\\\"  nav.path_to({fr[0]}, {fr[1]}) -> {path}  # nearest unexplored block\\\")\\n                if ready:\\n                    lines.append(\\\"Ready-made routes (pass one straight to action(...)):\\\\n\\\" + \\\"\\\\n\\\".join(ready))\\n                lines += self._gauge_lines(nav, cur)\\n                lines += self._budget_lines(nav, cur)\\n                return lines\\n            g = nav.gauge()\\n            if g:\\n                return [f\\\"Action gauge detected: colour {g.get('color')}, {g.get('size')} cells, {g.get('per_action')} per action, ~{g.get('actions_left')} actions left. \\\"\\n                        \\\"This strip is a counter of your remaining actions (a HUD), NOT a goal to fill or empty: ignore it when choosing targets and finish the level before it runs out.\\\"]\\n        except Exception as e:  # never let the helper break a turn\\n            return [f\\\"Navigation helper unavailable ({type(e).__name__}).\\\"]\\n        if any(a in self.valid_actions for a in (\\\"UP\\\", \\\"DOWN\\\", \\\"LEFT\\\", \\\"RIGHT\\\")):\\n            return [\\\"Navigation helper: no movement learned yet. Press each of UP/DOWN/LEFT/RIGHT once, then read nav.summary().\\\"]\\n        return [\\\"Navigation helper: this level has no movement keys (click game). Use current_frame.segmentation to enumerate objects; nodes with hud=True are edge strips (counters), not targets.\\\"]\\n\\n    def _seconds_left(self) -> float:\\n        left = self.max_minutes * 60 - (time.time() - self.t0)\\n        if self.deadline is not None:\\n            left = min(left, self.deadline - time.time())\\n        return left\\n\\n    def _level_recap(self, level: int) -> str:\\n        \\\"\\\"\\\"What won the level: the last attempt's action sequence, objects that vanished (collected), gauge refills.\\\"\\\"\\\"\\n        cur = [t for t in self.host_transitions if t.before_frame.level == level]\\n        # last attempt = transitions after the last reset on this level (a reset restores the initial board)\\n        first = cur[0].before_frame.ascii if cur else None\\n        start = 0\\n        for i, t in enumerate(cur):\\n            if i > 0 and t.before_frame.ascii == first:\\n                start = i\\n        att = cur[start:]\\n        acts = [t.action if isinstance(t.action, str) else f\\\"MOUSE({t.action['row']},{t.action['col']})\\\" for t in att]\\n        seq, run = [], None\\n        for a in acts:  # compress runs: UP x3\\n            if run and run[0] == a:\\n                run[1] += 1\\n            else:\\n                run = [a, 1]; seq.append(run)\\n        seq_txt = \\\", \\\".join(f\\\"{a} x{n}\\\" if n > 1 else a for a, n in seq)[:400]\\n        before_objs = {n[\\\"hash\\\"]: n for n in att[0].before_frame.segmentation[\\\"nodes\\\"]} if att else {}\\n        after_objs = {n[\\\"hash\\\"]: n for n in att[-1].before_frame.segmentation[\\\"nodes\\\"]} if att else {}\\n        gone = [f\\\"colour {n['color']} ({n['pixels']}px at {n['center']})\\\" for h, n in before_objs.items()\\n                if h not in after_objs and not n[\\\"hud\\\"] and n[\\\"pixels\\\"] <= 300][:6]\\n        refills = []\\n        nav = NavHelper(att, att[-1].after_frame) if att else None\\n        g = nav.gauge() if nav else None\\n        if g:\\n            color = g[\\\"color\\\"]\\n            for t in att[:-1]:   # the last transition enters the next level (its full gauge is not a refill)\\n                b = sum(v == color for row in t.before_frame.grid for v in row); a = sum(v == color for row in t.after_frame.grid for v in row)\\n                if a > b:\\n                    refills.append(f\\\"after {t.action if isinstance(t.action, str) else 'MOUSE'} (+{a - b})\\\")\\n        parts = [f\\\"Level {level} was completed in {len(att)} actions (attempt {sum(1 for i, t in enumerate(cur) if i > 0 and t.before_frame.ascii == first) + 1}).\\\",\\n                 f\\\"Winning sequence: {seq_txt}.\\\"]\\n        if gone:\\n            parts.append(\\\"Objects that disappeared during the win (probably collected/used): \\\" + \\\"; \\\".join(gone) + \\\".\\\")\\n        if refills:\\n            parts.append(\\\"Gauge refilled \\\" + \\\", \\\".join(refills[:4]) + \\\".\\\")\\n        parts.append(\\\"The next level normally keeps the same rules with a new layout: find the analogous objects and repeat the strategy with nav.path_to.\\\")\\n        return \\\" \\\".join(parts)\\n\\n    def _probe_sweep(self) -> str:\\n        \\\"\\\"\\\"Try every untried action type once (each movement key, SPACE, clicks on up to 6 untried objects) and\\n        tabulate what each did. Runs at most once per level and only with enough gauge left.\\\"\\\"\\\"\\n        cur = [t for t in self.host_transitions if t.after_frame.level == self.level]\\n        used = {t.action if isinstance(t.action, str) else \\\"MOUSE\\\" for t in cur}\\n        clicked = {(t.action[\\\"row\\\"] // 4, t.action[\\\"col\\\"] // 4) for t in cur if isinstance(t.action, dict)}\\n        plan = [{\\\"action\\\": a} for a in (\\\"UP\\\", \\\"DOWN\\\", \\\"LEFT\\\", \\\"RIGHT\\\", \\\"SPACE\\\", \\\"ACTION7\\\") if a in self.valid_actions and a not in used]\\n        if \\\"MOUSE\\\" in self.valid_actions and self.frame is not None:\\n            nodes = [n for n in self.frame.segmentation[\\\"nodes\\\"] if not n[\\\"hud\\\"] and (n[\\\"center\\\"][0] // 4, n[\\\"center\\\"][1] // 4) not in clicked]\\n            nodes.sort(key=lambda n: n[\\\"pixels\\\"])\\n            seen_colors = set()\\n            for n in nodes:  # one click per colour, smallest objects first\\n                if n[\\\"color\\\"] in seen_colors:\\n                    continue\\n                seen_colors.add(n[\\\"color\\\"]); plan.append({\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": n[\\\"center\\\"][0], \\\"col\\\": n[\\\"center\\\"][1], \\\"colour\\\": n[\\\"color\\\"]})\\n                if len(plan) >= 10:\\n                    break\\n        try:\\n            g = NavHelper([t for t in cur if t.before_frame.level == self.level], self.frame).gauge() if self.frame else None\\n            if g and g.get(\\\"actions_left\\\") is not None:\\n                plan = plan[:max(0, int(g[\\\"actions_left\\\"]) - 4)]\\n        except Exception:\\n            pass\\n        rows = []\\n        for act in plan:   # may be empty when the model already pressed every key: the interaction probes below still run\\n            before = self.frame\\n            res = self.execute([{k: v for k, v in act.items() if k != \\\"colour\\\"}])\\n            if not res[\\\"executed_count\\\"]:\\n                break\\n            after = self.frame\\n            diff = summarize_diff(before, after)\\n            label = act[\\\"action\\\"] if act[\\\"action\\\"] != \\\"MOUSE\\\" else f\\\"MOUSE({act['row']},{act['col']}) on colour {act.get('colour')}\\\"\\n            what = \\\"no change\\\" if diff[\\\"changed_cells\\\"] == 0 else (f\\\"{diff['changed_cells']} cells changed\\\" +\\n                    (f\\\"; moved: {[(m['color'], m['from'], m['to']) for m in diff.get('moved', [])][:3]}\\\" if diff.get(\\\"moved\\\") else \\\"\\\") +\\n                    (f\\\"; appeared: {[(m['color'], m['center']) for m in diff.get('appeared', [])][:3]}\\\" if diff.get(\\\"appeared\\\") else \\\"\\\") +\\n                    (f\\\"; disappeared: {[(m['color'], m['center']) for m in diff.get('disappeared', [])][:3]}\\\" if diff.get(\\\"disappeared\\\") else \\\"\\\"))\\n            rows.append(f\\\"  {label}: {what}\\\")\\n            if res[\\\"level_completed\\\"] or res[\\\"game_over\\\"]:\\n                rows.append(f\\\"  -> {'LEVEL COMPLETED' if res['level_completed'] else 'GAME OVER (level restarted)'}\\\")\\n                break\\n        # interaction probes: go to the nearest targets and press the interaction key there (many games end when you\\n        # interact ON an object, which single-key probes never reveal)\\n        inter = [a for a in (\\\"SPACE\\\", \\\"ACTION7\\\") if a in self.valid_actions]\\n        try:\\n            nav = NavHelper([t for t in cur if t.before_frame.level == self.level], self.frame)\\n            if nav.moves and nav.avatar():\\n                done = 0\\n                for t in [t for t in nav.targets(max_n=12) if t.get(\\\"path_len\\\") and not t.get(\\\"visited\\\")][:3 if inter else 2]:\\n                    path = nav.path_to(t[\\\"row\\\"], t[\\\"col\\\"])\\n                    if not path or len(path) > 14:\\n                        continue\\n                    g = nav.gauge()\\n                    if g and g.get(\\\"actions_left\\\") is not None and len(path) + 1 > int(g[\\\"actions_left\\\"]) - 2:\\n                        continue   # would exhaust the gauge: not worth a game over\\n                    before = self.frame\\n                    res = self.execute([{\\\"action\\\": a} for a in path] + ([{\\\"action\\\": inter[0]}] if inter else []))\\n                    diff = summarize_diff(before, self.frame) if self.frame else {}\\n                    rows.append(f\\\"  go to colour {t['color']} at ({t['row']},{t['col']}){' then ' + inter[0] if inter else ''}: {diff.get('changed_cells', 0)} cells changed\\\"\\n                                + (f\\\"; disappeared: {[(m['color'], m['center']) for m in diff.get('disappeared', [])][:2]}\\\" if diff.get(\\\"disappeared\\\") else \\\"\\\"))\\n                    done += 1\\n                    if res[\\\"level_completed\\\"] or res[\\\"game_over\\\"]:\\n                        rows.append(f\\\"  -> {'LEVEL COMPLETED' if res['level_completed'] else 'GAME OVER (level restarted)'}\\\")\\n                        break\\n                    nav = NavHelper([t2 for t2 in self.host_transitions if t2.before_frame.level == t2.after_frame.level == self.level], self.frame)\\n                    if not nav.avatar():\\n                        break\\n        except Exception as e:\\n            rows.append(f\\\"  (interaction probes skipped: {type(e).__name__})\\\")\\n        self._log(f\\\"probe sweep on level {self.level}: {len(rows)} probes\\\")\\n        if not rows:\\n            return \\\"\\\"\\n        return \\\"Harness probe sweep (each untried action once, then 'go to a target and interact'):\\\\n\\\" + \\\"\\\\n\\\".join(rows)\\n\\n    def _host_probe(self) -> str:\\n        \\\"\\\"\\\"After repeated identical calls: execute ONE untried action so the model gets new information.\\\"\\\"\\\"\\n        cur = [t for t in self.host_transitions if t.after_frame.level == self.level]\\n        used = {t.action if isinstance(t.action, str) else \\\"MOUSE\\\" for t in cur}\\n        clicked = {(t.action[\\\"row\\\"] // 4, t.action[\\\"col\\\"] // 4) for t in cur if isinstance(t.action, dict)}\\n        probe = None\\n        for a in (\\\"SPACE\\\", \\\"ACTION7\\\", \\\"UP\\\", \\\"DOWN\\\", \\\"LEFT\\\", \\\"RIGHT\\\"):\\n            if a in self.valid_actions and a not in used:\\n                probe = {\\\"action\\\": a}; break\\n        if probe is None and \\\"MOUSE\\\" in self.valid_actions and self.frame is not None:\\n            nodes = [n for n in self.frame.segmentation[\\\"nodes\\\"] if not n[\\\"hud\\\"] and (n[\\\"center\\\"][0] // 4, n[\\\"center\\\"][1] // 4) not in clicked]\\n            nodes.sort(key=lambda n: n[\\\"pixels\\\"])\\n            if nodes:\\n                r, c = nodes[0][\\\"center\\\"]; probe = {\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": r, \\\"col\\\": c}\\n        if probe is None:\\n            return \\\"Harness probe: nothing untried is left on this level; change the ORDER or the target of your actions.\\\"\\n        before = self.frame\\n        res = self.execute([probe])\\n        diff = summarize_diff(before, self.frame) if self.frame else {}\\n        label = probe[\\\"action\\\"] if probe[\\\"action\\\"] != \\\"MOUSE\\\" else f\\\"MOUSE(row={probe['row']}, col={probe['col']})\\\"\\n        self._log(f\\\"host probe: {label} -> changed={res['board_changed']}\\\")\\n        return (f\\\"Harness probe (because you repeated yourself): executed {label} -> board_changed={res['board_changed']}, \\\"\\n                f\\\"level_completed={res['level_completed']}, game_over={res['game_over']}; change: {json.dumps(diff, default=str)[:400]}. Build on this.\\\")\\n\\n    def _checklist_lines(self) -> list[str]:\\n        \\\"\\\"\\\"The turn starts from this state instead of from zero: harness facts + the model's own fields, with the next gap named.\\\"\\\"\\\"\\n        if self.checklist_level != self.level:   # new level: plan and tried restart; goal/roles carry over (same rules)\\n            self.checklist[\\\"plan\\\"] = \\\"\\\"; self.checklist[\\\"tried\\\"] = []; self.checklist_level = self.level\\n        cur = [t for t in self.host_transitions if t.before_frame.level == t.after_frame.level == self.level]\\n        used = {t.action if isinstance(t.action, str) else \\\"MOUSE\\\" for t in cur}\\n        nav = None\\n        try:\\n            nav = NavHelper(cur, self.frame) if self.frame else None\\n        except Exception:\\n            pass\\n        items = []\\n        keys = [a for a in self.valid_actions if a != \\\"MOUSE\\\"]\\n        untried = [a for a in keys if a not in used]\\n        moves = nav.moves if nav else {}\\n        ctrl = \\\", \\\".join(f\\\"{k}={moves[k]}\\\" for k in moves) if moves else \\\"no movement learned\\\"\\n        items.append((not untried and (moves or not keys), \\\"controls\\\", ctrl + (f\\\"; untried keys: {untried}\\\" if untried else \\\"\\\") +\\n                      (\\\"; MOUSE available\\\" if \\\"MOUSE\\\" in self.valid_actions else \\\"\\\")))\\n        av = nav.avatar() if nav else None\\n        if keys:\\n            items.append((bool(av), \\\"avatar\\\", f\\\"colours {av['colors']} at ({av['row']},{av['col']})\\\" if av else \\\"not identified (press each movement key once)\\\"))\\n        g = nav.gauge() if nav else None\\n        lim = (f\\\"gauge colour {g['color']}, ~{g.get('actions_left')} actions left\\\" if g else \\\"no gauge detected yet\\\") + f\\\"; game overs on this level: {self.game_overs_this_level}\\\"\\n        items.append((True, \\\"limits\\\", lim))\\n        if self.frame is not None:\\n            seg = [n for n in self.frame.segmentation[\\\"nodes\\\"] if not n[\\\"hud\\\"]]\\n            counts: dict[int, int] = {}\\n            for n in seg:\\n                counts[n[\\\"color\\\"]] = counts.get(n[\\\"color\\\"], 0) + 1\\n            roles = self.checklist.get(\\\"roles\\\") or {}\\n            desc = \\\", \\\".join(f\\\"colour {c} x{k}\\\" + (f\\\" = {roles.get(str(c)) or roles.get(c)}\\\" if (roles.get(str(c)) or roles.get(c)) else \\\" (role ?)\\\") for c, k in sorted(counts.items(), key=lambda x: -x[1])[:8])\\n            known = sum(1 for c in counts if roles.get(str(c)) or roles.get(c))\\n            items.append((known >= min(2, len(counts)), \\\"objects/roles\\\", desc + \\\"   <- fill checklist['roles'][colour] = 'role'\\\"))\\n        goal = self.checklist.get(\\\"goal\\\") or \\\"\\\"\\n        items.append((bool(goal), \\\"goal\\\", goal or \\\"(none)   <- fill checklist['goal'] = 'hypothesis' after a probe\\\"))\\n        tried = self.checklist.get(\\\"tried\\\") or []\\n        items.append((True, \\\"tried\\\", \\\"; \\\".join(tried[-5:]) if tried else \\\"(nothing recorded)   <- checklist['tried'].append('what + outcome')\\\"))\\n        plan = self.checklist.get(\\\"plan\\\") or \\\"\\\"\\n        items.append((bool(plan), \\\"plan\\\", plan or \\\"(none)   <- fill checklist['plan'] = 'next concrete step'\\\"))\\n        done = sum(1 for ok, _, _ in items if ok); first_gap = next((name for ok, name, _ in items if not ok), None)\\n        head = f\\\"CHECKLIST (level {self.level}) \\u2014 {done}/{len(items)} settled. \\\" + (f\\\"Resolve first: {first_gap.upper()}.\\\" if first_gap else \\\"All settled: execute the plan.\\\")\\n        return [head] + [f\\\"[{'x' if ok else ' '}] {name}: {text}\\\" for ok, name, text in items]\\n\\n    def _rule_lines(self) -> list[str]:\\n        \\\"\\\"\\\"Symbolic rules induced from this level's transitions (exact fit; confirmed vs tentative).\\\"\\\"\\\"\\n        cur = [t for t in self.host_transitions if t.before_frame.level == t.after_frame.level == self.level]\\n        if len(cur) < 2 or self.frame is None:\\n            return []\\n        try:\\n            rules, unex = rule_induction.induce(cur, self.frame)\\n            self.rules_text = rule_induction.summary(rules, unex)\\n            return [self.rules_text + \\\"\\\\n(These are induced by the harness from what actually happened; extend them, do not contradict them.)\\\"]\\n        except Exception as e:\\n            return [f\\\"(rule induction unavailable: {type(e).__name__})\\\"]\\n\\n    def _new_kinds_lines(self) -> list[str]:\\n        \\\"\\\"\\\"Object kinds (colour, size class) present on this level that never appeared on earlier levels: probe these first.\\\"\\\"\\\"\\n        if self.level <= 1 or self.frame is None:\\n            return []\\n        def kinds(frame):\\n            out = set()\\n            for n in frame.segmentation[\\\"nodes\\\"]:\\n                if n[\\\"hud\\\"]:\\n                    continue\\n                size = \\\"tiny\\\" if n[\\\"pixels\\\"] <= 4 else \\\"small\\\" if n[\\\"pixels\\\"] <= 30 else \\\"medium\\\" if n[\\\"pixels\\\"] <= 200 else \\\"large\\\"\\n                out.add((n[\\\"color\\\"], size))\\n            return out\\n        seen = set()\\n        for t in self.host_transitions:\\n            if t.before_frame.level < self.level:\\n                seen |= kinds(t.before_frame)\\n        now = kinds(self.frame)\\n        new = sorted(now - seen)\\n        if not new:\\n            return []\\n        ex = {}\\n        for n in self.frame.segmentation[\\\"nodes\\\"]:\\n            size = \\\"tiny\\\" if n[\\\"pixels\\\"] <= 4 else \\\"small\\\" if n[\\\"pixels\\\"] <= 30 else \\\"medium\\\" if n[\\\"pixels\\\"] <= 200 else \\\"large\\\"\\n            if (n[\\\"color\\\"], size) in new and (n[\\\"color\\\"], size) not in ex:\\n                ex[(n[\\\"color\\\"], size)] = n[\\\"center\\\"]\\n        items = \\\", \\\".join(f\\\"colour {c} ({sz}, e.g. at {ex.get((c, sz))})\\\" for c, sz in new[:6])\\n        return [f\\\"NEW ON THIS LEVEL (not seen on earlier levels): {items}. A new kind usually carries the new rule of this level: \\\"\\n                \\\"touch/click each one once early and record what it did.\\\"]\\n\\n    def _budget_lines(self, nav, cur) -> list[str]:\\n        \\\"\\\"\\\"Arithmetic the model tends to skip: moves left on the gauge vs. route lengths, and whether refills are known.\\\"\\\"\\\"\\n        g = nav.gauge() if nav else None\\n        if not g or g.get(\\\"actions_left\\\") is None:\\n            return []\\n        per = abs(float(g.get(\\\"per_action\\\") or 1)) or 1\\n        full = int(round(g[\\\"size\\\"] / per)) if g.get(\\\"size\\\") else None\\n        lines = [f\\\"BUDGET CHECK: ~{g['actions_left']} moves left on this gauge\\\" + (f\\\" (a full gauge is ~{full} moves at {per:g} per move)\\\" if full else \\\"\\\") + \\\".\\\"]\\n        try:\\n            reach = [t for t in nav.targets() if t.get(\\\"path_len\\\")]\\n            if reach:\\n                far = max(t[\\\"path_len\\\"] for t in reach); near = min(t[\\\"path_len\\\"] for t in reach)\\n                lines.append(f\\\"Known targets are {near}-{far} moves away. Anything beyond {g['actions_left']} moves is unreachable before the gauge runs out \\\"\\n                             \\\"unless you pass a refill item on the way (objects that increased the gauge when touched, see gauge lines).\\\")\\n        except Exception:\\n            pass\\n        return lines\\n\\n    def _micro_diff_lines(self, before_n: int) -> list[str]:\\n        \\\"\\\"\\\"When the last actions changed only a few cells (a sprite rotated/recoloured), zoom into that region.\\\"\\\"\\\"\\n        new = self.host_transitions[before_n:]\\n        if not new:\\n            return []\\n        b, a = new[0].before_frame, new[-1].after_frame\\n        mb, ma = masked_ascii(b).splitlines(), masked_ascii(a).splitlines()\\n        cells = [(r, c) for r in range(len(mb)) for c in range(len(mb[r])) if mb[r][c] != ma[r][c]]\\n        if not cells or len(cells) > 40:\\n            return []\\n        rs = [r for r, _ in cells]; cs = [c for _, c in cells]\\n        r0, r1 = max(0, min(rs) - 2), min(63, max(rs) + 2); c0, c1 = max(0, min(cs) - 2), min(63, max(cs) + 2)\\n        if (r1 - r0) > 14 or (c1 - c0) > 14:\\n            return []\\n        crop = lambda rows: \\\"\\\\n\\\".join(\\\"    \\\" + rows[r][c0:c1 + 1] for r in range(r0, r1 + 1))\\n        return [f\\\"SMALL CHANGE ZOOM (rows {r0}-{r1}, cols {c0}-{c1}; {len(cells)} cells changed, HUD excluded) \\u2014 before / after:\\\",\\n                crop(b.ascii.splitlines()), \\\"    ->\\\", crop(a.ascii.splitlines())]\\n\\n    def _gauge_lines(self, nav, cur) -> list[str]:\\n        g = nav.gauge()\\n        if not g:\\n            return []\\n        out = [f\\\"Gauge budget: ~{g.get('actions_left')} actions left before the level restarts (colour {g.get('color')}, {g.get('per_action')} per action). \\\"\\n               \\\"Any route longer than that fails: pick a target reachable within the budget or find what refills the gauge first.\\\"]\\n        # refill clues: transitions of this level where the gauge object grew\\n        color = g.get(\\\"color\\\"); grew = []\\n        for t in cur[-60:]:\\n            try:\\n                b = sum(1 for row in t.before_frame.grid for v in row if v == color)\\n                a = sum(1 for row in t.after_frame.grid for v in row if v == color)\\n            except Exception:\\n                continue\\n            if a > b:\\n                grew.append(f\\\"{t.action} (+{a - b})\\\")\\n        if grew:\\n            out.append(\\\"Gauge REFILLED after: \\\" + \\\", \\\".join(grew[-4:]) + \\\" \\u2014 whatever the avatar touched then refills it; plan to collect such objects on the way.\\\")\\n        return out\\n\\n    def _budget_line(self) -> str:\\n        left = self._seconds_left()\\n        return f\\\"Time left: {max(0, left) / 60:.1f} min. Action budget left: {self.max_actions - self.actions_used}.\\\"\\n\\n    def _user_message(self) -> str:\\n        parts = [turn_header(level=self.level, levels_total=self.levels_total, actions_used=self.actions_used, level_actions=self.level_actions,\\n                             valid_actions=self.valid_actions, budget_line=self._budget_line())]\\n        if self.oracle_rules:\\n            parts.append(\\\"KNOWN RULES OF THIS GAME (given, trust them fully):\\\\n\\\" + self.oracle_rules)\\n        try:\\n            block = self._checklist_lines() + self._rule_lines() + self._new_kinds_lines()\\n            key = \\\"\\\\n\\\".join(block)\\n            if key != getattr(self, \\\"_last_block\\\", None) or self.model_turns % 5 == 0:\\n                parts += block; self._last_block = key\\n            else:\\n                parts.append(\\\"(checklist and rules unchanged since last turn)\\\")\\n        except Exception as e:\\n            parts.append(f\\\"(checklist unavailable: {type(e).__name__})\\\")\\n        if self.last_outcome:\\n            parts += self.last_outcome\\n        parts += self._nav_lines()\\n        if (PROBE_SWEEP_AFTER_TURNS and self.level not in self.probe_sweeps and self.model_turns - self.level_turn_start >= PROBE_SWEEP_AFTER_TURNS\\n                and not (self.solver and self.solver.get(\\\"status\\\") == \\\"active\\\")):\\n            lvl = self.level\\n            text = self._probe_sweep()\\n            self.probe_sweeps[lvl] = text if self.level == lvl else \\\"\\\"   # the sweep itself completed the level: nothing to show on the new level\\n            if self.level != lvl:\\n                self.probe_sweeps.setdefault(self.level, \\\"\\\")\\n                self.level_turn_start = self.model_turns\\n        if self.goal_hypotheses:\\n            parts.append(goal_inference.summary(self.goal_hypotheses))\\n            if not self.checklist.get(\\\"goal\\\"):\\n                self.checklist[\\\"goal\\\"] = \\\"(harness hypothesis) \\\" + self.goal_hypotheses[0][\\\"text\\\"][:200]\\n        if self.probe_sweeps.get(self.level):\\n            parts.append(self.probe_sweeps[self.level])\\n        if self.level_recaps:\\n            parts.append(\\\"Recap of previous levels (written by the harness):\\\\n\\\" + \\\"\\\\n\\\".join(f\\\"- {r}\\\" for r in self.level_recaps[-2:]))\\n        parts.append(\\\"Your notes (the sandbox variable `notes`; keep it current instead of re-deriving the rules each turn):\\\\n\\\" +\\n                     (self.notes or \\\"(empty \\u2014 write what each key does, the goal hypothesis and the next plan)\\\"))\\n        if self.proposal_required > 0:\\n            parts.append(f\\\"REQUIRED: you just completed a level, so you know the rules. Call propose_solver(code) with a solve() that reproduces \\\"\\n                         f\\\"the winning strategy from `transitions` of the previous level (use nav.path_to / segmentation, not fixed coordinates). \\\"\\n                         f\\\"action() stays blocked until you do ({self.proposal_required} turn(s) left before the requirement is waived); \\\"\\n                         \\\"inspecting the board is allowed.\\\")\\n        if self.solver and self.solver.get(\\\"status\\\") == \\\"draft\\\":\\n            try:\\n                res = self.sandbox.run(f\\\"__solver_code = {json.dumps(self.solver['code'])}\\\\nexec(compile(__solver_code, 'solver.py', 'exec'), globals())\\\\nprint(list(solve() or [])[:12])\\\", timeout=15)\\n                self.solver[\\\"suggestion\\\"] = (res[\\\"stdout\\\"].strip() or res.get(\\\"error\\\") or \\\"?\\\")[:300]\\n            except Exception as e:\\n                self.solver[\\\"suggestion\\\"] = f\\\"(dry run failed: {e!r})\\\"\\n        parts += solver_policy.status_lines(self.solver, model_turns=self.model_turns, level_just_completed=self.level_just_completed)\\n        self.level_just_completed = False\\n        if self.solver and self.solver.get(\\\"status\\\") in (\\\"failed\\\", \\\"rejected\\\"):\\n            self.solver[\\\"status\\\"] = \\\"shown\\\"\\n        parts.append(\\\"Current board:\\\\n\\\" + self.frame.ascii)\\n        return \\\"\\\\n\\\".join(parts)\\n\\n    def _trim_context(self) -> None:\\n        \\\"\\\"\\\"Older turns lose their board text and long tool output; on overflow drop the oldest turns.\\\"\\\"\\\"\\n        user_idx = [i for i, m in enumerate(self.messages) if m[\\\"role\\\"] == \\\"user\\\"]\\n        for i in user_idx[:-self.keep_full_turns]:\\n            c = self.messages[i][\\\"content\\\"]\\n            if isinstance(c, list):\\n                continue\\n            if \\\"Current board:\\\" in c:\\n                self.messages[i][\\\"content\\\"] = c.split(\\\"Current board:\\\")[0] + \\\"[board omitted]\\\"\\n        tool_idx = [i for i, m in enumerate(self.messages) if m[\\\"role\\\"] == \\\"tool\\\"]\\n        for i in tool_idx[:-self.keep_full_turns]:\\n            c = self.messages[i][\\\"content\\\"]\\n            if len(c) > 800:\\n                self.messages[i][\\\"content\\\"] = c[:600] + \\\"\\\\n...[older tool output trimmed]...\\\"\\n        # token estimate calibrated on the last real prompt_tokens (hex boards tokenize at ~1 token per character)\\n        budget = (self.context_tokens - self.client.max_tokens) * 0.85 if self.client else self.context_tokens * 0.8\\n        est = self._estimate_tokens()\\n        while est > budget and len(self.messages) > 4:\\n            self._drop_oldest_turn(); est = self._estimate_tokens()\\n\\n    def _estimate_tokens(self) -> float:\\n        def _len(c):\\n            return len(c) if isinstance(c, str) else sum(len(p.get(\\\"text\\\", \\\"\\\")) for p in c if p.get(\\\"type\\\") == \\\"text\\\") + 1200 * sum(1 for p in c if p.get(\\\"type\\\") == \\\"image_url\\\")\\n        chars = sum(_len(m.get(\\\"content\\\") or \\\"\\\") + len(json.dumps(m.get(\\\"tool_calls\\\") or \\\"\\\")) for m in self.messages)\\n        return chars * getattr(self, \\\"_tok_per_char\\\", 0.45) + 300\\n\\n    def _drop_oldest_turn(self) -> None:\\n        # messages[0] is the system prompt; a turn is user, assistant(, tool)\\n        k = 1\\n        while k < len(self.messages) and self.messages[k][\\\"role\\\"] != \\\"user\\\":\\n            k += 1\\n        end = k + 1\\n        while end < len(self.messages) and self.messages[end][\\\"role\\\"] != \\\"user\\\":\\n            end += 1\\n        if end - k >= len(self.messages) - 1:  # never drop the only turn\\n            return\\n        del self.messages[k:end]\\n\\n    def _run_tool(self, code: str, *, who: str) -> str:\\n        self._run_actions = 0\\n        res = self.sandbox.run(code, timeout=self.tool_timeout)\\n        if res.get(\\\"notes\\\"):\\n            self.notes = res[\\\"notes\\\"]\\n        if isinstance(res.get(\\\"checklist\\\"), dict) and res[\\\"checklist\\\"]:\\n            ck = res[\\\"checklist\\\"]\\n            self.checklist = {\\\"goal\\\": str(ck.get(\\\"goal\\\") or \\\"\\\")[:300], \\\"roles\\\": ck.get(\\\"roles\\\") if isinstance(ck.get(\\\"roles\\\"), dict) else {},\\n                              \\\"plan\\\": str(ck.get(\\\"plan\\\") or \\\"\\\")[:300], \\\"tried\\\": [str(t)[:120] for t in (ck.get(\\\"tried\\\") or [])][-8:]}\\n        out = res[\\\"stdout\\\"]\\n        if res[\\\"error\\\"]:\\n            out += (\\\"\\\\n\\\" if out else \\\"\\\") + \\\"Error: \\\" + res[\\\"error\\\"]\\n        self._event(kind=\\\"tool\\\", who=who, code=code, stdout=res[\\\"stdout\\\"][:4000], error=res[\\\"error\\\"], actions=res[\\\"actions_executed\\\"])\\n        return out or \\\"(no output)\\\"\\n\\n    def _untried_here(self) -> str:\\n        \\\"\\\"\\\"What has never been tried on this level: keys, clicked objects (by 4x4 cell), reachable targets.\\\"\\\"\\\"\\n        cur = [t for t in self.host_transitions if t.before_frame.level == t.after_frame.level == self.level]\\n        used = {t.action for t in cur if isinstance(t.action, str)}\\n        parts = []\\n        keys = [a for a in self.valid_actions if a != \\\"MOUSE\\\" and a not in used]\\n        if keys:\\n            parts.append(f\\\"keys never pressed: {keys}\\\")\\n        if \\\"MOUSE\\\" in self.valid_actions and self.frame is not None:\\n            clicked = {(t.action[\\\"row\\\"] // 4, t.action[\\\"col\\\"] // 4) for t in cur if isinstance(t.action, dict)}\\n            nodes = [n for n in self.frame.segmentation[\\\"nodes\\\"] if not n[\\\"hud\\\"] and (n[\\\"center\\\"][0] // 4, n[\\\"center\\\"][1] // 4) not in clicked]\\n            nodes.sort(key=lambda n: n[\\\"pixels\\\"])\\n            if nodes:\\n                parts.append(\\\"objects never clicked: \\\" + \\\", \\\".join(f\\\"colour {n['color']} at {tuple(n['center'])}\\\" for n in nodes[:6]) + (f\\\" (+{len(nodes) - 6} more)\\\" if len(nodes) > 6 else \\\"\\\"))\\n        try:\\n            nav = NavHelper(cur, self.frame) if self.frame else None\\n            if nav and nav.avatar():\\n                targets = [tg for tg in nav.targets() if tg.get(\\\"path_len\\\") is not None and not tg.get(\\\"visited\\\")]\\n                if targets:\\n                    parts.append(\\\"reachable targets: \\\" + \\\", \\\".join(f\\\"colour {tg['color']} at ({tg['row']},{tg['col']}) in {tg['path_len']} moves (never visited)\\\" for tg in targets[:5]))\\n        except Exception:\\n            pass\\n        return \\\"; \\\".join(parts) if parts else \\\"nothing obvious is untried: change the ORDER (e.g. interact right after arriving) or combine keys.\\\"\\n\\n    def _cycle_lines(self, before_n: int) -> list[str]:\\n        \\\"\\\"\\\"Detect (a) a board state already visited in this level attempt and (b) a repeated action sequence; both mean the model is looping.\\n        Only evaluated on turns that executed at least one action (a warning is not repeated while the model merely thinks).\\\"\\\"\\\"\\n        if len(self.host_transitions) == before_n:\\n            return []\\n        cur = [t for t in self.host_transitions[self.attempt_start_index:] if t.before_frame.level == t.after_frame.level == self.level]\\n        if len(cur) < 4 or self.frame is None:\\n            return []\\n        out = []\\n        labels = []\\n        for t in cur:\\n            a = t.action\\n            labels.append(a if isinstance(a, str) else f\\\"MOUSE({a['row'] // 4},{a['col'] // 4})\\\")\\n        # (b) period detection on the last actions: the same sequence of length k (2..8) executed 3x in a row for short k, 2x for k >= 4\\n        #     (k = 1 is excluded: walking UP UP UP or clicking a toggle twice is normal play; a real loop shows up as a state revisit)\\n        period = None\\n        for k in range(2, 9):\\n            reps = 3 if k <= 3 else 2\\n            if len(labels) >= reps * k and all(labels[-k:] == labels[-(i + 1) * k:-i * k or None] for i in range(1, reps)) and len(set(labels[-k:])) >= 2:\\n                period = k\\n        # (a) state revisit: the current masked board equals the board after an earlier action on this level\\n        now = masked_ascii(self.frame)\\n        first = None\\n        for i, t in enumerate(cur[:-1]):\\n            if masked_ascii(t.after_frame) == now:\\n                first = i; break\\n        if first is not None and len(cur) - 1 - first >= 2:\\n            since = labels[first + 1:]\\n            compact = []\\n            for a in since:\\n                if compact and compact[-1][0] == a:\\n                    compact[-1][1] += 1\\n                else:\\n                    compact.append([a, 1])\\n            seq = \\\" \\\".join(f\\\"{a}x{n}\\\" if n > 1 else a for a, n in compact)[:200]\\n            out.append(f\\\"LOOP DETECTED: the board is identical to what it was after action #{first + 1} of this level, {len(since)} actions ago. \\\"\\n                       f\\\"The actions since then ({seq}) form a cycle and cannot progress the level. Do NOT repeat them. Untried here: {self._untried_here()}\\\")\\n        elif period:\\n            out.append(f\\\"REPEATED SEQUENCE: your last {period} actions ({' '.join(labels[-period:])}) repeat the {period} before them. \\\"\\n                       f\\\"If the board did not move closer to the goal, stop repeating. Untried here: {self._untried_here()}\\\")\\n        if out:\\n            self.cycle_hits_this_level = getattr(self, \\\"cycle_hits_this_level\\\", 0) + 1\\n            self._log(f\\\"cycle guard: {out[0][:90]!r} (hit {self.cycle_hits_this_level})\\\")\\n            if self.cycle_hits_this_level >= 3 and self.cycle_hits_this_level % 3 == 0 and self.valid_actions:\\n                try:\\n                    out.append(self._host_probe())\\n                except Exception:\\n                    pass\\n        return out\\n\\n    def _outcome_lines(self, before_n: int) -> list[str]:\\n        new = self.host_transitions[before_n:]\\n        extra = []\\n        if self.game_overs_this_level:\\n            extra.append(f\\\"GAME OVER count on this level: {self.game_overs_this_level} (each one reset the level). The action limit or the gauge ran out \\\"\\n                         \\\"before the goal: stop repeating the same sweep, re-read what changed the board, and try a different hypothesis.\\\")\\n        if not new:\\n            return [\\\"Last turn executed no actions.\\\"] + extra\\n        acts = [t.action if isinstance(t.action, str) else \\\"MOUSE\\\" for t in new]\\n        diff = summarize_diff(new[0].before_frame, new[-1].after_frame)\\n        return [f\\\"Last turn executed {len(new)} action(s): {', '.join(acts[:12])}{'...' if len(acts) > 12 else ''}. \\\"\\n                f\\\"Board change over the turn: {json.dumps(diff, default=str)[:700]}\\\"] + extra\\n\\n    def _flatten_old_images(self) -> None:\\n        \\\"\\\"\\\"Keep only the newest user message multimodal; older ones become plain text (saves tokens).\\\"\\\"\\\"\\n        users = [m for m in self.messages if m[\\\"role\\\"] == \\\"user\\\" and isinstance(m.get(\\\"content\\\"), list)]\\n        for m in users[:-1]:\\n            m[\\\"content\\\"] = \\\"\\\\n\\\".join(part.get(\\\"text\\\", \\\"\\\") for part in m[\\\"content\\\"] if part.get(\\\"type\\\") == \\\"text\\\") + \\\"\\\\n[board image omitted]\\\"\\n\\n    def model_turn(self) -> bool:\\n        \\\"\\\"\\\"One model call + tool execution. Returns False when the model produced nothing usable.\\\"\\\"\\\"\\n        text = self._user_message()\\n        self._event(kind=\\\"user\\\", turn=self.model_turns + 1, content=text[:6000])   # what the model saw (post-mortems need the harness lines too)\\n        if self.image_context and self.frame is not None:\\n            self.messages.append({\\\"role\\\": \\\"user\\\", \\\"content\\\": [{\\\"type\\\": \\\"text\\\", \\\"text\\\": text + \\\"\\\\nThe same board is attached as an image (8-cell grid lines).\\\"},\\n                                                              {\\\"type\\\": \\\"image_url\\\", \\\"image_url\\\": {\\\"url\\\": \\\"data:image/png;base64,\\\" + grid_to_png_b64(self.frame.grid)}}]})\\n            self._flatten_old_images()\\n        else:\\n            self.messages.append({\\\"role\\\": \\\"user\\\", \\\"content\\\": text})\\n        self._trim_context()\\n        tools = [PYTHON_TOOL, PROPOSE_TOOL]\\n        # after a level completion: one free inspection turn, then the proposal call is forced via tool_choice\\n        choice = {\\\"type\\\": \\\"function\\\", \\\"function\\\": {\\\"name\\\": \\\"propose_solver\\\"}} if 0 < self.proposal_required < PROPOSAL_ATTEMPTS else (\\\"required\\\" if self.tool_choice_required else \\\"auto\\\")\\n        override = None\\n        if self.think_first_turns and (self.model_turns - self.level_turn_start) < self.think_first_turns:\\n            override = {\\\"chat_template_kwargs\\\": {\\\"enable_thinking\\\": True}, \\\"max_tokens\\\": max(self.client.max_tokens, 8192), \\\"temperature\\\": 0.6, \\\"top_p\\\": 0.95}\\n        r = None\\n        for attempt in range(6):\\n            try:\\n                r = self.client.chat(self.messages, tools=tools, tool_choice=choice, override=override); break\\n            except ContextLengthError:\\n                self._tok_per_char = min(1.2, getattr(self, \\\"_tok_per_char\\\", 0.45) * 1.3)   # we under-estimated: be more aggressive\\n                n_before = len(self.messages)\\n                for _ in range(2 + attempt):\\n                    self._drop_oldest_turn()\\n                if len(self.messages) == n_before:   # nothing left to drop: shrink the board text of the current turn\\n                    if isinstance(self.messages[-1][\\\"content\\\"], str):\\n                        self.messages[-1][\\\"content\\\"] = self.messages[-1][\\\"content\\\"].split(\\\"Current board:\\\")[0] + \\\"[board omitted: context full]\\\"\\n                self._log(f\\\"context overflow: dropped turns ({n_before} -> {len(self.messages)} messages), retry {attempt + 1}\\\")\\n        if r is None:\\n            raise RuntimeError(\\\"context overflow could not be resolved\\\")\\n        # calibrate the estimate with the real prompt size\\n        pt = int(r.usage.get(\\\"prompt_tokens\\\") or 0); chars = sum((len(m.get(\\\"content\\\")) if isinstance(m.get(\\\"content\\\"), str) else 1200) for m in self.messages if m.get(\\\"content\\\"))\\n        if pt and chars:\\n            self._tok_per_char = max(0.25, min(1.2, pt / chars))\\n        self.model_turns += 1\\n        msg = r.message\\n        self._event(kind=\\\"model\\\", turn=self.model_turns, content=msg.get(\\\"content\\\", \\\"\\\")[:2000], reasoning=r.reasoning[:1500],\\n                    tool_calls=msg.get(\\\"tool_calls\\\"), latency=round(r.latency, 1), usage=r.usage)\\n        self.messages.append({\\\"role\\\": \\\"assistant\\\", \\\"content\\\": msg.get(\\\"content\\\", \\\"\\\"), **({\\\"tool_calls\\\": msg[\\\"tool_calls\\\"]} if msg.get(\\\"tool_calls\\\") else {})})\\n        for line in (msg.get(\\\"content\\\") or \\\"\\\").splitlines():   # the five-line memory feeds the checklist\\n            low = line.strip().lower()\\n            if low.startswith(\\\"goal model:\\\"):\\n                self.checklist[\\\"goal\\\"] = line.split(\\\":\\\", 1)[1].strip()[:300]\\n            elif low.startswith(\\\"plan:\\\"):\\n                self.checklist[\\\"plan\\\"] = line.split(\\\":\\\", 1)[1].strip()[:300]\\n        self._log(f\\\"model turn {self.model_turns} ({r.latency:.0f}s, {r.usage.get('completion_tokens', '?')} tok): {(msg.get('content') or r.reasoning)[:200]!r}\\\")\\n        calls = msg.get(\\\"tool_calls\\\") or []\\n        if not calls:\\n            # content-as-code fallback: some models write the python call in the message body instead of a tool call\\n            body = (msg.get(\\\"content\\\") or \\\"\\\").strip()\\n            if body.startswith(\\\"```\\\"):\\n                body = body.strip(\\\"`\\\"); body = body.split(\\\"\\\\n\\\", 1)[1] if \\\"\\\\n\\\" in body else body\\n                body = body.rsplit(\\\"```\\\", 1)[0] if \\\"```\\\" in body else body\\n            # JSON fallback: the tool arguments emitted as a bare JSON object in the body (seen with gpt-oss + tool_choice=required)\\n            if body.startswith(\\\"{\\\") and body.endswith(\\\"}\\\"):\\n                try:\\n                    obj = json.loads(body)\\n                    if isinstance(obj, dict) and isinstance(obj.get(\\\"code\\\"), str) and obj[\\\"code\\\"].strip():\\n                        calls = [{\\\"id\\\": \\\"fallback_json\\\", \\\"type\\\": \\\"function\\\", \\\"function\\\": {\\\"name\\\": \\\"python\\\", \\\"arguments\\\": json.dumps(obj)}}]\\n                        self.messages[-1][\\\"tool_calls\\\"] = calls; self.messages[-1][\\\"content\\\"] = \\\"\\\"\\n                        self._log(\\\"json-content fallback: executing the body's `code` field as the python tool\\\")\\n                except Exception:\\n                    pass\\n            looks_like_code = not calls and any(k in body for k in (\\\"action(\\\", \\\"propose_solver(\\\", \\\"print(\\\", \\\"nav.\\\", \\\"checklist[\\\")) and not body.startswith(\\\"{\\\")\\n            if looks_like_code and len(body) < 6000:\\n                try:\\n                    import ast as _ast; _ast.parse(body)\\n                    calls = [{\\\"id\\\": \\\"fallback_0\\\", \\\"type\\\": \\\"function\\\", \\\"function\\\": {\\\"name\\\": \\\"python\\\", \\\"arguments\\\": json.dumps({\\\"code\\\": body})}}]\\n                    self.messages[-1][\\\"tool_calls\\\"] = calls; self.messages[-1][\\\"content\\\"] = \\\"\\\"\\n                    self._log(\\\"content-as-code fallback: executing the message body as python\\\")\\n                except SyntaxError:\\n                    pass\\n        if not calls:\\n            if r.finish_reason == \\\"length\\\":\\n                self.last_outcome = [\\\"Your last reply was cut off by the output limit while you were still reasoning, so nothing happened. \\\"\\n                                     \\\"Reason briefly (a few sentences) and act with one python tool call; let the code do the arithmetic.\\\"]\\n            else:\\n                self.last_outcome = [\\\"Your last reply contained no python tool call; nothing happened. Reply with exactly one python tool call.\\\"]\\n            return False\\n        before_n = len(self.host_transitions)\\n        for call in calls[:1]:\\n            try:\\n                args = json.loads(call[\\\"function\\\"].get(\\\"arguments\\\") or \\\"{}\\\")\\n                code = args.get(\\\"code\\\") or \\\"\\\"\\n            except Exception:\\n                args, code = {}, \\\"\\\"\\n            for key in (\\\"goal\\\", \\\"plan\\\"):   # checklist fields carried by the tool call itself\\n                if isinstance(args.get(key), str) and args[key].strip():\\n                    self.checklist[key] = args[key].strip()[:300]\\n            if isinstance(args.get(\\\"roles\\\"), str) and args[\\\"roles\\\"].strip():\\n                for part in args[\\\"roles\\\"].replace(\\\";\\\", \\\",\\\").split(\\\",\\\"):\\n                    if \\\"=\\\" in part:\\n                        k, v = part.split(\\\"=\\\", 1)\\n                        if k.strip().isdigit() and v.strip():\\n                            self.checklist.setdefault(\\\"roles\\\", {})[k.strip()] = v.strip()[:40]\\n            if call[\\\"function\\\"].get(\\\"name\\\") == \\\"propose_solver\\\" and code:\\n                code = f\\\"print(propose_solver({json.dumps(code)}))\\\"   # routed through the sandbox's verifier\\n            norm = \\\" \\\".join(code.split())\\n            if code and norm in self.recent_codes[-2:]:\\n                used = {t.action if isinstance(t.action, str) else \\\"MOUSE\\\" for t in self.host_transitions if t.after_frame.level == self.level}\\n                unused = [a for a in self.valid_actions if a not in used]\\n                out = (\\\"Rejected: this code is identical to one of your last two calls, so it was NOT executed (repeating it cannot give new information). \\\"\\n                       \\\"Do something different: \\\" + (f\\\"actions not yet tried on this level: {unused}. \\\" if unused else \\\"\\\") +\\n                       \\\"Try a different key, a different object, SPACE/MOUSE on things you have not touched, or update `notes` with a new hypothesis first.\\\")\\n                self._log(\\\"repetition guard: identical tool code rejected\\\")\\n                self.rejections_in_row += 1\\n                if self.rejections_in_row >= 2:\\n                    out += \\\"\\\\n\\\" + self._host_probe()\\n                    self.rejections_in_row = 0\\n            else:\\n                self.rejections_in_row = 0\\n                out = self._run_tool(code, who=\\\"model\\\") if code else \\\"Error: tool call had no `code` argument\\\"\\n            if code:\\n                self.recent_codes = (self.recent_codes + [norm])[-4:]\\n            if self.proposal_required > 0:\\n                if \\\"propose_solver(\\\" in code:\\n                    self.proposal_required = 0\\n                else:\\n                    self.proposal_required -= 1\\n                    if self.proposal_required == 0:\\n                        out += \\\"\\\\n(The solver requirement is now waived; you may act directly.)\\\"\\n            self._log(f\\\"tool ({len(self.host_transitions) - before_n} actions): {out[:300]!r}\\\")\\n            self.messages.append({\\\"role\\\": \\\"tool\\\", \\\"tool_call_id\\\": call.get(\\\"id\\\", \\\"call_0\\\"), \\\"content\\\": out[:6000]})\\n        for call in calls[1:]:  # extra calls are acknowledged, not executed\\n            self.messages.append({\\\"role\\\": \\\"tool\\\", \\\"tool_call_id\\\": call.get(\\\"id\\\", \\\"call_x\\\"), \\\"content\\\": \\\"Skipped: only one python call per reply is executed.\\\"})\\n        self.last_outcome = self._outcome_lines(before_n)\\n        try:\\n            self.last_outcome += self._micro_diff_lines(before_n)\\n        except Exception:\\n            pass\\n        try:\\n            self.last_outcome += self._cycle_lines(before_n)\\n        except Exception as e:\\n            self._log(f\\\"cycle guard error: {type(e).__name__}: {e}\\\")\\n        if len(self.host_transitions) == before_n:\\n            self.zero_action_turns += 1\\n            if self.zero_action_turns >= 2 and self.valid_actions:\\n                try:   # analysis paralysis: keep the game moving with one untried action and report what it did\\n                    self.last_outcome.append(\\\"You have spent 2 turns without acting. \\\" + self._host_probe())\\n                except Exception:\\n                    pass\\n                self.zero_action_turns = 0\\n        else:\\n            self.zero_action_turns = 0\\n        # harness-written 'tried' entry: what this turn did and what happened\\n        new = self.host_transitions[before_n:]\\n        acts = [t.action if isinstance(t.action, str) else \\\"MOUSE\\\" for t in new]\\n        if acts:\\n            compact = []\\n            for a in acts:\\n                if compact and compact[-1][0] == a:\\n                    compact[-1][1] += 1\\n                else:\\n                    compact.append([a, 1])\\n            what = \\\" \\\".join(f\\\"{a}x{n}\\\" if n > 1 else a for a, n in compact)[:80]\\n            changed = sum(1 for t in new if masked_ascii(t.before_frame) != masked_ascii(t.after_frame))\\n            outcome = (\\\"LEVEL DONE\\\" if any(t.before_frame.level != t.after_frame.level for t in new) else\\n                       f\\\"{changed}/{len(new)} moves changed the board\\\")\\n            self.checklist.setdefault(\\\"tried\\\", []).append(f\\\"T{self.model_turns}: {what} -> {outcome}\\\")\\n            self.checklist[\\\"tried\\\"] = self.checklist[\\\"tried\\\"][-8:]\\n        return True\\n\\n    def solver_turn(self) -> None:\\n        s = self.solver\\n        before_n = len(self.host_transitions); prev_level = self.level; prev_go = self.game_overs_this_level + self.resets\\n        budget = solver_policy.MAX_ACTIONS_PER_TURN\\n        try:  # never let a solver turn spend the last actions of a gauge blindly\\n            cur = [t for t in self.host_transitions if t.before_frame.level == t.after_frame.level == self.level]\\n            g = NavHelper(cur, self.frame).gauge() if self.frame else None\\n            if g and g.get(\\\"actions_left\\\") is not None:\\n                budget = min(budget, int(g[\\\"actions_left\\\"]) - 2)\\n        except Exception:\\n            pass\\n        if budget <= 0:\\n            s[\\\"status\\\"], s[\\\"reason\\\"] = \\\"failed\\\", \\\"the action gauge is nearly exhausted; the solver is paused so you can decide what to do with the last actions\\\"\\n            self._log(\\\"solver paused: gauge nearly exhausted\\\"); self.last_outcome = [f\\\"The stored solver was paused: {s['reason']}.\\\"]\\n            return\\n        out = self._run_tool(solver_policy.run_snippet(s[\\\"code\\\"], budget), who=\\\"solver\\\")\\n        s[\\\"turns\\\"] += 1; self.solver_turns += 1\\n        n = len(self.host_transitions) - before_n; s[\\\"actions_run\\\"] += n\\n        # progress = change outside HUD strips (a ticking gauge alone is not progress)\\n        changed = any(masked_ascii(t.before_frame) != masked_ascii(t.after_frame) for t in self.host_transitions[before_n:])\\n        self._log(f\\\"solver turn {s['turns']}: {n} actions, changed={changed}: {out[:160]!r}\\\")\\n        if \\\"Error:\\\" in out:\\n            s[\\\"status\\\"], s[\\\"reason\\\"] = \\\"failed\\\", \\\"solve() raised: \\\" + out.split(\\\"Error:\\\", 1)[1].strip()[:300]\\n        elif \\\"SOLVER_EMPTY\\\" in out:\\n            s[\\\"status\\\"], s[\\\"reason\\\"] = \\\"failed\\\", \\\"solve() returned [] (it could not decide)\\\"\\n        elif n == 0:\\n            s[\\\"status\\\"], s[\\\"reason\\\"] = \\\"failed\\\", \\\"solve() produced no executable action\\\"\\n        elif self.game_overs_this_level + self.resets > prev_go and self.level <= prev_level:\\n            s[\\\"status\\\"], s[\\\"reason\\\"] = \\\"failed\\\", \\\"its actions ran into a GAME OVER (gauge/action limit exhausted before the goal)\\\"\\n        if self.level > prev_level:\\n            s[\\\"noop_turns\\\"] = 0; s[\\\"no_progress_actions\\\"] = 0; s[\\\"board_seen\\\"] = {}\\n            return\\n        s[\\\"noop_turns\\\"] = 0 if changed else s[\\\"noop_turns\\\"] + 1\\n        s[\\\"no_progress_actions\\\"] += n\\n        key = masked_ascii(self.frame) if self.frame else \\\"\\\"\\n        s[\\\"board_seen\\\"][key] = s[\\\"board_seen\\\"].get(key, 0) + 1\\n        if s[\\\"status\\\"] == \\\"active\\\":\\n            if s[\\\"noop_turns\\\"] >= solver_policy.NOOP_TURN_LIMIT:\\n                s[\\\"status\\\"], s[\\\"reason\\\"] = \\\"failed\\\", f\\\"{s['noop_turns']} consecutive solver turns without any board change\\\"\\n            elif s[\\\"no_progress_actions\\\"] >= solver_policy.NO_PROGRESS_ACTIONS:\\n                s[\\\"status\\\"], s[\\\"reason\\\"] = \\\"failed\\\", f\\\"{s['no_progress_actions']} solver actions without completing the level\\\"\\n            elif s[\\\"board_seen\\\"][key] >= solver_policy.CYCLE_WINDOW:\\n                s[\\\"status\\\"], s[\\\"reason\\\"] = \\\"failed\\\", \\\"the solver keeps returning to the same board (cycle)\\\"\\n        if s[\\\"status\\\"] == \\\"failed\\\":\\n            self._log(f\\\"solver failed: {s['reason']}\\\")\\n            self.last_outcome = [f\\\"The stored solver ran {s['actions_run']} actions in total and was stopped: {s['reason']}.\\\"]\\n\\n    # ------------------------------------------------------------ main loop\\n    def _out_of_budget(self) -> Optional[str]:\\n        if self.state == \\\"WIN\\\":\\n            return \\\"won\\\"\\n        if self._seconds_left() <= 0:\\n            return \\\"time\\\"\\n        if self.actions_used >= self.max_actions:\\n            return \\\"actions\\\"\\n        if self.model_turns >= self.max_model_turns:\\n            return \\\"turns\\\"\\n        return None\\n\\n    def _maybe_autopilot(self) -> bool:\\n        \\\"\\\"\\\"Programmatic-first: when the induced rules show a mirrored second body and movement keys, let the\\n        cell-space planner try the level before spending model turns. Returns True if it acted.\\\"\\\"\\\"\\n        if not any(a in self.valid_actions for a in (\\\"UP\\\", \\\"DOWN\\\", \\\"LEFT\\\", \\\"RIGHT\\\")) or self.frame is None:\\n            return False\\n        if self.autopilot_tries.get(self.level, 0) >= 2:\\n            return False\\n        cur = [t for t in self.host_transitions if t.before_frame.level == t.after_frame.level == self.level]\\n        if len(cur) < 4:\\n            return False\\n        try:\\n            rules, _ = rule_induction.induce(cur, self.frame)\\n        except Exception:\\n            return False\\n        mirror = next((r for r in rules if r.kind == \\\"mirror\\\" and r.support >= 3), None)\\n        if not mirror:\\n            # single-avatar: act on a goal hypothesis inferred from an earlier level\\n            hyp = next((h for h in self.goal_hypotheses if h.get(\\\"type\\\") in (\\\"reach\\\", \\\"collect_reach\\\", \\\"collect_all\\\") and h.get(\\\"level\\\", 0) < self.level), None)\\n            if not hyp or not NavHelper(cur, self.frame).moves:\\n                return False\\n            self.autopilot_tries[self.level] = self.autopilot_tries.get(self.level, 0) + 1\\n            lvl = self.level\\n            res = run_reach(self, hyp, max_actions=60, log=lambda m: self._log(m[:200]))\\n            self._event(kind=\\\"autopilot\\\", level=lvl, result=res, hypothesis=hyp)\\n            if res[\\\"completed\\\"]:\\n                self.last_outcome = [f\\\"HARNESS AUTOPILOT completed level {lvl} in {res['actions']} actions by following the inferred goal ({hyp['text']}). \\\"\\n                                     \\\"Press each movement key once on the new level; it will try again.\\\"]\\n            else:\\n                self.last_outcome = [f\\\"HARNESS AUTOPILOT followed the inferred goal ({hyp['text']}) on level {lvl} and stopped: {res['reason']} after {res['actions']} actions. \\\"\\n                                     \\\"Something else is required first (a key, a switch, an item, a refill, an order): find it.\\\"]\\n            return True\\n        nav = NavHelper(cur, self.frame); floor = set(nav.floor_colors) | {nav.background}\\n        self.autopilot_tries[self.level] = self.autopilot_tries.get(self.level, 0) + 1\\n        lvl = self.level\\n        res = run_two_body(self, body_color=mirror.params[\\\"color\\\"], transform=mirror.params[\\\"how\\\"], floor_colors=floor,\\n                           max_replans=10, max_actions=80, log=lambda m: self._log(m[:200]))\\n        self._event(kind=\\\"autopilot\\\", level=lvl, result=res)\\n        if res[\\\"completed\\\"]:\\n            self.last_outcome = [f\\\"HARNESS AUTOPILOT completed level {lvl} in {res['actions']} actions using the confirmed two-body rules \\\"\\n                                 \\\"(mirrored movement, walls, hazards). Press each movement key once on the new level so the rules re-confirm; the autopilot will try again.\\\"]\\n        else:\\n            self.last_outcome = [f\\\"HARNESS AUTOPILOT tried the two-body plan on level {lvl} and stopped: {res['reason']} after {res['actions']} actions. \\\"\\n                                 \\\"A rule the planner does not know is in play (e.g. a gate or a selectable block): find it.\\\"]\\n        return True\\n\\n    def play(self) -> dict:\\n        self.reset()\\n        self.messages = [{\\\"role\\\": \\\"system\\\", \\\"content\\\": system_prompt()}]\\n        self._log(f\\\"start: {self.levels_total} levels, valid={self.valid_actions}\\\")\\n        idle = 0\\n        while not (why := self._out_of_budget()):\\n            if self.solver and self.solver.get(\\\"status\\\") == \\\"active\\\":\\n                self.solver_turn(); continue\\n            try:\\n                if self._maybe_autopilot():\\n                    continue\\n            except Exception as e:\\n                self._log(f\\\"autopilot error: {e!r}\\\")\\n            if self.client is None:\\n                why = \\\"no model\\\"; break\\n            ok = self.model_turn()\\n            idle = 0 if ok else idle + 1\\n            if idle >= 6:\\n                why = \\\"model produced no tool calls\\\"; break\\n        summary = {\\\"game_id\\\": self.game_id, \\\"state\\\": self.state, \\\"levels_completed\\\": self.level - 1 if self.state != \\\"WIN\\\" else self.levels_total,\\n                   \\\"levels_total\\\": self.levels_total, \\\"actions\\\": self.actions_used, \\\"resets\\\": self.resets, \\\"model_turns\\\": self.model_turns,\\n                   \\\"solver_turns\\\": self.solver_turns, \\\"solver_stored\\\": bool(self.solver and self.solver.get(\\\"code\\\")),\\n                   \\\"level_actions\\\": self.level_action_log, \\\"elapsed_s\\\": round(time.time() - self.t0, 1), \\\"stop_reason\\\": why,\\n                   \\\"prompt_tokens\\\": getattr(self.client, \\\"prompt_tokens\\\", 0), \\\"completion_tokens\\\": getattr(self.client, \\\"completion_tokens\\\", 0)}\\n        self._log(f\\\"end: {json.dumps(summary)}\\\")\\n        self.sandbox.close(); self.transcript.close(); self.events.close()\\n        return summary\\n\", \"autopilot.py\": \"\\\"\\\"\\\"Autopilot for two-body (mirrored) merge games in cell space: lattice from the body, plan on cells,\\nexecute step by step, verify against the prediction, refine passable colours, replan. No model calls.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom typing import Optional\\n\\nfrom .frame import texture_colors\\nfrom .nav import extract_objects\\nfrom .planner import lattice_from_body, cell_of, cell_colors, two_body_merge_cells\\n\\nTF = {\\\"same\\\": (1, 1), \\\"mirror_x\\\": (1, -1), \\\"mirror_y\\\": (-1, 1), \\\"mirror_xy\\\": (-1, -1)}\\nMOVES = {\\\"UP\\\": (-1, 0), \\\"DOWN\\\": (1, 0), \\\"LEFT\\\": (0, -1), \\\"RIGHT\\\": (0, 1)}\\n\\n\\ndef _bodies(grid, color, min_size=4):\\n    objs, _ = extract_objects(grid)\\n    return sorted([o for o in objs if o[\\\"color\\\"] == color and o[\\\"size\\\"] >= min_size], key=lambda o: (o[\\\"center\\\"][1], o[\\\"center\\\"][0]))\\n\\n\\ndef run_two_body(session, *, body_color: int, transform: str, floor_colors: set, max_replans: int = 8, max_actions: int = 120, log=print) -> dict:\\n    level0 = session.level; start_actions = session.actions_used\\n    grid = session.frame.grid; tex = texture_colors(grid)\\n    passable = set(floor_colors) | {body_color}; hazards = set(tex); forbidden = set(); blocked_cells = set()\\n    replans = 0\\n    while replans <= max_replans and session.actions_used - start_actions < max_actions:\\n        grid = session.frame.grid; b = _bodies(grid, body_color)\\n        if len(b) != 2:\\n            return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start_actions, \\\"replans\\\": replans, \\\"reason\\\": f\\\"{len(b)} bodies visible\\\"}\\n        rows, cols = lattice_from_body(b[0][\\\"bbox\\\"])\\n        cells = cell_colors(grid, rows, cols)\\n        A, B = cell_of(b[0][\\\"bbox\\\"], rows, cols), cell_of(b[1][\\\"bbox\\\"], rows, cols)\\n        plan = two_body_merge_cells(cells, A, B, transform, passable, hazards, forbidden=forbidden, blocked=blocked_cells)\\n        log(f\\\"autopilot: replan {replans}: lattice {len(rows)-1}x{len(cols)-1}, A{A} B{B}, passable={sorted(passable)}, hazards={sorted(hazards)}, forbidden={sorted(forbidden)}, plan={plan}\\\")\\n        if not plan:\\n            return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start_actions, \\\"replans\\\": replans, \\\"reason\\\": \\\"no plan\\\"}\\n        replans += 1\\n        tf = TF[transform]\\n        for k, act in enumerate(plan):\\n            dr, dc = MOVES[act]\\n            def stp(pos, r_, c_):\\n                r, c = pos[0] + r_, pos[1] + c_\\n                if not (0 <= r < len(cells) and 0 <= c < len(cells[0])):\\n                    return pos\\n                if cells[r][c][1] & hazards:\\n                    return (r, c)\\n                if not cells[r][c][1] <= passable or (r, c) in blocked_cells:\\n                    return pos\\n                return (r, c)\\n            pa, pb = stp(A, dr, dc), stp(B, tf[0] * dr, tf[1] * dc)\\n            res = session.execute([{\\\"action\\\": act}])\\n            if res[\\\"level_completed\\\"] or session.level > level0:\\n                return {\\\"completed\\\": True, \\\"actions\\\": session.actions_used - start_actions, \\\"replans\\\": replans, \\\"reason\\\": \\\"level completed\\\"}\\n            if res[\\\"game_over\\\"]:\\n                return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start_actions, \\\"replans\\\": replans, \\\"reason\\\": \\\"game over\\\"}\\n            grid = session.frame.grid; b2 = _bodies(grid, body_color)\\n            if len(b2) != 2:   # bodies touched and segment as one: finish the next planned steps blindly\\n                rest = plan[k + 1:][:3]\\n                if rest:\\n                    res = session.execute([{\\\"action\\\": x} for x in rest])\\n                    if res[\\\"level_completed\\\"] or session.level > level0:\\n                        return {\\\"completed\\\": True, \\\"actions\\\": session.actions_used - start_actions, \\\"replans\\\": replans, \\\"reason\\\": \\\"level completed\\\"}\\n                break\\n            cells = cell_colors(grid, rows, cols)\\n            c1, c2 = cell_of(b2[0][\\\"bbox\\\"], rows, cols), cell_of(b2[1][\\\"bbox\\\"], rows, cols)\\n            # identity: the bodies may cross; assign observed cells to predictions by total distance\\n            d = lambda x, y: abs(x[0] - y[0]) + abs(x[1] - y[1])\\n            A2, B2 = ((c1, c2) if d(c1, pa) + d(c2, pb) <= d(c2, pa) + d(c1, pb) else (c2, c1))\\n            if A2 == pa and B2 == pb:\\n                A, B = A2, B2\\n                continue\\n            log(f\\\"autopilot: divergence on {act}: predicted A{pa} B{pb}, actual A{A2} B{B2}\\\")\\n            jumped = [(pred, act_pos, body, d) for pred, act_pos, body, d in ((pa, A2, A, (dr, dc)), (pb, B2, B, (tf[0] * dr, tf[1] * dc)))\\n                      if act_pos != pred and act_pos != body and act_pos != (body[0] + d[0], body[1] + d[1])]\\n            if jumped:   # a reset: blame the body whose target cell shows hazard texture; if none does, blame both\\n                blamed = [(body, d) for _, _, body, d in jumped\\n                          if 0 <= body[0] + d[0] < len(cells) and 0 <= body[1] + d[1] < len(cells[0]) and (cells[body[0] + d[0]][body[1] + d[1]][1] & hazards)]\\n                if not blamed:\\n                    blamed = [(body, d) for _, _, body, d in jumped]\\n                for body, d in blamed:\\n                    forbidden.add((body[0] + d[0], body[1] + d[1]))\\n                log(f\\\"autopilot: reset; forbidding {sorted(forbidden)}\\\")\\n                break\\n            for pred, act_pos, body, (r_, c_) in ((pa, A2, A, (dr, dc)), (pb, B2, B, (tf[0] * dr, tf[1] * dc))):\\n                if act_pos == pred:\\n                    continue\\n                target = (body[0] + r_, body[1] + c_)\\n                in_grid = 0 <= target[0] < len(cells) and 0 <= target[1] < len(cells[0])\\n                if act_pos == body and in_grid:        # blocked where we predicted a move\\n                    extra = cells[target[0]][target[1]][1] - passable\\n                    if extra:\\n                        log(f\\\"autopilot: colours {sorted(extra)} block\\\")   # already implied by the whitelist; nothing to change\\n                    else:\\n                        blocked_cells.add(target); log(f\\\"autopilot: cell {target} blocks although it looks like floor\\\")\\n                elif act_pos == target and in_grid:    # moved where we predicted a block\\n                    passable |= cells[target[0]][target[1]][1]; blocked_cells.discard(target); log(f\\\"autopilot: colours {sorted(cells[target[0]][target[1]][1])} are passable\\\")\\n                elif in_grid:\\n                    forbidden.add(target); log(f\\\"autopilot: unexpected position after entering {target}; forbidding it\\\")\\n            break\\n    return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start_actions, \\\"replans\\\": replans, \\\"reason\\\": \\\"budget exhausted\\\"}\\n\\n\\n\\ndef run_reach(session, hypothesis: dict, *, max_actions: int = 60, log=print) -> dict:\\n    \\\"\\\"\\\"Single-avatar autopilot for 'reach' / 'collect_reach' / 'collect_all' hypotheses: pick the next target from the\\n    hypothesis (collectibles nearest-first, then the goal colour), route with nav.path_to over known-walkable cells,\\n    execute at most 12 steps at a time, re-plan from the live board. Stops on level completion, game over, no path,\\n    or no movement. Never spends more than max_actions.\\\"\\\"\\\"\\n    from .nav import NavHelper\\n    level0 = session.level; start = session.actions_used; stalls = 0; visited_targets = set()\\n    kind = hypothesis.get(\\\"type\\\"); goal_c = hypothesis.get(\\\"reach\\\"); coll_c = hypothesis.get(\\\"collect\\\")\\n    while session.actions_used - start < max_actions and session.level == level0:\\n        cur = [t for t in session.host_transitions if t.before_frame.level == t.after_frame.level == session.level]\\n        nav = NavHelper(cur, session.frame)\\n        if not nav.moves or not nav.avatar():\\n            return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"movement not learned\\\"}\\n        targets = [t for t in nav.targets(max_n=20) if t.get(\\\"path_len\\\")]\\n        want = None\\n        if kind in (\\\"collect_reach\\\", \\\"collect_all\\\") and coll_c is not None:\\n            left = [t for t in targets if t[\\\"color\\\"] == coll_c and (t[\\\"row\\\"], t[\\\"col\\\"]) not in visited_targets]\\n            want = min(left, key=lambda t: t[\\\"path_len\\\"]) if left else None\\n        if want is None and kind in (\\\"reach\\\", \\\"collect_reach\\\") and goal_c is not None:\\n            goals = [t for t in targets if t[\\\"color\\\"] == goal_c]\\n            want = min(goals, key=lambda t: t[\\\"path_len\\\"]) if goals else None\\n        if want is None:\\n            return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"no reachable target for the hypothesis\\\"}\\n        path = nav.path_to(want[\\\"row\\\"], want[\\\"col\\\"])\\n        if not path:\\n            return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"no path\\\"}\\n        before = nav.avatar(); step = path[:12]\\n        log(f\\\"reach-autopilot: target colour {want['color']} at ({want['row']},{want['col']}), {len(path)} moves, executing {len(step)}\\\")\\n        res = session.execute([{\\\"action\\\": a} for a in step])\\n        if res[\\\"level_completed\\\"] or session.level > level0:\\n            return {\\\"completed\\\": True, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"level completed\\\"}\\n        if res[\\\"game_over\\\"]:\\n            return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"game over\\\"}\\n        after = NavHelper([t for t in session.host_transitions if t.before_frame.level == t.after_frame.level == session.level], session.frame).avatar()\\n        if after and before and (after[\\\"row\\\"], after[\\\"col\\\"]) == (before[\\\"row\\\"], before[\\\"col\\\"]):\\n            stalls += 1\\n            if stalls >= 2:\\n                return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"avatar did not move (blocked target?)\\\"}\\n        else:\\n            stalls = 0\\n        if len(step) == len(path):\\n            visited_targets.add((want[\\\"row\\\"], want[\\\"col\\\"]))\\n    return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"budget exhausted\\\" if session.level == level0 else \\\"level completed\\\"}\\n\", \"frame.py\": \"\\\"\\\"\\\"Observation encoding: grid -> ascii, connected-component segmentation,\\nobject hashes, containment and adjacency. Pure python, no dependencies.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import Counter\\nfrom dataclasses import dataclass, field\\nfrom typing import Any, Optional\\n\\n# 16 ARC colours as single characters (hex digits keep the mapping obvious)\\nCOLOR_CHARS = \\\"0123456789abcdef\\\"\\nCOLOR_NAMES = {0: \\\"black\\\", 1: \\\"blue\\\", 2: \\\"red\\\", 3: \\\"green\\\", 4: \\\"yellow\\\", 5: \\\"grey\\\", 6: \\\"magenta\\\",\\n               7: \\\"orange\\\", 8: \\\"sky\\\", 9: \\\"brown\\\", 10: \\\"white\\\", 11: \\\"purple\\\", 12: \\\"cyan\\\",\\n               13: \\\"lime\\\", 14: \\\"pink\\\", 15: \\\"teal\\\"}\\n\\n\\ndef grid_to_ascii(grid: list[list[int]]) -> str:\\n    return \\\"\\\\n\\\".join(\\\"\\\".join(COLOR_CHARS[v & 15] for v in row) for row in grid)\\n\\n\\ndef _shape_hash(cells: list[tuple[int, int]], color: int) -> str:\\n    y0 = min(r for r, _ in cells); x0 = min(c for _, c in cells)\\n    norm = tuple(sorted((r - y0, c - x0) for r, c in cells))\\n    return f\\\"{color:x}:{abs(hash(norm)) % 10**8:08d}\\\"\\n\\n\\n@dataclass\\nclass Frame:\\n    grid: list[list[int]]\\n    step: int = 0\\n    level: int = 1\\n    _ascii: Optional[str] = field(default=None, repr=False)\\n    _seg: Optional[dict] = field(default=None, repr=False)\\n\\n    @property\\n    def shape(self) -> tuple[int, int]:\\n        return (len(self.grid), len(self.grid[0]) if self.grid else 0)\\n\\n    @property\\n    def ascii(self) -> str:\\n        if self._ascii is None:\\n            self._ascii = grid_to_ascii(self.grid)\\n        return self._ascii\\n\\n    @property\\n    def background(self) -> int:\\n        return Counter(v for row in self.grid for v in row).most_common(1)[0][0]\\n\\n    @property\\n    def segmentation(self) -> dict:\\n        \\\"\\\"\\\"{'nodes': [...], 'adjacency': [[i, j], ...], 'background': color}.\\n        Node: id, color, pixels, bbox (r0, c0, r1, c1), center (row, col), hash,\\n        children (ids fully inside this node's bbox), hud (edge strip heuristic).\\\"\\\"\\\"\\n        if self._seg is None:\\n            self._seg = segment(self.grid)\\n        return self._seg\\n\\n    def to_payload(self) -> dict:\\n        return {\\\"grid\\\": self.grid, \\\"step\\\": self.step, \\\"level\\\": self.level}\\n\\n    @staticmethod\\n    def from_payload(p: dict) -> \\\"Frame\\\":\\n        return Frame(grid=[list(map(int, r)) for r in p[\\\"grid\\\"]], step=int(p.get(\\\"step\\\", 0)), level=int(p.get(\\\"level\\\", 1)))\\n\\n\\ndef segment(grid: list[list[int]], max_nodes: int = 60) -> dict:\\n    h, w = len(grid), len(grid[0])\\n    bg = Counter(v for row in grid for v in row).most_common(1)[0][0]\\n    comp = [[-1] * w for _ in range(h)]\\n    nodes: list[dict] = []\\n    for sr in range(h):\\n        for sc in range(w):\\n            if comp[sr][sc] >= 0 or grid[sr][sc] == bg:\\n                continue\\n            color = grid[sr][sc]; nid = len(nodes)\\n            stack, cells = [(sr, sc)], []\\n            comp[sr][sc] = nid\\n            while stack:\\n                r, c = stack.pop(); cells.append((r, c))\\n                for nr, nc in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)):\\n                    if 0 <= nr < h and 0 <= nc < w and comp[nr][nc] < 0 and grid[nr][nc] == color:\\n                        comp[nr][nc] = nid; stack.append((nr, nc))\\n            rs = [r for r, _ in cells]; cs = [c for _, c in cells]\\n            r0, r1, c0, c1 = min(rs), max(rs), min(cs), max(cs)\\n            hud = ((r1 - r0 <= 2 and c1 - c0 >= 24 and (r0 <= 1 or r1 >= h - 2)) or\\n                   (c1 - c0 <= 2 and r1 - r0 >= 24 and (c0 <= 1 or c1 >= w - 2)))\\n            nodes.append({\\\"id\\\": nid, \\\"color\\\": color, \\\"pixels\\\": len(cells), \\\"bbox\\\": (r0, c0, r1, c1),\\n                          \\\"center\\\": ((r0 + r1) // 2, (c0 + c1) // 2), \\\"hash\\\": _shape_hash(cells, color),\\n                          \\\"children\\\": [], \\\"hud\\\": hud})\\n    # adjacency (edge-sharing components, ignoring background)\\n    adj: set[tuple[int, int]] = set()\\n    for r in range(h):\\n        for c in range(w):\\n            a = comp[r][c]\\n            if a < 0:\\n                continue\\n            for nr, nc in ((r + 1, c), (r, c + 1)):\\n                if nr < h and nc < w:\\n                    b = comp[nr][nc]\\n                    if b >= 0 and b != a:\\n                        adj.add((min(a, b), max(a, b)))\\n    # containment by bbox (small inside larger)\\n    for n in nodes:\\n        r0, c0, r1, c1 = n[\\\"bbox\\\"]\\n        for m in nodes:\\n            if m is n:\\n                continue\\n            mr0, mc0, mr1, mc1 = m[\\\"bbox\\\"]\\n            if r0 < mr0 and c0 < mc0 and mr1 < r1 and mc1 < c1:\\n                n[\\\"children\\\"].append(m[\\\"id\\\"])\\n    keep = sorted(nodes, key=lambda n: -n[\\\"pixels\\\"])[:max_nodes]\\n    keep_ids = {n[\\\"id\\\"] for n in keep}\\n    return {\\\"nodes\\\": sorted(keep, key=lambda n: n[\\\"id\\\"]),\\n            \\\"adjacency\\\": sorted([list(p) for p in adj if p[0] in keep_ids and p[1] in keep_ids]),\\n            \\\"background\\\": bg}\\n\\n\\ndef summarize_diff(before: Frame, after: Frame, max_items: int = 6) -> dict:\\n    \\\"\\\"\\\"Compact description of what changed between two frames.\\\"\\\"\\\"\\n    changed = [(r, c) for r in range(len(before.grid)) for c in range(len(before.grid[0]))\\n               if before.grid[r][c] != after.grid[r][c]]\\n    out: dict[str, Any] = {\\\"changed_cells\\\": len(changed)}\\n    if not changed:\\n        return out\\n    rs = [r for r, _ in changed]; cs = [c for _, c in changed]\\n    out[\\\"bbox\\\"] = (min(rs), min(cs), max(rs), max(cs))\\n    bs = {n[\\\"hash\\\"]: n for n in before.segmentation[\\\"nodes\\\"]}\\n    as_ = {n[\\\"hash\\\"]: n for n in after.segmentation[\\\"nodes\\\"]}\\n    moved, appeared, gone = [], [], []\\n    for hsh, n in as_.items():\\n        if hsh in bs:\\n            b = bs[hsh]\\n            if b[\\\"center\\\"] != n[\\\"center\\\"]:\\n                moved.append({\\\"color\\\": n[\\\"color\\\"], \\\"pixels\\\": n[\\\"pixels\\\"], \\\"from\\\": b[\\\"center\\\"], \\\"to\\\": n[\\\"center\\\"]})\\n        else:\\n            appeared.append({\\\"color\\\": n[\\\"color\\\"], \\\"pixels\\\": n[\\\"pixels\\\"], \\\"center\\\": n[\\\"center\\\"]})\\n    for hsh, b in bs.items():\\n        if hsh not in as_:\\n            gone.append({\\\"color\\\": b[\\\"color\\\"], \\\"pixels\\\": b[\\\"pixels\\\"], \\\"center\\\": b[\\\"center\\\"]})\\n    out[\\\"moved\\\"] = moved[:max_items]; out[\\\"appeared\\\"] = appeared[:max_items]; out[\\\"disappeared\\\"] = gone[:max_items]\\n    return out\\n\\n\\ndef masked_ascii(frame: \\\"Frame\\\") -> str:\\n    \\\"\\\"\\\"Board text with HUD strips (edge counters/gauges) blanked out, for change and cycle detection.\\\"\\\"\\\"\\n    rows = [list(r) for r in frame.ascii.splitlines()]\\n    for n in frame.segmentation[\\\"nodes\\\"]:\\n        if n.get(\\\"hud\\\"):\\n            r0, c0, r1, c1 = n[\\\"bbox\\\"]\\n            for r in range(r0, r1 + 1):\\n                for c in range(c0, c1 + 1):\\n                    rows[r][c] = \\\".\\\"\\n    return \\\"\\\\n\\\".join(\\\"\\\".join(r) for r in rows)\\n\\n\\ndef infer_cell_grid(grid: list[list[int]], min_n: int = 6, max_n: int = 32, tolerance: float = 0.06):\\n    \\\"\\\"\\\"Recover the game's logical N x N cell grid from a 64 x 64 nearest-neighbour render.\\n    Returns (n, cells) with cells[r][c] = the dominant colour of block (r, c), or None if no N fits.\\\"\\\"\\\"\\n    h = len(grid)\\n    best = None\\n    for n in range(min_n, max_n + 1):\\n        bounds = [int(i * h / n) for i in range(n + 1)]\\n        bad = 0; cells = []\\n        for r in range(n):\\n            row = []\\n            for c in range(n):\\n                vals = Counter(grid[i][j] for i in range(bounds[r], bounds[r + 1]) for j in range(bounds[c], bounds[c + 1]))\\n                colour, cnt = vals.most_common(1)[0]\\n                total = sum(vals.values())\\n                if cnt < total:\\n                    bad += 1\\n                row.append(colour)\\n            cells.append(row)\\n        if bad <= tolerance * n * n:\\n            return n, cells\\n    return None\\n\\n\\ndef cell_bounds(n: int, size: int = 64) -> list[int]:\\n    return [int(i * size / n) for i in range(n + 1)]\\n\\n\\ndef cell_lattice(grid: list[list[int]], hud_rows: set | None = None, hud_cols: set | None = None, min_votes: int = 2):\\n    \\\"\\\"\\\"Recover the cell lattice of a rendered game grid from colour-change positions.\\n    Returns (row_bounds, col_bounds): boundary indices (a cell spans [b[k], b[k+1])). Works for any scaling.\\\"\\\"\\\"\\n    h, w = len(grid), len(grid[0])\\n    hud_rows = hud_rows or set(); hud_cols = hud_cols or set()\\n    col_votes = [0] * (w + 1); row_votes = [0] * (h + 1)\\n    for i in range(h):\\n        if i in hud_rows:\\n            continue\\n        for j in range(1, w):\\n            if grid[i][j] != grid[i][j - 1]:\\n                col_votes[j] += 1\\n    for j in range(w):\\n        if j in hud_cols:\\n            continue\\n        for i in range(1, h):\\n            if grid[i][j] != grid[i - 1][j]:\\n                row_votes[i] += 1\\n    col_b = [0] + [j for j in range(1, w) if col_votes[j] >= min_votes] + [w]\\n    row_b = [0] + [i for i in range(1, h) if row_votes[i] >= min_votes] + [h]\\n    return row_b, col_b\\n\\n\\ndef to_cells(grid: list[list[int]], row_b: list[int], col_b: list[int]) -> list[list[int]]:\\n    \\\"\\\"\\\"Dominant colour per lattice cell.\\\"\\\"\\\"\\n    out = []\\n    for r in range(len(row_b) - 1):\\n        row = []\\n        for c in range(len(col_b) - 1):\\n            vals = Counter(grid[i][j] for i in range(row_b[r], row_b[r + 1]) for j in range(col_b[c], col_b[c + 1]))\\n            row.append(vals.most_common(1)[0][0])\\n        out.append(row)\\n    return out\\n\\n\\ndef hud_lines(frame) -> tuple[set, set]:\\n    rows, cols = set(), set()\\n    for n in frame.segmentation[\\\"nodes\\\"]:\\n        if n[\\\"hud\\\"]:\\n            r0, c0, r1, c1 = n[\\\"bbox\\\"]\\n            if r1 - r0 <= 2:\\n                rows |= set(range(r0, r1 + 1))\\n            if c1 - c0 <= 2:\\n                cols |= set(range(c0, c1 + 1))\\n    return rows, cols\\n\\n\\ndef texture_colors(grid: list[list[int]]) -> set:\\n    \\\"\\\"\\\"Colours that occur mostly as isolated 1-px runs (checkerboard textures inside cells).\\\"\\\"\\\"\\n    stats: dict = {}\\n    for row in grid:\\n        prev = row[0]; k = 0\\n        for v in row + [None]:\\n            if v == prev:\\n                k += 1\\n            else:\\n                s = stats.setdefault(prev, [0, 0]); s[0] += 1; s[1] += (k == 1)\\n                prev = v; k = 1\\n    return {c for c, (n, ones) in stats.items() if n >= 8 and ones / n > 0.8}\\n\\n\\ndef find_lattice(grid: list[list[int]], ignore: set | None = None, n_range=(6, 24), origins=(0, 1, 2, 3, 4)):\\n    \\\"\\\"\\\"Find (n, origin, bounds) such that colour-change positions (ignoring texture colours) fall on the lattice\\n    round(origin + k*(64-2*origin)/n). Returns (coverage, n, origin, bounds) or None.\\\"\\\"\\\"\\n    ignore = ignore or set()\\n    h, w = len(grid), len(grid[0])\\n    col_changes: Counter = Counter(); row_changes: Counter = Counter()\\n    for i in range(h):\\n        for j in range(1, w):\\n            if grid[i][j] != grid[i][j - 1] and grid[i][j] not in ignore and grid[i][j - 1] not in ignore:\\n                col_changes[j] += 1\\n    for j in range(w):\\n        for i in range(1, h):\\n            if grid[i][j] != grid[i - 1][j] and grid[i][j] not in ignore and grid[i - 1][j] not in ignore:\\n                row_changes[i] += 1\\n    total = sum(col_changes.values()) + sum(row_changes.values())\\n    if not total:\\n        return None\\n    best = None\\n    for n in range(n_range[0], n_range[1] + 1):\\n        for o in origins:\\n            step = (w - 2 * o) / n\\n            bounds = [round(o + k * step) for k in range(n + 1)]\\n            bs = set(bounds)\\n            hit = sum(v for j, v in col_changes.items() if j in bs) + sum(v for i, v in row_changes.items() if i in bs)\\n            cov = hit / total\\n            if best is None or cov > best[0] + 1e-9 or (abs(cov - best[0]) < 1e-9 and n < best[1]):\\n                best = (cov, n, o, bounds)\\n    return best\\n\\n\\ndef lattice_cells(grid: list[list[int]], bounds: list[int]) -> list[list[Counter]]:\\n    \\\"\\\"\\\"Colour histogram per lattice cell (cells outside the lattice margin are ignored).\\\"\\\"\\\"\\n    out = []\\n    for r in range(len(bounds) - 1):\\n        row = []\\n        for c in range(len(bounds) - 1):\\n            row.append(Counter(grid[i][j] for i in range(bounds[r], bounds[r + 1]) for j in range(bounds[c], bounds[c + 1])))\\n        out.append(row)\\n    return out\\n\\n\\n# ARC colour palette (index -> RGB), used for the optional image observation\\nPALETTE = [(0, 0, 0), (0, 116, 217), (255, 65, 54), (46, 204, 64), (255, 220, 0), (170, 170, 170), (240, 18, 190), (255, 133, 27),\\n           (127, 219, 255), (135, 12, 37), (255, 255, 255), (177, 13, 201), (0, 200, 200), (140, 240, 60), (255, 160, 180), (0, 140, 120)]\\n\\n\\ndef grid_to_png_b64(grid: list[list[int]], scale: int = 6, gridlines: bool = True) -> str:\\n    \\\"\\\"\\\"Render the board as a PNG (base64) with a thin grid every cell so the model can count cells.\\\"\\\"\\\"\\n    import base64, io\\n    from PIL import Image, ImageDraw\\n    h, w = len(grid), len(grid[0])\\n    img = Image.new(\\\"RGB\\\", (w * scale, h * scale))\\n    d = ImageDraw.Draw(img)\\n    for r in range(h):\\n        for c in range(w):\\n            d.rectangle([c * scale, r * scale, (c + 1) * scale - 1, (r + 1) * scale - 1], fill=PALETTE[grid[r][c] & 15])\\n    if gridlines and scale >= 4:\\n        for k in range(0, w * scale, scale * 8):\\n            d.line([(k, 0), (k, h * scale - 1)], fill=(60, 60, 60), width=1)\\n        for k in range(0, h * scale, scale * 8):\\n            d.line([(0, k), (w * scale - 1, k)], fill=(60, 60, 60), width=1)\\n    buf = io.BytesIO(); img.save(buf, format=\\\"PNG\\\")\\n    return base64.b64encode(buf.getvalue()).decode()\\n\", \"goals.py\": \"\\\"\\\"\\\"Win-condition inference from a completed level (harness side, no model).\\n\\nGiven the transitions of the winning attempt (same level, ending with the transition that entered the next\\nlevel), produce ranked goal hypotheses that planners can act on in later levels:\\n  reach:          the final move entered a cell of colour G (a goal tile / door)\\n  collect_reach:  all objects of colour C disappeared during the attempt, then the avatar reached G\\n  collect_all:    the last object of colour C disappeared on the final action (no separate goal tile)\\n  click_sequence: (click games) the ordered colours clicked in the winning attempt\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import Counter\\nfrom typing import Any\\n\\nfrom .nav import NavHelper, extract_objects\\n\\n\\ndef _objs(frame):\\n    return extract_objects(frame.grid)[0]\\n\\n\\ndef infer(attempt: list, level: int, movement_keys: bool = True) -> list[dict]:\\n    \\\"\\\"\\\"attempt: transitions of the winning attempt on `level` (last one enters level+1).\\\"\\\"\\\"\\n    if not attempt:\\n        return []\\n    same = [t for t in attempt if t.before_frame.level == level]\\n    if not same:\\n        return []\\n    first, last = same[0].before_frame, same[-1].before_frame   # last board seen on this level\\n    final = same[-1]\\n    hyps: list[dict] = []\\n    # colours that vanished completely during the attempt (collectibles), ignoring the avatar's own colours\\n    nav = NavHelper(same[:-1] or same, last)\\n    av = nav.avatar(); avatar_cols = set(av[\\\"colors\\\"]) if av else set()\\n    for (c, _), _ in nav.player_votes.most_common(4):\\n        avatar_cols.add(c)\\n    bg = nav.background\\n    def counts(frame):\\n        cnt = Counter()\\n        for o in _objs(frame):\\n            if o[\\\"size\\\"] <= 200 and o[\\\"color\\\"] != bg and o[\\\"color\\\"] not in avatar_cols and o[\\\"color\\\"] not in nav.floor_colors:\\n                cnt[o[\\\"color\\\"]] += 1\\n        return cnt\\n    c0, c1 = counts(first), counts(last)\\n    vanished = {c: n for c, n in c0.items() if n >= 1 and c1.get(c, 0) == 0}\\n    # what did the final move enter? (movement games)\\n    action = final.action if isinstance(final.action, str) else \\\"MOUSE\\\"\\n    entered = None\\n    if action in nav.moves and av:\\n        dx, dy = nav.moves[action]\\n        r, c = av[\\\"row\\\"] + dy, av[\\\"col\\\"] + dx\\n        if 0 <= r < 64 and 0 <= c < 64:\\n            col = last.grid[r][c]\\n            if col != bg and col not in nav.floor_colors and col not in avatar_cols:\\n                entered = col\\n    if entered is not None:\\n        if vanished:\\n            for c, n in sorted(vanished.items(), key=lambda x: -x[1]):\\n                hyps.append({\\\"type\\\": \\\"collect_reach\\\", \\\"collect\\\": c, \\\"collect_count\\\": n, \\\"reach\\\": entered,\\n                             \\\"text\\\": f\\\"collect all colour-{c} objects ({n} on level {level}) and then step onto a colour-{entered} object\\\"})\\n        hyps.append({\\\"type\\\": \\\"reach\\\", \\\"reach\\\": entered, \\\"text\\\": f\\\"step onto a colour-{entered} object (the final move of level {level} entered one)\\\"})\\n    elif vanished:\\n        for c, n in sorted(vanished.items(), key=lambda x: -x[1]):\\n            hyps.append({\\\"type\\\": \\\"collect_all\\\", \\\"collect\\\": c, \\\"collect_count\\\": n, \\\"text\\\": f\\\"make every colour-{c} object disappear ({n} on level {level}); the level ended when the last one vanished\\\"})\\n    # click games: the winning click order by colour\\n    clicks = [t.action for t in same if isinstance(t.action, dict)]\\n    if clicks and not movement_keys:\\n        seq = []\\n        for t in same:\\n            if isinstance(t.action, dict):\\n                r, c = t.action[\\\"row\\\"], t.action[\\\"col\\\"]; seq.append(t.before_frame.grid[r][c])\\n        hyps.append({\\\"type\\\": \\\"click_sequence\\\", \\\"sequence\\\": seq[-12:], \\\"text\\\": f\\\"level {level} was won by clicking colours in this order: {seq[-12:]}\\\"})\\n    # two bodies of the avatar colour became one on the final move (merge games)\\n    if not hyps and avatar_cols:\\n        def n_bodies(frame):\\n            return sum(1 for o in _objs(frame) if o[\\\"color\\\"] in avatar_cols and o[\\\"size\\\"] >= 4)\\n        bodies_last = [o for o in _objs(last) if o[\\\"color\\\"] in avatar_cols and o[\\\"size\\\"] >= 4]\\n        adjacent = len(bodies_last) == 2 and abs(bodies_last[0][\\\"center\\\"][0] - bodies_last[1][\\\"center\\\"][0]) + abs(bodies_last[0][\\\"center\\\"][1] - bodies_last[1][\\\"center\\\"][1]) <= 12\\n        if n_bodies(first) >= 2 and (len(bodies_last) <= 1 or adjacent):\\n            hyps.append({\\\"type\\\": \\\"merge\\\", \\\"colors\\\": sorted(avatar_cols), \\\"text\\\": f\\\"bring the two colour-{sorted(avatar_cols)} bodies onto the same cell (they merged on the final move of level {level})\\\"})\\n    if not hyps:\\n        hyps.append({\\\"type\\\": \\\"unknown\\\", \\\"text\\\": f\\\"level {level} ended after {action}; no clear goal pattern (avatar colours {sorted(avatar_cols)}, vanished {dict(vanished)})\\\"})\\n    for h in hyps:\\n        h[\\\"level\\\"] = level\\n    return hyps\\n\\n\\ndef summary(hyps: list[dict]) -> str:\\n    if not hyps:\\n        return \\\"\\\"\\n    return \\\"GOAL HYPOTHESES (inferred by the harness from how earlier levels were won; the same rule usually applies):\\\\n\\\" + \\\\\\n        \\\"\\\\n\\\".join(f\\\"  {i + 1}. {h['text']}\\\" for i, h in enumerate(hyps[:3]))\\n\", \"llm.py\": \"\\\"\\\"\\\"Minimal OpenAI-compatible chat client (tool calling, reasoning field),\\ndependency-free so it runs inside the Kaggle rerun container.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport time\\nimport urllib.error\\nimport urllib.request\\nfrom dataclasses import dataclass, field\\nfrom typing import Any, Optional\\n\\n\\n@dataclass\\nclass ChatResult:\\n    message: dict\\n    usage: dict = field(default_factory=dict)\\n    reasoning: str = \\\"\\\"\\n    latency: float = 0.0\\n    finish_reason: str = \\\"\\\"\\n\\n\\nclass ContextLengthError(RuntimeError):\\n    pass\\n\\n\\nclass ChatClient:\\n    def __init__(self, base_url: str = \\\"http://127.0.0.1:1234/v1\\\", model: str = \\\"local-qwen\\\", api_key: str = \\\"x\\\",\\n                 temperature: float = 0.6, top_p: float = 0.95, max_tokens: int = 4096, timeout: float = 600.0,\\n                 extra_body: Optional[dict] = None):\\n        self.base_url, self.model, self.api_key = base_url.rstrip(\\\"/\\\"), model, api_key\\n        self.temperature, self.top_p, self.max_tokens, self.timeout = temperature, top_p, max_tokens, timeout\\n        self.extra_body = extra_body or {}\\n        self.prompt_tokens = self.completion_tokens = 0\\n\\n    def chat(self, messages: list[dict], tools: Optional[list[dict]] = None, *, tool_choice: Any = \\\"auto\\\", retries: int = 3,\\n             override: Optional[dict] = None) -> ChatResult:\\n        body: dict[str, Any] = {\\\"model\\\": self.model, \\\"messages\\\": messages, \\\"temperature\\\": self.temperature, \\\"top_p\\\": self.top_p,\\n                                \\\"max_tokens\\\": self.max_tokens, **self.extra_body, **(override or {})}\\n        if tools:\\n            body[\\\"tools\\\"] = tools; body[\\\"tool_choice\\\"] = tool_choice\\n        data = json.dumps(body).encode()\\n        last: Exception = RuntimeError(\\\"no attempt\\\")\\n        for attempt in range(retries):\\n            t0 = time.time()\\n            req = urllib.request.Request(self.base_url + \\\"/chat/completions\\\", data=data,\\n                                         headers={\\\"Content-Type\\\": \\\"application/json\\\", \\\"Authorization\\\": f\\\"Bearer {self.api_key}\\\"})\\n            try:\\n                with urllib.request.urlopen(req, timeout=self.timeout) as resp:\\n                    out = json.loads(resp.read())\\n            except urllib.error.HTTPError as e:\\n                text = e.read().decode(errors=\\\"replace\\\")\\n                if e.code == 400 and (\\\"context length\\\" in text or \\\"maximum context\\\" in text or \\\"too long\\\" in text):\\n                    raise ContextLengthError(text[:300])\\n                last = RuntimeError(f\\\"HTTP {e.code}: {text[:300]}\\\")\\n            except Exception as e:  # network / timeout\\n                last = e\\n            else:\\n                choice = out[\\\"choices\\\"][0]; msg = choice[\\\"message\\\"]\\n                usage = out.get(\\\"usage\\\") or {}\\n                self.prompt_tokens += int(usage.get(\\\"prompt_tokens\\\", 0)); self.completion_tokens += int(usage.get(\\\"completion_tokens\\\", 0))\\n                reasoning = msg.pop(\\\"reasoning_content\\\", None) or msg.pop(\\\"reasoning\\\", None) or \\\"\\\"\\n                msg.setdefault(\\\"content\\\", \\\"\\\")\\n                if msg.get(\\\"content\\\") is None:\\n                    msg[\\\"content\\\"] = \\\"\\\"\\n                return ChatResult(message=msg, usage=usage, reasoning=reasoning, latency=time.time() - t0, finish_reason=choice.get(\\\"finish_reason\\\", \\\"\\\"))\\n            time.sleep(min(30, 2 ** attempt))\\n        raise last\\n\", \"nav.py\": \"\\\"\\\"\\\"Navigation helper (`nav`): learned from the transition history, rebuilt every tool call.\\n\\nAvatar detection from co-moving objects, cell-size estimate, coarse map,\\nfloor/wall/closed-target learning from actual avatar moves, BFS path_to,\\ntargets, frontier and action-gauge detection. Coordinates are (row, col).\\n\\nTypical use inside the tool:\\n    print(nav.summary())                      # avatar, moves, targets, gauge\\n    seq = nav.path_to(row, col)               # list of action names or None\\n    if seq: action(seq)\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import Counter, deque\\nfrom typing import Any, Optional\\n\\nHEX = \\\"0123456789abcdef\\\"\\n_BIG = 400          # objects larger than this are never the avatar\\n_MAX_HISTORY = 300  # transitions considered (recent)\\n\\n\\ndef _grid_of(frame: Any) -> Optional[list[list[int]]]:\\n    if frame is None:\\n        return None\\n    g = getattr(frame, \\\"_grid\\\", None)\\n    if g is None and isinstance(frame, dict):\\n        g = frame.get(\\\"grid\\\")\\n    if g is None:\\n        g = getattr(frame, \\\"grid\\\", None)\\n    if not g:\\n        return None\\n    return [list(map(int, r)) for r in g]\\n\\n\\ndef extract_objects(grid: list[list[int]], max_objects: int = 40) -> list[dict]:\\n    h, w = len(grid), len(grid[0])\\n    background = Counter(c for row in grid for c in row).most_common(1)[0][0]\\n    seen = [[False] * w for _ in range(h)]\\n    objects = []\\n    for sy in range(h):\\n        for sx in range(w):\\n            if seen[sy][sx] or grid[sy][sx] == background:\\n                continue\\n            color = grid[sy][sx]\\n            stack, cells = [(sy, sx)], []\\n            seen[sy][sx] = True\\n            while stack:\\n                y, x = stack.pop()\\n                cells.append((y, x))\\n                for ny, nx in ((y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)):\\n                    if 0 <= ny < h and 0 <= nx < w and not seen[ny][nx] and grid[ny][nx] == color:\\n                        seen[ny][nx] = True\\n                        stack.append((ny, nx))\\n            ys = [c[0] for c in cells]; xs = [c[1] for c in cells]\\n            objects.append({\\\"color\\\": color, \\\"size\\\": len(cells),\\n                            \\\"bbox\\\": (min(xs), min(ys), max(xs), max(ys)),          # x0,y0,x1,y1\\n                            \\\"center\\\": (sum(xs) // len(xs), sum(ys) // len(ys))})    # x,y\\n    objects.sort(key=lambda o: -o[\\\"size\\\"])\\n    return objects[:max_objects], background\\n\\n\\ndef _moved(prev: list[dict], cur: list[dict]) -> list[tuple[dict, int, int]]:\\n    \\\"\\\"\\\"(new object, dx, dy) for objects with identical color+size that shifted.\\\"\\\"\\\"\\n    pool = list(prev)\\n    out = []\\n    for c in cur:\\n        best, best_d = None, None\\n        for i, p in enumerate(pool):\\n            if p is None or p[\\\"color\\\"] != c[\\\"color\\\"] or p[\\\"size\\\"] != c[\\\"size\\\"]:\\n                continue\\n            d = abs(p[\\\"center\\\"][0] - c[\\\"center\\\"][0]) + abs(p[\\\"center\\\"][1] - c[\\\"center\\\"][1])\\n            if best_d is None or d < best_d:\\n                best, best_d = i, d\\n        if best is not None:\\n            p = pool[best]; pool[best] = None\\n            if best_d:\\n                out.append((c, c[\\\"center\\\"][0] - p[\\\"center\\\"][0], c[\\\"center\\\"][1] - p[\\\"center\\\"][1]))\\n    return out\\n\\n\\ndef _gcd(a: int, b: int) -> int:\\n    while b:\\n        a, b = b, a % b\\n    return a\\n\\n\\nclass NavHelper:\\n    def __init__(self, transitions: list, current_frame: Any):\\n        self.grid = _grid_of(current_frame)\\n        self.objects, self.background = extract_objects(self.grid) if self.grid else ([], None)\\n        self.move_log: dict[str, list[tuple[int, int]]] = {}\\n        self.player_votes: Counter = Counter()\\n        self.player_parts: set[tuple[int, int]] = set()\\n        self.gauge_hist: dict[int, list[int]] = {}\\n        self.last_avatar_center: Optional[tuple[int, int]] = None\\n        self._steps: list[tuple[str, list[list[int]], list[list[int]]]] = []\\n        for t in (transitions or [])[-_MAX_HISTORY:]:\\n            b, a = _grid_of(getattr(t, \\\"before_frame\\\", None)), _grid_of(getattr(t, \\\"after_frame\\\", None))\\n            name = str(getattr(t, \\\"action\\\", \\\"\\\") or \\\"\\\").strip().upper()\\n            if b is None or a is None or not name:\\n                continue\\n            self._steps.append((name, b, a))\\n        self._learn()\\n        self.md = self._map_data()\\n        self.floor_votes: Counter = Counter()\\n        self.walls: set[tuple[int, int]] = set()\\n        self.visited: set[tuple[int, int]] = set()\\n        self.last_visit: dict[tuple[int, int], int] = {}   # block -> step index of last visit\\n        self.blocked: dict[tuple[int, int], int] = {}      # non-floor block we tried to enter and failed -> step\\n        self._learn_terrain()\\n\\n    # \\u2500\\u2500 learning \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def _learn(self) -> None:\\n        prev_objs = None\\n        for name, b, a in self._steps:\\n            po, _ = extract_objects(b); co, _ = extract_objects(a)\\n            moved = [m for m in _moved(po, co) if m[0][\\\"size\\\"] <= _BIG]\\n            if moved and not name.startswith(\\\"MOUSE\\\"):\\n                c, dx, dy = max(moved, key=lambda m: (abs(m[1]) + abs(m[2]), m[0][\\\"size\\\"]))\\n                self.move_log.setdefault(name, []).append((dx, dy))\\n                self.player_votes[(c[\\\"color\\\"], c[\\\"size\\\"])] += 1\\n                self.player_parts = {(m[0][\\\"color\\\"], m[0][\\\"size\\\"]) for m in moved if (m[1], m[2]) == (dx, dy)}\\n                self.last_avatar_center = c[\\\"center\\\"]   # disambiguates look-alike objects (trails, copies)\\n            # gauge: same-color object whose size changed a little near the same spot\\n            for o in co:\\n                for p in po:\\n                    if p[\\\"color\\\"] == o[\\\"color\\\"] and p[\\\"size\\\"] != o[\\\"size\\\"] and \\\\\\n                       abs(p[\\\"center\\\"][0] - o[\\\"center\\\"][0]) + abs(p[\\\"center\\\"][1] - o[\\\"center\\\"][1]) <= 3 and \\\\\\n                       o[\\\"size\\\"] > 8:\\n                        h = self.gauge_hist.setdefault(o[\\\"color\\\"], [])\\n                        if not h:\\n                            h.append(p[\\\"size\\\"])\\n                        h.append(o[\\\"size\\\"])\\n                        break\\n\\n    def move_delta(self, name: str) -> Optional[tuple[int, int]]:\\n        log = self.move_log.get(name.upper()) or []\\n        if not log:\\n            return None\\n        if len(log) == 1:\\n            d = log[0]\\n            return d if abs(d[0]) + abs(d[1]) <= 8 else None\\n        (delta, n), = Counter(log).most_common(1)\\n        return delta if n >= 2 and n * 2 > len(log) else None\\n\\n    @property\\n    def moves(self) -> dict[str, tuple[int, int]]:\\n        \\\"\\\"\\\"action name -> (dx, dy) in pixels (x=col, y=row).\\\"\\\"\\\"\\n        out = {}\\n        for a in self.move_log:\\n            d = self.move_delta(a)\\n            if d:\\n                out[a] = d\\n        # opposite keys are assumed to move the opposite way until observed (a wrong guess costs one probe)\\n        for a, b in ((\\\"UP\\\", \\\"DOWN\\\"), (\\\"DOWN\\\", \\\"UP\\\"), (\\\"LEFT\\\", \\\"RIGHT\\\"), (\\\"RIGHT\\\", \\\"LEFT\\\")):\\n            if a in out and b not in out and b not in self.move_log:\\n                out[b] = (-out[a][0], -out[a][1])\\n        return out\\n\\n    def avatar(self) -> Optional[dict]:\\n        main = None\\n        last = getattr(self, \\\"last_avatar_center\\\", None)\\n        for (color, size), _ in self.player_votes.most_common(3):\\n            cands = [o for o in self.objects if o[\\\"color\\\"] == color and o[\\\"size\\\"] == size]\\n            if cands:\\n                main = min(cands, key=lambda o: abs(o[\\\"center\\\"][0] - last[0]) + abs(o[\\\"center\\\"][1] - last[1])) if last else cands[0]\\n                break\\n        if main is None:\\n            return None\\n        x0, y0, x1, y1 = main[\\\"bbox\\\"]; size = main[\\\"size\\\"]; colors = {main[\\\"color\\\"]}\\n        for o in self.objects:\\n            if o is main or (o[\\\"color\\\"], o[\\\"size\\\"]) not in self.player_parts:\\n                continue\\n            bx0, by0, bx1, by1 = o[\\\"bbox\\\"]\\n            if bx0 > x1 + 1 or bx1 < x0 - 1 or by0 > y1 + 1 or by1 < y0 - 1:\\n                continue\\n            x0, y0, x1, y1 = min(x0, bx0), min(y0, by0), max(x1, bx1), max(y1, by1)\\n            size += o[\\\"size\\\"]; colors.add(o[\\\"color\\\"])\\n        return {\\\"colors\\\": sorted(colors), \\\"size\\\": size, \\\"bbox_xyxy\\\": (x0, y0, x1, y1),\\n                \\\"row\\\": (y0 + y1) // 2, \\\"col\\\": (x0 + x1) // 2}\\n\\n    def cell_size(self) -> int:\\n        g = 0\\n        for d in self.moves.values():\\n            for v in (abs(d[0]), abs(d[1])):\\n                if v:\\n                    g = _gcd(g, v)\\n        return g if 2 <= g <= 8 else 4\\n\\n    # \\u2500\\u2500 coarse map \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def _map_data(self) -> Optional[dict]:\\n        if not self.grid:\\n            return None\\n        cell = self.cell_size(); av = self.avatar()\\n        ox = oy = 0\\n        if av:\\n            ox, oy = av[\\\"bbox_xyxy\\\"][0] % cell, av[\\\"bbox_xyxy\\\"][1] % cell\\n        colors, y = [], oy\\n        while y < 64 and len(colors) < 16:\\n            row, x = [], ox\\n            while x < 64 and len(row) < 16:\\n                block = [self.grid[yy][xx] for yy in range(y, min(64, y + cell)) for xx in range(x, min(64, x + cell))]\\n                row.append(Counter(block).most_common(1)[0][0]); x += cell\\n            colors.append(row); y += cell\\n        pb = ((av[\\\"bbox_xyxy\\\"][0] - ox) // cell, (av[\\\"bbox_xyxy\\\"][1] - oy) // cell) if av else None\\n        return {\\\"cell\\\": cell, \\\"ox\\\": ox, \\\"oy\\\": oy, \\\"colors\\\": colors, \\\"player\\\": pb,\\n                \\\"ncols\\\": len(colors[0]), \\\"nrows\\\": len(colors)}\\n\\n    def block_of(self, row: int, col: int) -> tuple[int, int]:\\n        md = self.md\\n        return (max(0, min(md[\\\"ncols\\\"] - 1, (col - md[\\\"ox\\\"]) // md[\\\"cell\\\"])),\\n                max(0, min(md[\\\"nrows\\\"] - 1, (row - md[\\\"oy\\\"]) // md[\\\"cell\\\"])))\\n\\n    def block_moves(self) -> dict[str, tuple[int, int]]:\\n        md = self.md; out = {}\\n        for a, (dx, dy) in self.moves.items():\\n            if dx % md[\\\"cell\\\"] == 0 and dy % md[\\\"cell\\\"] == 0:\\n                out[a] = (dx // md[\\\"cell\\\"], dy // md[\\\"cell\\\"])\\n        return out\\n\\n    def _learn_terrain(self) -> None:\\n        \\\"\\\"\\\"Replay the history through the current map geometry: blocks the avatar\\n        actually moved from are floor; moves that did not move it mark walls.\\\"\\\"\\\"\\n        if not self.md:\\n            return\\n        md = self.md; bm = self.block_moves()\\n        if not bm:\\n            return\\n        prev_block = None\\n        sig = None\\n        for step_i, (name, b, a) in enumerate(self._steps):\\n            objs_a, _ = extract_objects(a)\\n            # locate avatar in the after-frame by main color/size votes\\n            av = None\\n            for (color, size), _ in self.player_votes.most_common(3):\\n                for o in objs_a:\\n                    if o[\\\"color\\\"] == color and o[\\\"size\\\"] == size:\\n                        av = o; break\\n                if av:\\n                    break\\n            blk = ((av[\\\"bbox\\\"][0] - md[\\\"ox\\\"]) // md[\\\"cell\\\"], (av[\\\"bbox\\\"][1] - md[\\\"oy\\\"]) // md[\\\"cell\\\"]) if av else None\\n            if blk is not None:\\n                self.visited.add(blk)\\n                self.last_visit[blk] = step_i\\n            mv = bm.get(name)\\n            if prev_block is not None and blk is not None and mv is not None:\\n                if blk == prev_block:\\n                    tgt = (prev_block[0] + mv[0], prev_block[1] + mv[1])\\n                    tc, tr = tgt\\n                    colour = None\\n                    if 0 <= tr < md[\\\"nrows\\\"] and 0 <= tc < md[\\\"ncols\\\"]:\\n                        cell = md[\\\"cell\\\"]\\n                        x0, y0 = md[\\\"ox\\\"] + tc * cell, md[\\\"oy\\\"] + tr * cell\\n                        block = [a[yy][xx] for yy in range(y0, min(64, y0 + cell)) for xx in range(x0, min(64, x0 + cell))]\\n                        colour = Counter(block).most_common(1)[0][0]\\n                    # a failed step into a non-floor block is a closed target (door/lock), not a wall\\n                    if colour is not None and colour not in self.floor_colors and colour != self.background \\\\\\n                            and self.floor_votes:\\n                        self.blocked[tgt] = step_i\\n                    else:\\n                        self.walls.add(tgt)\\n                elif blk == (prev_block[0] + mv[0], prev_block[1] + mv[1]):\\n                    c, r = prev_block\\n                    if 0 <= r < md[\\\"nrows\\\"] and 0 <= c < md[\\\"ncols\\\"]:\\n                        # colour of the block we just left, in the after-frame\\n                        cell = md[\\\"cell\\\"]\\n                        x0, y0 = md[\\\"ox\\\"] + c * cell, md[\\\"oy\\\"] + r * cell\\n                        block = [a[yy][xx] for yy in range(y0, min(64, y0 + cell)) for xx in range(x0, min(64, x0 + cell))]\\n                        self.floor_votes[Counter(block).most_common(1)[0][0]] += 1\\n            prev_block = blk\\n        if self.md[\\\"player\\\"] is not None:\\n            self.visited.add(self.md[\\\"player\\\"])\\n            self.last_visit[self.md[\\\"player\\\"]] = len(self._steps)\\n\\n    @property\\n    def floor_colors(self) -> set[int]:\\n        strong = {c for c, n in self.floor_votes.items() if n >= 2}\\n        if strong:\\n            return strong\\n        return {self.floor_votes.most_common(1)[0][0]} if self.floor_votes else set()\\n\\n    def walkable(self, b: tuple[int, int]) -> bool:\\n        md = self.md; c, r = b\\n        if not (0 <= r < md[\\\"nrows\\\"] and 0 <= c < md[\\\"ncols\\\"]) or b in self.walls:\\n            return False\\n        return md[\\\"colors\\\"][r][c] in self.floor_colors or b == md[\\\"player\\\"]\\n\\n    # \\u2500\\u2500 planning \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def path_to(self, row: int, col: int, max_len: int = 40) -> Optional[list[str]]:\\n        \\\"\\\"\\\"BFS over known-walkable blocks to the block containing pixel (row, col).\\n        Returns action names (e.g. ['LEFT','LEFT','UP']) or None if unreachable\\n        / navigation not learned yet. The final step may enter any colour.\\\"\\\"\\\"\\n        if not self.md or self.md[\\\"player\\\"] is None:\\n            return None\\n        bm = self.block_moves()\\n        if not bm:\\n            return None\\n        start, target = self.md[\\\"player\\\"], self.block_of(row, col)\\n        if start == target:\\n            return []\\n        if target in self.walls:\\n            return None\\n        prev = {start: (start, \\\"\\\")}\\n        q = deque([start])\\n        while q:\\n            cur = q.popleft()\\n            for a, (dc, dr) in bm.items():\\n                nb = (cur[0] + dc, cur[1] + dr)\\n                if nb in prev:\\n                    continue\\n                if nb == target or self.walkable(nb):\\n                    prev[nb] = (cur, a)\\n                    if nb == target:\\n                        q.clear(); break\\n                    q.append(nb)\\n        if target not in prev:\\n            return None\\n        acts, b = [], target\\n        while b != start:\\n            b, a = prev[b]; acts.append(a)\\n        acts.reverse()\\n        return acts[:max_len]\\n\\n    def targets(self, max_n: int = 10) -> list[dict]:\\n        av = self.avatar()\\n        \\\"\\\"\\\"Non-floor, non-background small objects with (row, col) and path length.\\\"\\\"\\\"\\n        if not self.md or self.md[\\\"player\\\"] is None:\\n            return []\\n        av = self.avatar(); pb = av[\\\"bbox_xyxy\\\"] if av else None\\n        out, seen = [], set()\\n        for o in self.objects:\\n            if o[\\\"size\\\"] > 120 or o[\\\"color\\\"] == self.background or o[\\\"color\\\"] in self.floor_colors:\\n                continue\\n            bx0, by0, bx1, by1 = o[\\\"bbox\\\"]\\n            if ((bx1 - bx0 <= 2 and (bx0 <= 1 or bx1 >= 62)) or (by1 - by0 <= 2 and (by0 <= 1 or by1 >= 62))):\\n                continue   # thin sliver hugging an edge (gauge / counter pieces), never a target\\n            # small marks drawn inside a larger non-floor object are parts of that object, not targets\\n            ox, oy = o[\\\"center\\\"]; contained = False\\n            for big in self.objects:\\n                if big is o or big[\\\"size\\\"] <= o[\\\"size\\\"] * 3 or big[\\\"color\\\"] == self.background or big[\\\"color\\\"] in self.floor_colors:\\n                    continue\\n                bx0, by0, bx1, by1 = big[\\\"bbox\\\"]\\n                if bx0 < ox < bx1 and by0 < oy < by1:\\n                    contained = True; break\\n            if contained:\\n                continue\\n            if False:\\n                continue\\n            x0, y0, x1, y1 = o[\\\"bbox\\\"]\\n            if pb and not (x0 > pb[2] or x1 < pb[0] or y0 > pb[3] or y1 < pb[1]):\\n                continue\\n            b = self.block_of(o[\\\"center\\\"][1], o[\\\"center\\\"][0])\\n            if b in seen or b == self.md[\\\"player\\\"]:\\n                continue\\n            seen.add(b)\\n            p = self.path_to(o[\\\"center\\\"][1], o[\\\"center\\\"][0])\\n            near = [(b[0] + dc, b[1] + dr) for dc in (-1, 0, 1) for dr in (-1, 0, 1)]\\n            touched = [self.last_visit[q] for q in near if q in self.last_visit]\\n            if b in self.blocked:\\n                touched.append(self.blocked[b])\\n            out.append({\\\"color\\\": o[\\\"color\\\"], \\\"colour\\\": o[\\\"color\\\"], \\\"size\\\": o[\\\"size\\\"], \\\"row\\\": o[\\\"center\\\"][1], \\\"col\\\": o[\\\"center\\\"][0],\\n                        \\\"path_len\\\": None if p is None else len(p),\\n                        \\\"visited\\\": bool(touched) or b in self.walls,\\n                        \\\"last_visit\\\": max(touched) if touched else -1})\\n        out.sort(key=lambda t: (t[\\\"path_len\\\"] is None, t[\\\"path_len\\\"] or 0))\\n        return out[:max_n]\\n\\n    def frontier(self) -> Optional[tuple[int, int]]:\\n        \\\"\\\"\\\"Nearest unvisited walkable block as (row, col) pixel centre, else an\\n        unknown-colour block adjacent to the reachable area.\\\"\\\"\\\"\\n        if not self.md or self.md[\\\"player\\\"] is None:\\n            return None\\n        bm = self.block_moves()\\n        if not bm:\\n            return None\\n        md = self.md; start = md[\\\"player\\\"]\\n        seen, q, unknown = {start}, deque([start]), []\\n        while q:\\n            cur = q.popleft()\\n            for dc, dr in bm.values():\\n                nb = (cur[0] + dc, cur[1] + dr)\\n                if nb in seen:\\n                    continue\\n                seen.add(nb)\\n                if self.walkable(nb):\\n                    if nb not in self.visited:\\n                        return self._block_center(nb)\\n                    q.append(nb)\\n                elif nb not in self.walls and 0 <= nb[1] < md[\\\"nrows\\\"] and 0 <= nb[0] < md[\\\"ncols\\\"]:\\n                    unknown.append(nb)\\n        return self._block_center(unknown[0]) if unknown else None\\n\\n    def _block_center(self, b: tuple[int, int]) -> tuple[int, int]:\\n        md = self.md; c, r = b\\n        return (md[\\\"oy\\\"] + r * md[\\\"cell\\\"] + md[\\\"cell\\\"] // 2, md[\\\"ox\\\"] + c * md[\\\"cell\\\"] + md[\\\"cell\\\"] // 2)\\n\\n    def gauge(self) -> Optional[dict]:\\n        for color, hist in self.gauge_hist.items():\\n            if len(hist) < 3:\\n                continue\\n            steps = [b - a for a, b in zip(hist, hist[1:])]\\n            s = steps[-1]; k = 0\\n            while k < len(steps) and steps[-1 - k] == s:\\n                k += 1\\n            if s != 0 and k >= 2:\\n                return {\\\"color\\\": color, \\\"size\\\": hist[-1], \\\"per_action\\\": s,\\n                        \\\"actions_left\\\": hist[-1] // abs(s) if s < 0 else None}\\n            # monotonic but uneven steps (e.g. -1, -2, -1): still a counter; report the average step\\n            recent = steps[-4:]\\n            if len(recent) >= 3 and all(x < 0 for x in recent) or (len(recent) >= 3 and all(x > 0 for x in recent)):\\n                avg = sum(recent) / len(recent)\\n                return {\\\"color\\\": color, \\\"size\\\": hist[-1], \\\"per_action\\\": round(avg, 1),\\n                        \\\"actions_left\\\": int(hist[-1] / abs(avg)) if avg < 0 else None}\\n        return None\\n\\n    # \\u2500\\u2500 text \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def map(self) -> str:\\n        if not self.md:\\n            return \\\"\\\"\\n        md = self.md; pb = md[\\\"player\\\"]\\n        av = self.avatar(); pw = ph = 1\\n        if av:\\n            pw = max(1, (av[\\\"bbox_xyxy\\\"][2] - av[\\\"bbox_xyxy\\\"][0] + md[\\\"cell\\\"]) // md[\\\"cell\\\"])\\n            ph = max(1, (av[\\\"bbox_xyxy\\\"][3] - av[\\\"bbox_xyxy\\\"][1] + md[\\\"cell\\\"]) // md[\\\"cell\\\"])\\n        rows = []\\n        for r, row in enumerate(md[\\\"colors\\\"]):\\n            line = []\\n            for c, col in enumerate(row):\\n                if pb and pb[0] <= c < pb[0] + pw and pb[1] <= r < pb[1] + ph:\\n                    line.append(\\\"P\\\")\\n                elif (c, r) in self.walls:\\n                    line.append(\\\"#\\\")\\n                else:\\n                    line.append(HEX[col])\\n            rows.append(f\\\"{r:2d} {''.join(line)}\\\")\\n        return (f\\\"cell={md['cell']}px origin(row={md['oy']},col={md['ox']}); char=(col,row) block majority colour hex, \\\"\\n                f\\\"P=avatar, #=known wall; pixel row=r*{md['cell']}+{md['oy']}, col=c*{md['cell']}+{md['ox']}\\\\n\\\" + \\\"\\\\n\\\".join(rows))\\n\\n    def summary(self) -> str:\\n        av = self.avatar(); mv = self.moves\\n        lines = []\\n        if av:\\n            lines.append(f\\\"avatar: colours {av['colors']} size {av['size']} at row {av['row']}, col {av['col']}\\\")\\n        else:\\n            lines.append(\\\"avatar: not identified yet (take each movement action once or twice)\\\")\\n        if mv:\\n            lines.append(\\\"moves (dx=cols,dy=rows): \\\" + \\\", \\\".join(f\\\"{a}=({dx:+d},{dy:+d})\\\" for a, (dx, dy) in mv.items()))\\n            lines.append(f\\\"floor colours {sorted(self.floor_colors) or '?'}, walls known {len(self.walls)}, blocks visited {len(self.visited)}\\\")\\n        g = self.gauge()\\n        if g:\\n            lines.append(f\\\"gauge: colour {g['color']} {g['size']} cells, {g['per_action']:+d}/action\\\"\\n                         + (f\\\", ~{g['actions_left']} actions left\\\" if g[\\\"actions_left\\\"] is not None else \\\"\\\"))\\n        ts = self.targets()\\n        if ts:\\n            lines.append(\\\"targets (row,col colour size path visited):\\\")\\n            for t in ts:\\n                lines.append(f\\\"  ({t['row']},{t['col']}) c{t['color']} s{t['size']} \\\"\\n                             + (f\\\"{t['path_len']} moves\\\" if t[\\\"path_len\\\"] is not None else \\\"unreachable\\\")\\n                             + (\\\" visited\\\" if t[\\\"visited\\\"] else \\\"\\\"))\\n        fr = self.frontier()\\n        if fr:\\n            lines.append(f\\\"frontier: nearest unexplored block at row {fr[0]}, col {fr[1]}\\\")\\n        return \\\"\\\\n\\\".join(lines)\\n\\n\\ndef build_nav(transitions: list, current_frame: Any) -> Optional[NavHelper]:\\n    try:\\n        return NavHelper(transitions, current_frame)\\n    except Exception:\\n        return None\\n\", \"planner.py\": \"\\\"\\\"\\\"Rule-based planners that run on confirmed rules instead of the model's arithmetic.\\n\\ntwo_body_merge: joint-state BFS for games where a second body moves with the avatar under a fixed\\ntransform (mirror), walls stop each body separately, hazards reset, and the goal is to bring both\\nbodies onto the same cell (or adjacent-swap). Cells are the movement lattice (cell size from nav).\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import deque\\nfrom typing import Optional\\n\\n\\ndef _cell_color(grid, r, c):\\n    if 0 <= r < len(grid) and 0 <= c < len(grid[0]):\\n        return grid[r][c]\\n    return None\\n\\n\\ndef _body_blocked(grid, r, c, size, wall_colors, body_colors):\\n    \\\"\\\"\\\"Would a size x size body with top-left (r, c) overlap a wall colour or leave the grid?\\\"\\\"\\\"\\n    h, w = len(grid), len(grid[0])\\n    if r < 0 or c < 0 or r + size > h or c + size > w:\\n        return True\\n    for i in range(r, r + size):\\n        for j in range(c, c + size):\\n            v = grid[i][j]\\n            if v in wall_colors and v not in body_colors:\\n                return True\\n    return False\\n\\n\\ndef _touches(grid, r, c, size, colors):\\n    for i in range(r, r + size):\\n        for j in range(c, c + size):\\n            if 0 <= i < len(grid) and 0 <= j < len(grid[0]) and grid[i][j] in colors:\\n                return True\\n    return False\\n\\n\\ndef two_body_merge(grid, body_a, body_b, moves: dict, transform: str, wall_colors, hazard_colors=(), body_colors=(),\\n                   max_nodes: int = 200000) -> Optional[list[str]]:\\n    \\\"\\\"\\\"body_a/body_b: (top_row, left_col, size). moves: action -> (dx, dy) in pixels. transform: mirror_x|mirror_y|mirror_xy|same.\\n    Returns the action list that brings both bodies to the same top-left, avoiding hazards, or None.\\\"\\\"\\\"\\n    (ra, ca, sa), (rb, cb, sb) = body_a, body_b\\n    walls = set(wall_colors); hazards = set(hazard_colors); bcol = set(body_colors)\\n    tf = {\\\"same\\\": (1, 1), \\\"mirror_x\\\": (-1, 1), \\\"mirror_y\\\": (1, -1), \\\"mirror_xy\\\": (-1, -1)}[transform]\\n    start = (ra, ca, rb, cb)\\n    prev = {start: None}; q = deque([start]); n = 0\\n    while q and n < max_nodes:\\n        s = q.popleft(); n += 1\\n        r1, c1, r2, c2 = s\\n        if (r1, c1) == (r2, c2):\\n            path = []\\n            while prev[s] is not None:\\n                s, a = prev[s]; path.append(a)\\n            return path[::-1]\\n        for a, (dx, dy) in moves.items():\\n            n1 = (r1 + dy, c1 + dx); n2 = (r2 + tf[1] * dy, c2 + tf[0] * dx)\\n            if _body_blocked(grid, n1[0], n1[1], sa, walls, bcol):\\n                n1 = (r1, c1)\\n            if _body_blocked(grid, n2[0], n2[1], sb, walls, bcol):\\n                n2 = (r2, c2)\\n            if n1 == (r1, c1) and n2 == (r2, c2):\\n                continue\\n            if hazards and (_touches(grid, n1[0], n1[1], sa, hazards) or _touches(grid, n2[0], n2[1], sb, hazards)):\\n                continue\\n            # swap across adjacent cells also counts as a merge\\n            if n1 == (r2, c2) and n2 == (r1, c1):\\n                path = [a]\\n                t = s\\n                while prev[t] is not None:\\n                    t, b = prev[t]; path.append(b)\\n                return path[::-1]\\n            ns = (n1[0], n1[1], n2[0], n2[1])\\n            if ns not in prev:\\n                prev[ns] = (s, a); q.append(ns)\\n    return None\\n\\n\\n# ---------------------------------------------------------------- cell-space version (lattice from the body itself)\\ndef lattice_from_body(bbox_xyxy, size: int = 64):\\n    \\\"\\\"\\\"A one-cell body at pixel bbox (x0, y0, x1, y1) fixes the lattice: step = body width, origin = x0 mod step.\\\"\\\"\\\"\\n    x0, y0, x1, y1 = bbox_xyxy\\n    step = max(x1 - x0 + 1, y1 - y0 + 1)\\n    ox, oy = x0 % step, y0 % step\\n    cols = [ox + k * step for k in range(0, (size - ox) // step + 1)]\\n    rows = [oy + k * step for k in range(0, (size - oy) // step + 1)]\\n    return rows, cols\\n\\n\\ndef cell_of(bbox_xyxy, rows, cols):\\n    x0, y0, x1, y1 = bbox_xyxy\\n    r = max(i for i, b in enumerate(rows) if b <= y0) if any(b <= y0 for b in rows) else 0\\n    c = max(i for i, b in enumerate(cols) if b <= x0) if any(b <= x0 for b in cols) else 0\\n    return r, c\\n\\n\\ndef cell_colors(grid, rows, cols):\\n    \\\"\\\"\\\"Per cell: (dominant colour, set of colours).\\\"\\\"\\\"\\n    out = []\\n    for r in range(len(rows) - 1):\\n        row = []\\n        for c in range(len(cols) - 1):\\n            vals = {}\\n            for i in range(rows[r], min(rows[r + 1], len(grid))):\\n                for j in range(cols[c], min(cols[c + 1], len(grid[0]))):\\n                    vals[grid[i][j]] = vals.get(grid[i][j], 0) + 1\\n            dom = max(vals, key=vals.get) if vals else None\\n            row.append((dom, set(vals)))\\n        out.append(row)\\n    return out\\n\\n\\ndef two_body_merge_cells(cells, a, b, transform: str, passable, hazard_colors=(), max_nodes: int = 200000, forbidden=(), blocked=()) -> Optional[list[str]]:\\n    \\\"\\\"\\\"cells[r][c] = (dominant, colour set). a, b = (row, col) cell coords. Moves are one cell.\\n    A move is blocked for a body if the target cell's dominant colour is not passable (or off-grid).\\n    Hazard cells (containing a hazard colour) are never entered.\\\"\\\"\\\"\\n    H, W = len(cells), len(cells[0])\\n    MOVES = {\\\"UP\\\": (-1, 0), \\\"DOWN\\\": (1, 0), \\\"LEFT\\\": (0, -1), \\\"RIGHT\\\": (0, 1)}\\n    tf = {\\\"same\\\": (1, 1), \\\"mirror_x\\\": (1, -1), \\\"mirror_y\\\": (-1, 1), \\\"mirror_xy\\\": (-1, -1)}[transform]   # (row sign, col sign)\\n    hz = set(hazard_colors); fb = set(forbidden); blocked = set(blocked); passable = set(passable)\\n\\n    def step(pos, dr, dc):\\n        r, c = pos[0] + dr, pos[1] + dc\\n        if not (0 <= r < H and 0 <= c < W):\\n            return pos\\n        dom, cols = cells[r][c]\\n        if cols & hz:            # hazard-textured cells are enterable (and deadly), never walls\\n            return (r, c)\\n        if not cols <= passable or (r, c) in blocked:   # any unknown sprite in the cell blocks (gates, doors)\\n            return pos\\n        return (r, c)\\n\\n    start = (a, b); prev = {start: None}; q = deque([start]); n = 0\\n    while q and n < max_nodes:\\n        s = q.popleft(); n += 1\\n        pa, pb = s\\n        if pa == pb:\\n            path = []\\n            while prev[s] is not None:\\n                s, act = prev[s]; path.append(act)\\n            return path[::-1]\\n        for act, (dr, dc) in MOVES.items():\\n            na = step(pa, dr, dc); nb = step(pb, tf[0] * dr, tf[1] * dc)\\n            if na == pa and nb == pb:\\n                continue\\n            if hz and (cells[na[0]][na[1]][1] & hz or cells[nb[0]][nb[1]][1] & hz):\\n                continue\\n            if fb and (na in fb or nb in fb):   # cells that caused a reset before\\n                continue\\n            if na == pb and nb == pa:   # swap counts as a merge\\n                path = [act]; t = s\\n                while prev[t] is not None:\\n                    t, x = prev[t]; path.append(x)\\n                return path[::-1]\\n            ns = (na, nb)\\n            if ns not in prev:\\n                prev[ns] = (s, act); q.append(ns)\\n    return None\\n\", \"prompts.py\": \"\\\"\\\"\\\"System prompt and per-turn text for the arcnav agent (our wording).\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nPYTHON_TOOL = {\\n    \\\"type\\\": \\\"function\\\",\\n    \\\"function\\\": {\\n        \\\"name\\\": \\\"python\\\",\\n        \\\"description\\\": \\\"Run Python code in the persistent game sandbox. Use action(...) inside it to play. Print what you need to see.\\\",\\n        \\\"parameters\\\": {\\\"type\\\": \\\"object\\\",\\n                       \\\"properties\\\": {\\\"code\\\": {\\\"type\\\": \\\"string\\\", \\\"description\\\": \\\"Python source to execute\\\"},\\n                                      \\\"goal\\\": {\\\"type\\\": \\\"string\\\", \\\"description\\\": \\\"Your current goal hypothesis for this game (repeat or update it every call)\\\"},\\n                                      \\\"plan\\\": {\\\"type\\\": \\\"string\\\", \\\"description\\\": \\\"The concrete next step this call performs and what comes after\\\"},\\n                                      \\\"roles\\\": {\\\"type\\\": \\\"string\\\", \\\"description\\\": \\\"Object roles learned so far, e.g. '5=door, 8=key, 11=gauge'\\\"}},\\n                       \\\"required\\\": [\\\"code\\\", \\\"goal\\\", \\\"plan\\\"]},\\n    },\\n}\\n\\nPROPOSE_TOOL = {\\n    \\\"type\\\": \\\"function\\\",\\n    \\\"function\\\": {\\n        \\\"name\\\": \\\"propose_solver\\\",\\n        \\\"description\\\": \\\"Store a persistent solver. `code` is Python source defining `def solve():` (and optionally `def predict(before_frame, action_name)`) \\\"\\n                       \\\"that reads current_frame, transitions, level_transitions, nav, valid_actions and returns the next actions (a list) or [].\\\",\\n        \\\"parameters\\\": {\\\"type\\\": \\\"object\\\", \\\"properties\\\": {\\\"code\\\": {\\\"type\\\": \\\"string\\\", \\\"description\\\": \\\"Python source defining def solve(): ...\\\"}},\\n                       \\\"required\\\": [\\\"code\\\"]},\\n    },\\n}\\n\\nSYSTEM_PROMPT = \\\"\\\"\\\"You are an autonomous agent playing an unknown grid puzzle game (ARC-AGI-3). The board is a 64x64 grid of colour\\nindices 0-15, shown as ascii rows using the hex digits 0-9a-f (one character per cell, row 0 at the top). Nobody tells you the rules:\\ndiscover them by acting, then complete as many levels as possible using as FEW actions as possible (the score per level is\\n(baseline_actions/your_actions)^2, so wasted actions cost score; unfinished levels score 0).\\n\\nYou act only through the `python` tool. The sandbox is a persistent Python process with these variables and helpers:\\n- `current_frame`: Frame with `.ascii` (the board as text), `.grid` (list of 64 rows of ints), `.level`, `.step`,\\n  `.segmentation` = {'nodes': [{id, color, pixels, bbox(r0,c0,r1,c1), center(row,col), hash, children, hud}], 'adjacency': [[i,j]...], 'background'}.\\n  `hud` marks thin strips along an edge (usually a status bar / action counter, not a clickable object).\\n- `transitions`: list of Transition(action, before_frame, after_frame, changed) for every action taken so far (newest last);\\n  `level_transitions` is the part that belongs to the current level.\\n- `valid_actions`: names allowed now, among UP, DOWN, LEFT, RIGHT, SPACE, ACTION7 (a game-specific extra key: probe it once) and\\n  MOUSE (a click: {'action':'MOUSE','row':r,'col':c}).\\n- `action(x)`: execute one action or a list of actions, e.g. action('UP') or action(['LEFT','LEFT']) or action({'action':'MOUSE','row':10,'col':20}).\\n  It returns {'executed_count','board_changed','level_completed','game_over','stopped_reason'} and refreshes every variable above.\\n  Execution stops early when a level completes or the game ends, and at most 24 actions run per call. After a game over the harness resets the game for you; the level restarts.\\n- `nav`: navigation helper rebuilt from `transitions` on every call (None until the first frame). `nav.summary()` gives the avatar\\n  (the object your movement keys move), learned move deltas, floor/wall knowledge, an action gauge if the game has one, candidate targets\\n  with (row, col) and BFS path length, and the nearest unexplored block. `nav.path_to(row, col)` returns the list of moves to reach the block\\n  containing that pixel (pass it straight to action), `nav.frontier()` the nearest unexplored walkable block (row, col) or None, `nav.targets()` a list of dicts with keys\\n  row, col, color, size, path_len (None if unreachable), visited, last_visit,\\n  `nav.map()` a coarse map (P = avatar, # = wall). It learns only from real moves, so press each movement key once or twice first.\\n- `goal_hypotheses`: list of dicts the harness inferred from how earlier levels were won (type reach / collect_reach /\\n  collect_all / click_sequence with colours); the same win rule usually holds on later levels \\u2014 plan for it.\\n- `rules_text`: rules the harness induced from this level's recorded transitions (movement deltas, walls, gauge cost,\\n  refills, collectibles, click effects) with support counts; CONFIRMED rules fit every transition. Build on them.\\n- `level_recaps`: harness-written summaries of how each earlier level was won (winning action sequence, collected objects,\\n  gauge refills). Levels of one game share the rules, so reuse the strategy on the new layout.\\n- `checklist`: a dict you share with the harness: `goal` (str), `roles` (dict colour -> role), `plan` (str), `tried` (list of\\n  'what + outcome'). The harness fills the facts (controls, avatar, limits, object counts) and shows the whole checklist at the top\\n  of every turn with the first unresolved item. `goal`, `plan` and `roles` are arguments of the python tool (fill them on every\\n  call); `tried` is written by the harness from what each turn did and what happened.\\n- `notes`: a string you own. Keep the rules learned so far in it (what each key does, objects and their roles, the goal\\n  hypothesis, the next plan). It persists and is shown to you at every turn, so update it instead of re-analysing the board.\\n- `propose_solver(code)`: store a persistent solver. `code` is a string defining `def solve():` that reads the variables above and returns the\\n  next actions (a list, or [] when undecided). Optionally define `def predict(before_frame, action_name)` returning the expected next\\n  board as an ascii string; it is scored against the recorded transitions and proposals with accuracy below 0.5 are rejected.\\n  A solver whose predict() reaches accuracy >= 0.8 on at least 6 recorded transitions is VERIFIED: the harness then runs solve() on its\\n  own every turn until it returns [], raises, stops changing the board, causes a game over, or the level stops advancing. A solver\\n  without predict() (or below 0.8) is kept as a DRAFT: its suggestion is shown to you each turn, but you decide what to execute. Never define your own\\n  `propose_solver` or `action`.\\n\\nHow to work \\u2014 START FROM THE CHECKLIST, NOT FROM ZERO:\\n0. ACT EVERY TURN: each python call should execute at least one action unless the previous turn's result is still unread.\\n   Inspection alone is only acceptable on the very first turn. Every turn begins with the CHECKLIST. Do not re-derive settled items; go straight to the first unresolved one (the header names it)\\n   and spend the turn resolving it: untried key -> press it; role unknown -> touch/click that object once; goal missing -> state a\\n   hypothesis and test it; plan present -> execute it. Record outcomes in `checklist['tried']` so they are never repeated.\\n1. Look first: print `current_frame.ascii` (or parts of it) and `current_frame.segmentation` summaries. Identify the avatar, walls,\\n   collectables, doors, counters, and the HUD.\\n2. Probe cheaply: one action per movement key, one click per distinct object type, and compare `transitions[-1].before_frame.ascii`\\n   with `.after_frame.ascii` (or print `nav.summary()`). Write down what each action does.\\n3. Form a hypothesis about the goal (reach a target, collect items in some order, match a pattern, toggle objects), then act on it with\\n   short scripted sequences, not one action per turn. Use `nav.path_to` for movement instead of hand-written step lists.\\n4. When the rules are clear or a level was just completed, encode them in `propose_solver(code)` so the following levels are played\\n   programmatically and cheaply. Keep solve() deterministic and short; prefer search over hand-written sequences.\\n5. A thin strip along an edge whose length changes by a fixed amount every action is an action counter (gauge). It is never the\\n   goal: do not try to fill or empty it, and never click it. When it runs out the level restarts (game over), so plan within it.\\n6. Every tool call should either gain information or make progress. Do not repeat a probe whose result you already know. If the board\\n   stops changing, change strategy (different key, different object, the SPACE key, a different order).\\n\\nReply format \\u2014 every turn, before the tool call, write exactly these five short lines (one sentence each, no more):\\nWorld model: <what the objects are and how the board works, as far as you know>\\nGoal model: <what you believe ends the level>\\nAction model: <what each key/click does, with deltas>\\nRecent findings: <what the last turn taught you>\\nPlan: <the concrete actions this call will execute>\\nThen make exactly one tool call (`python`, or `propose_solver` when you are ready to store a solver). Except for the very first\\nturn (inspect + probe), every call must execute at least one action; prefer a whole route or a probe batch over a single step.\\nDo not restate the analysis at length: the five lines are your memory, update them and act.\\n\\\"\\\"\\\"\\n\\nNAV_TEMPLATE = '''def solve():\\n    if nav is None:\\n        return []\\n    recent = [t.action for t in level_transitions[-8:]]\\n    for a in ['UP', 'DOWN', 'LEFT', 'RIGHT']:   # learn every movement key once before routing\\n        if a in valid_actions and a not in nav.moves and a not in recent:\\n            return [a]\\n    for t in nav.targets():\\n        if not t['visited'] and t['path_len']:\\n            return nav.path_to(t['row'], t['col']) or []\\n    reach = [t for t in nav.targets() if t['path_len']]   # all visited: revisit the least recently visited\\n    if reach:\\n        t = min(reach, key=lambda t: t['last_visit'])\\n        return nav.path_to(t['row'], t['col']) or []\\n    fr = nav.frontier()\\n    return (nav.path_to(fr[0], fr[1]) or []) if fr else []\\n'''\\n\\nCLICK_TEMPLATE = '''def solve():\\n    nodes = [n for n in current_frame.segmentation['nodes'] if not n['hud'] and n['pixels'] <= 600]\\n    nodes.sort(key=lambda n: n['pixels'])\\n    clicked = {}\\n    for t in transitions:\\n        a = t.action if isinstance(t.action, dict) else None\\n        if a and a.get('action') == 'MOUSE':\\n            key = (a['row'] // 4, a['col'] // 4)\\n            clicked.setdefault(key, [0, 0]); clicked[key][0] += 1; clicked[key][1] += int(t.changed)\\n    key = lambda n: (n['center'][0] // 4, n['center'][1] // 4)\\n    fresh = [n for n in nodes if key(n) not in clicked]\\n    pool = fresh or [n for n in nodes if clicked.get(key(n), [0, 0])[1] > 0] or nodes\\n    if not pool:\\n        return []\\n    r, c = pool[0]['center']\\n    return [{'action': 'MOUSE', 'row': r, 'col': c}]\\n'''\\n\\nSOLVER_GUIDE = f\\\"\\\"\\\"\\nSolver templates (adapt the target choice to the rules you inferred):\\n- Navigation game (visit the nearest unvisited target, else explore):\\npropose_solver('''\\n{NAV_TEMPLATE}''')\\n- Click game (no avatar, MOUSE only: sweep objects, re-click the ones that changed the board, skip HUD strips):\\npropose_solver('''\\n{CLICK_TEMPLATE}''')\\n\\\"\\\"\\\"\\n\\n\\ndef system_prompt() -> str:\\n    return SYSTEM_PROMPT + SOLVER_GUIDE\\n\\n\\ndef turn_header(*, level: int, levels_total: int, actions_used: int, level_actions: int, valid_actions: list[str], budget_line: str) -> str:\\n    return (f\\\"Level {level}/{levels_total}. Actions used: {actions_used} total, {level_actions} on this level. \\\"\\n            f\\\"Valid actions now: {', '.join(valid_actions) or 'none'}. {budget_line}\\\")\\n\", \"rules.py\": \"\\\"\\\"\\\"Symbolic rule induction over recorded transitions (object level, exact fit).\\n\\nEach rule is a parametric hypothesis fitted to the transitions of the current level. A rule reports\\n`support` (transitions it explains) and `counter` (transitions it contradicts); rules with counter == 0\\nand support >= 2 are 'confirmed'. The harness shows confirmed rules and unexplained transitions to the\\nmodel, and planners may rely on confirmed rules.\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import Counter, defaultdict\\nfrom dataclasses import dataclass, field\\nfrom typing import Any, Optional\\n\\nfrom .frame import masked_ascii\\nfrom .nav import NavHelper, extract_objects\\n\\n\\n@dataclass\\nclass Rule:\\n    kind: str\\n    params: dict\\n    support: int = 0\\n    counter: int = 0\\n    examples: list = field(default_factory=list)\\n\\n    @property\\n    def confirmed(self) -> bool:\\n        return self.counter == 0 and self.support >= 2\\n\\n    def text(self) -> str:\\n        p = self.params\\n        if self.kind == \\\"move\\\":\\n            return f\\\"{p['action']} moves the avatar by (dx={p['dx']}, dy={p['dy']}) [{self.support} ok, {self.counter} blocked]\\\"\\n        if self.kind == \\\"wall\\\":\\n            return f\\\"the avatar cannot enter colour {p['color']} (blocked {self.support}x)\\\"\\n        if self.kind == \\\"gauge\\\":\\n            return f\\\"the gauge (colour {p['color']}) drops by {p['per_action']} per action\\\"\\n        if self.kind == \\\"refill\\\":\\n            return f\\\"touching colour {p['color']} refills the gauge (+{p['amount']}) and consumes it\\\"\\n        if self.kind == \\\"collect\\\":\\n            return f\\\"touching colour {p['color']} objects makes them disappear (collected, {self.support}x)\\\"\\n        if self.kind == \\\"hazard\\\":\\n            return f\\\"touching colour {p['color']} resets the avatar to its start position ({self.support}x)\\\"\\n        if self.kind == \\\"click\\\":\\n            return f\\\"clicking colour {p['color']} changes colour(s) {sorted(p['changes'])} ({self.support}x)\\\" if p[\\\"changes\\\"] else f\\\"clicking colour {p['color']} does nothing ({self.support}x)\\\"\\n        if self.kind == \\\"mirror\\\":\\n            how = {\\\"mirror_x\\\": \\\"horizontally mirrored (-dx, dy)\\\", \\\"mirror_y\\\": \\\"vertically mirrored (dx, -dy)\\\", \\\"mirror_xy\\\": \\\"point-mirrored (-dx, -dy)\\\"}[p[\\\"how\\\"]]\\n            return f\\\"a second body (colour {p['color']}, {p['size']}px) moves with every move, {how} \\u2014 walls stop each body separately ({self.support}x)\\\"\\n        if self.kind == \\\"noop\\\":\\n            return f\\\"{p['action']} changed nothing ({self.support}x)\\\"\\n        return f\\\"{self.kind} {p}\\\"\\n\\n\\ndef _objs(frame):\\n    objs, bg = extract_objects(frame.grid)\\n    return objs, bg\\n\\n\\ndef _count(frame, color):\\n    return sum(1 for row in frame.grid for v in row if v == color)\\n\\n\\ndef induce(transitions: list, current_frame) -> tuple[list[Rule], list[str]]:\\n    \\\"\\\"\\\"transitions: objects with .action (str or dict), .before_frame, .after_frame (same level).\\n    Returns (rules, unexplained) where unexplained lists transitions no rule accounts for.\\\"\\\"\\\"\\n    rules: list[Rule] = []\\n    if not transitions:\\n        return rules, []\\n    nav = NavHelper(transitions, current_frame)\\n    moves = nav.moves\\n    # 1. movement + walls\\n    per_action_ok: dict[str, int] = Counter(); per_action_blocked: dict[str, int] = Counter()\\n    wall_hits: Counter = Counter()\\n    avatar_positions = []\\n    for t in transitions:\\n        a = t.action if isinstance(t.action, str) else \\\"MOUSE\\\"\\n        if a not in moves:\\n            continue\\n        nb = NavHelper([t], t.after_frame)\\n        av_b = NavHelper(transitions, t.before_frame).avatar(); av_a = NavHelper(transitions, t.after_frame).avatar()\\n        if not av_b or not av_a:\\n            continue\\n        moved = (av_a[\\\"row\\\"] - av_b[\\\"row\\\"], av_a[\\\"col\\\"] - av_b[\\\"col\\\"]) != (0, 0)\\n        if moved:\\n            per_action_ok[a] += 1\\n        else:\\n            per_action_blocked[a] += 1\\n            dx, dy = moves[a]; r, c = av_b[\\\"row\\\"] + dy, av_b[\\\"col\\\"] + dx\\n            if 0 <= r < 64 and 0 <= c < 64:\\n                wall_hits[t.before_frame.grid[r][c]] += 1\\n        avatar_positions.append((av_b[\\\"row\\\"], av_b[\\\"col\\\"], av_a[\\\"row\\\"], av_a[\\\"col\\\"], a, t))\\n    for a, (dx, dy) in moves.items():\\n        rules.append(Rule(\\\"move\\\", {\\\"action\\\": a, \\\"dx\\\": dx, \\\"dy\\\": dy}, support=per_action_ok[a], counter=0))\\n    for color, n in wall_hits.items():\\n        rules.append(Rule(\\\"wall\\\", {\\\"color\\\": color}, support=n))\\n    # 1b. mirrored / co-moving second body: another object that moves whenever the avatar moves, with a fixed transform\\n    if moves:\\n        pairs: Counter = Counter(); seen_moves = 0\\n        for t in transitions:\\n            a = t.action if isinstance(t.action, str) else \\\"MOUSE\\\"\\n            if a not in moves:\\n                continue\\n            po, _ = _objs(t.before_frame); co, _ = _objs(t.after_frame)\\n            from .nav import _moved\\n            mv = [(o, dx, dy) for o, dx, dy in _moved(po, co) if 4 <= o[\\\"size\\\"] <= 400]   # ignore 1-3 px specks (ar25 false trigger)\\n            if len(mv) < 2:\\n                continue\\n            seen_moves += 1\\n            adx, ady = moves[a]\\n            for o, dx, dy in mv:\\n                if (dx, dy) == (adx, ady):\\n                    continue\\n                if (dx, dy) == (-adx, ady):\\n                    pairs[(\\\"mirror_x\\\", o[\\\"color\\\"], o[\\\"size\\\"])] += 1\\n                elif (dx, dy) == (adx, -ady):\\n                    pairs[(\\\"mirror_y\\\", o[\\\"color\\\"], o[\\\"size\\\"])] += 1\\n                elif (dx, dy) == (-adx, -ady):\\n                    pairs[(\\\"mirror_xy\\\", o[\\\"color\\\"], o[\\\"size\\\"])] += 1\\n        for (kind, color, size), n in pairs.items():\\n            if n >= 2 and n * 2 >= seen_moves:   # the second body must move in at least half of the observed moves\\n                rules.append(Rule(\\\"mirror\\\", {\\\"how\\\": kind, \\\"color\\\": color, \\\"size\\\": size}, support=n, counter=max(0, seen_moves - n)))\\n    # 2. gauge and refills\\n    g = nav.gauge()\\n    av0 = nav.avatar(); avatar_cols = set(av0[\\\"colors\\\"]) if av0 else set()\\n    for (color, size), _ in nav.player_votes.most_common(5):\\n        avatar_cols.add(color)\\n    if g:\\n        rules.append(Rule(\\\"gauge\\\", {\\\"color\\\": g[\\\"color\\\"], \\\"per_action\\\": g[\\\"per_action\\\"]}, support=len(transitions)))\\n        color = g[\\\"color\\\"]\\n        refills: Counter = Counter(); amounts: dict = {}\\n        for t in transitions:\\n            b, af = _count(t.before_frame, color), _count(t.after_frame, color)\\n            if af > b + 2:\\n                # a small non-avatar object that vanished where the avatar arrived\\n                av_a = NavHelper(transitions, t.after_frame).avatar()\\n                after_objs = _objs(t.after_frame)[0]\\n                gone = [o for o in _objs(t.before_frame)[0] if o[\\\"size\\\"] <= 40 and o[\\\"color\\\"] not in avatar_cols and o[\\\"color\\\"] != nav.background\\n                        and o[\\\"color\\\"] not in nav.floor_colors\\n                        and not any(p[\\\"color\\\"] == o[\\\"color\\\"] and p[\\\"center\\\"] == o[\\\"center\\\"] for p in after_objs)\\n                        and (not av_a or (abs(o[\\\"center\\\"][1] - av_a[\\\"row\\\"]) <= 6 and abs(o[\\\"center\\\"][0] - av_a[\\\"col\\\"]) <= 6))]\\n                for o in gone[:1]:\\n                    refills[o[\\\"color\\\"]] += 1; amounts[o[\\\"color\\\"]] = af - b\\n        for c, n in refills.items():\\n            rules.append(Rule(\\\"refill\\\", {\\\"color\\\": c, \\\"amount\\\": amounts[c]}, support=n))\\n    # 3. collectibles and hazards (movement games)\\n    if moves:\\n        vanish: Counter = Counter(); reset_colors: Counter = Counter()\\n        av = nav.avatar(); avatar_colors = set(av[\\\"colors\\\"]) if av else set()\\n        for (color, size), _ in nav.player_votes.most_common(5):\\n            avatar_colors.add(color)\\n        start = avatar_positions[0][:2] if avatar_positions else None\\n        for rb, cb, ra, ca, a, t in avatar_positions:\\n            before_objs, _ = _objs(t.before_frame); after_objs, _ = _objs(t.after_frame)\\n            after_keys = {(o[\\\"color\\\"], o[\\\"center\\\"]) for o in after_objs}\\n            for o in before_objs:\\n                if o[\\\"size\\\"] <= 60 and o[\\\"color\\\"] not in avatar_colors and (o[\\\"color\\\"], o[\\\"center\\\"]) not in after_keys:\\n                    # object vanished; did the avatar arrive at it?\\n                    if abs(o[\\\"center\\\"][1] - ra) <= 4 and abs(o[\\\"center\\\"][0] - ca) <= 4:\\n                        vanish[o[\\\"color\\\"]] += 1\\n            if start and (ra, ca) == start and (rb, cb) != start and abs(rb - ra) + abs(cb - ca) > 6:\\n                dx, dy = moves[a]; r, c = rb + dy, cb + dx\\n                if 0 <= r < 64 and 0 <= c < 64:\\n                    reset_colors[t.before_frame.grid[r][c]] += 1\\n        for color, n in vanish.items():\\n            if color != nav.background and color not in avatar_colors and color not in nav.floor_colors:\\n                rules.append(Rule(\\\"collect\\\", {\\\"color\\\": color}, support=n))\\n        for color, n in reset_colors.items():\\n            rules.append(Rule(\\\"hazard\\\", {\\\"color\\\": color}, support=n))\\n    # 4. click effects\\n    clicks: dict[int, Counter] = defaultdict(Counter); click_n: Counter = Counter()\\n    for t in transitions:\\n        if not isinstance(t.action, dict):\\n            continue\\n        r, c = t.action[\\\"row\\\"], t.action[\\\"col\\\"]\\n        color = t.before_frame.grid[r][c]\\n        click_n[color] += 1\\n        changed = {t.before_frame.grid[i][j] for i in range(64) for j in range(64)\\n                   if masked_ascii(t.before_frame).splitlines()[i][j] != masked_ascii(t.after_frame).splitlines()[i][j]}\\n        clicks[color].update(changed)\\n    for color, n in click_n.items():\\n        rules.append(Rule(\\\"click\\\", {\\\"color\\\": color, \\\"changes\\\": set(clicks[color])}, support=n))\\n    # 5. no-ops\\n    noop: Counter = Counter()\\n    for t in transitions:\\n        a = t.action if isinstance(t.action, str) else \\\"MOUSE\\\"\\n        if a != \\\"MOUSE\\\" and masked_ascii(t.before_frame) == masked_ascii(t.after_frame):\\n            noop[a] += 1\\n    for a, n in noop.items():\\n        if a not in moves:\\n            rules.append(Rule(\\\"noop\\\", {\\\"action\\\": a}, support=n))\\n    # unexplained: transitions with a masked change that no rule kind covers\\n    unexplained = []\\n    for i, t in enumerate(transitions):\\n        a = t.action if isinstance(t.action, str) else \\\"MOUSE\\\"\\n        if a == \\\"MOUSE\\\" or a in moves or a in noop:\\n            continue\\n        if masked_ascii(t.before_frame) != masked_ascii(t.after_frame):\\n            unexplained.append(f\\\"transition {i}: {a} changed the board but matches no known rule\\\")\\n    return rules, unexplained\\n\\n\\ndef summary(rules: list[Rule], unexplained: list[str]) -> str:\\n    conf = [r for r in rules if r.confirmed]; weak = [r for r in rules if not r.confirmed and r.support >= 1]\\n    lines = [\\\"CONFIRMED RULES (fit every recorded transition):\\\"] + [f\\\"  - {r.text()}\\\" for r in conf] if conf else [\\\"CONFIRMED RULES: none yet\\\"]\\n    if weak:\\n        lines.append(\\\"TENTATIVE (seen once or with counter-examples): \\\" + \\\"; \\\".join(r.text() for r in weak[:6]))\\n    if unexplained:\\n        lines.append(\\\"UNEXPLAINED: \\\" + \\\"; \\\".join(unexplained[:4]))\\n    return \\\"\\\\n\\\".join(lines)\\n\", \"runner.py\": \"\\\"\\\"\\\"Play a list of games (thread pool) and write an experiments-style result JSON.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport datetime as dt\\nimport json\\nimport logging\\nimport os\\nimport subprocess\\nimport sys\\nimport threading\\nfrom concurrent.futures import ThreadPoolExecutor\\nfrom pathlib import Path\\nfrom typing import Optional\\n\\nROOT = Path(__file__).resolve().parents[1]\\n\\n# default = no chain-of-thought (set C: 2.30 vs 0.9 with thinking on the 4-game set; turns are 2x cheaper); configs/think.json re-enables it\\nDEFAULT_CONFIG = {\\\"base_url\\\": \\\"http://127.0.0.1:1234/v1\\\", \\\"model\\\": \\\"local-qwen\\\", \\\"temperature\\\": 0.7, \\\"top_p\\\": 0.8, \\\"max_tokens\\\": 4096,\\n                  \\\"extra_body\\\": {\\\"chat_template_kwargs\\\": {\\\"enable_thinking\\\": False}},\\n                  \\\"max_minutes\\\": 20, \\\"max_actions\\\": 3000, \\\"max_model_turns\\\": 400, \\\"tool_timeout\\\": 30, \\\"context_tokens\\\": 32768, \\\"jobs\\\": 2}\\n\\n\\ndef _git_info() -> dict:\\n    try:\\n        rev = subprocess.check_output([\\\"git\\\", \\\"rev-parse\\\", \\\"--short\\\", \\\"HEAD\\\"], cwd=ROOT, text=True).strip()\\n        dirty = bool(subprocess.check_output([\\\"git\\\", \\\"status\\\", \\\"--porcelain\\\", \\\"arcnav\\\"], cwd=ROOT, text=True).strip())\\n        return {\\\"commit\\\": rev, \\\"dirty\\\": dirty}\\n    except Exception:\\n        return {}\\n\\n\\ndef make_arcade(environments_dir: Optional[str] = None, *, competition: bool = False, base_url: Optional[str] = None):\\n    import arc_agi\\n    from arc_agi import OperationMode\\n    os.environ.setdefault(\\\"ARC_API_KEY\\\", \\\"local-dev\\\")\\n    if competition:\\n        return arc_agi.Arcade(operation_mode=OperationMode.COMPETITION, arc_base_url=base_url or os.environ.get(\\\"ARC_BASE_URL\\\", \\\"http://gateway:8001/\\\"))\\n    return arc_agi.Arcade(operation_mode=OperationMode.OFFLINE, environments_dir=environments_dir or str(ROOT / \\\"environment_files\\\"))\\n\\n\\ndef play_game(arc, game_id: str, cfg: dict, log_dir: Path, *, lock: threading.Lock) -> dict:\\n    from .agent import GameSession\\n    from .llm import ChatClient\\n    client = None if cfg.get(\\\"no_model\\\") else ChatClient(cfg[\\\"base_url\\\"], cfg[\\\"model\\\"], temperature=cfg[\\\"temperature\\\"], top_p=cfg[\\\"top_p\\\"],\\n                                                          max_tokens=cfg[\\\"max_tokens\\\"], extra_body=cfg.get(\\\"extra_body\\\") or {})\\n    with lock:\\n        env = arc.make(game_id)\\n    if env is None:\\n        return {\\\"game_id\\\": game_id, \\\"error\\\": \\\"could not create env\\\"}\\n    sess = GameSession(env, game_id, client, log_dir=log_dir, max_minutes=cfg[\\\"max_minutes\\\"], max_actions=cfg[\\\"max_actions\\\"],\\n                       max_model_turns=cfg[\\\"max_model_turns\\\"], tool_timeout=cfg[\\\"tool_timeout\\\"], context_tokens=cfg[\\\"context_tokens\\\"],\\n                       verbose=cfg.get(\\\"verbose\\\", True), deadline=cfg.get(\\\"deadline\\\"), think_first_turns=int(cfg.get(\\\"think_first_turns\\\", 0)), tool_choice_required=bool(cfg.get(\\\"tool_choice_required\\\", False)),\\n                       oracle_rules=(cfg.get(\\\"oracle_rules\\\") or {}).get(game_id.split(\\\"-\\\")[0], \\\"\\\"), image_context=bool(cfg.get(\\\"image_context\\\", False)))\\n    try:\\n        return sess.play()\\n    except Exception as e:\\n        logging.exception(\\\"game %s crashed\\\", game_id)\\n        sess.sandbox.close()\\n        return {\\\"game_id\\\": game_id, \\\"error\\\": f\\\"{type(e).__name__}: {e}\\\", \\\"actions\\\": sess.actions_used, \\\"levels_completed\\\": sess.level - 1}\\n\\n\\ndef run(game_ids: list[str], cfg: dict, *, out_dir: Path, tag: str = \\\"\\\", environments_dir: Optional[str] = None, arc=None) -> dict:\\n    run_id = dt.datetime.now(dt.timezone.utc).strftime(\\\"%Y%m%d-%H%M%S\\\") + f\\\"-{os.getpid()}\\\"\\n    log_dir = out_dir / \\\"logs\\\" / run_id\\n    arc = arc or make_arcade(environments_dir)\\n    lock = threading.Lock()\\n    started = dt.datetime.now(dt.timezone.utc)\\n    with ThreadPoolExecutor(max_workers=int(cfg.get(\\\"jobs\\\", 2))) as ex:\\n        games = list(ex.map(lambda g: play_game(arc, g, cfg, log_dir, lock=lock), game_ids))\\n    try:\\n        sc = arc.get_scorecard(); scd = sc.model_dump() if hasattr(sc, \\\"model_dump\\\") else {}\\n    except Exception as e:  # the competition gateway scores on its own side\\n        scd = {\\\"error\\\": repr(e)}\\n    by_id = {}\\n    for e in scd.get(\\\"environments\\\", []) or []:\\n        if e.get(\\\"runs\\\"):\\n            by_id[e[\\\"id\\\"].split(\\\"-\\\")[0]] = e[\\\"runs\\\"][-1]\\n    for g in games:\\n        r = by_id.get(g[\\\"game_id\\\"].split(\\\"-\\\")[0])\\n        if r:\\n            g[\\\"score\\\"] = r.get(\\\"score\\\"); g[\\\"level_scores\\\"] = r.get(\\\"level_scores\\\"); g[\\\"level_baseline_actions\\\"] = r.get(\\\"level_baseline_actions\\\")\\n    scores = [g.get(\\\"score\\\") or 0.0 for g in games]\\n    result = {\\\"schema\\\": 1, \\\"experiment\\\": \\\"arcnav\\\", \\\"run_id\\\": run_id, \\\"started_at\\\": started.isoformat(),\\n              \\\"finished_at\\\": dt.datetime.now(dt.timezone.utc).isoformat(), \\\"git\\\": _git_info(), \\\"tag\\\": tag,\\n              \\\"config\\\": {\\\"description\\\": \\\"arcnav: tool-using LLM harness with nav helper and solver synthesis\\\", \\\"games\\\": game_ids, \\\"params\\\": cfg},\\n              \\\"aggregate\\\": {\\\"score\\\": sum(scores) / max(1, len(scores)), \\\"levels_completed\\\": sum(g.get(\\\"levels_completed\\\", 0) for g in games),\\n                            \\\"games_level2plus\\\": sum(1 for g in games if g.get(\\\"levels_completed\\\", 0) >= 2),\\n                            \\\"actions\\\": sum(g.get(\\\"actions\\\", 0) for g in games), \\\"games_played\\\": len(games)},\\n              \\\"games\\\": games, \\\"scorecard\\\": scd}\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    out = out_dir / f\\\"run-{run_id}.json\\\"\\n    out.write_text(json.dumps(result, indent=2, default=str))\\n    print(f\\\"\\\\n========= arcnav {tag} =========\\\")\\n    for g in games:\\n        print(f\\\"  {g['game_id']:8} levels={g.get('levels_completed', '?'):>3}/{g.get('levels_total', '?')} actions={g.get('actions', '?'):>5} \\\"\\n              f\\\"score={g.get('score', 0) or 0:.3f} turns={g.get('model_turns', '?')} solver={g.get('solver_stored', '?')} stop={g.get('stop_reason', g.get('error'))}\\\")\\n    print(f\\\"Aggregate score: {result['aggregate']['score']:.4f} | games with level>=2: {result['aggregate']['games_level2plus']}/{len(games)}\\\\nResult saved: {out}\\\")\\n    return result\\n\\n\\ndef main(argv=None) -> None:\\n    ap = argparse.ArgumentParser(description=\\\"Play ARC-AGI-3 games with the arcnav agent\\\")\\n    ap.add_argument(\\\"--games\\\", default=\\\"ls20\\\", help=\\\"comma-separated game ids or 'all'\\\")\\n    ap.add_argument(\\\"--config\\\", help=\\\"JSON file overriding DEFAULT_CONFIG\\\")\\n    ap.add_argument(\\\"--out\\\", default=str(ROOT / \\\"experiments\\\" / \\\"arcnav\\\" / \\\"results\\\"))\\n    ap.add_argument(\\\"--tag\\\", default=\\\"\\\")\\n    ap.add_argument(\\\"--minutes\\\", type=float); ap.add_argument(\\\"--jobs\\\", type=int); ap.add_argument(\\\"--max-actions\\\", type=int)\\n    ap.add_argument(\\\"--no-model\\\", action=\\\"store_true\\\", help=\\\"only the environment/sandbox path (smoke test)\\\")\\n    ap.add_argument(\\\"--quiet\\\", action=\\\"store_true\\\")\\n    a = ap.parse_args(argv)\\n    logging.basicConfig(level=logging.WARNING)\\n    cfg = dict(DEFAULT_CONFIG)\\n    if a.config:\\n        cfg.update(json.loads(Path(a.config).read_text()))\\n    if a.minutes: cfg[\\\"max_minutes\\\"] = a.minutes\\n    if a.jobs: cfg[\\\"jobs\\\"] = a.jobs\\n    if a.max_actions: cfg[\\\"max_actions\\\"] = a.max_actions\\n    cfg[\\\"no_model\\\"] = a.no_model; cfg[\\\"verbose\\\"] = not a.quiet\\n    arc = make_arcade()\\n    if a.games == \\\"all\\\":\\n        games = sorted(e.game_id.split(\\\"-\\\")[0] for e in arc.get_environments())\\n    else:\\n        games = [g.strip() for g in a.games.split(\\\",\\\") if g.strip()]\\n    run(games, cfg, out_dir=Path(a.out), tag=a.tag, arc=arc)\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"sandbox.py\": \"\\\"\\\"\\\"Python tool sandbox: runs model-written code in a separate `python -I`\\nprocess with a JSON-lines protocol. Inside the child the code sees\\n`current_frame`, `history`, `transitions`, `valid_actions`, `nav`,\\n`action(...)` (executes actions on the host and refreshes state) and\\n`propose_solver(code)` (stores a persistent `solve()` on the host).\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport os\\nimport subprocess\\nimport sys\\nimport threading\\nimport time\\nfrom pathlib import Path\\nfrom typing import Any, Callable, Optional\\n\\n_PKG = Path(__file__).resolve().parent\\n_SANDBOX_SRC_FILES = (\\\"frame.py\\\", \\\"nav.py\\\")\\n\\nCHILD_PROGRAM = r'''\\nimport ast, json, os, resource, signal, sys, traceback, types\\n_out = os.fdopen(os.dup(1), \\\"w\\\"); _in = sys.stdin; _err = sys.stderr\\n_devnull = os.open(os.devnull, os.O_WRONLY); os.dup2(_devnull, 1)   # fd 1 is no longer the protocol channel\\nsys.stdout = open(os.devnull, \\\"w\\\")\\n\\ndef _send(obj):\\n    _out.write(json.dumps(obj) + \\\"\\\\n\\\"); _out.flush()\\n\\ndef _recv():\\n    line = _in.readline()\\n    if not line:\\n        raise SystemExit(0)\\n    return json.loads(line)\\n\\n_init = _recv()\\n_mods = {}\\nfor _name, _src in _init[\\\"sources\\\"].items():\\n    _m = types.ModuleType(\\\"arcnav.\\\" + _name); sys.modules[\\\"arcnav.\\\" + _name] = _m\\n    exec(compile(_src, _name + \\\".py\\\", \\\"exec\\\"), _m.__dict__); _mods[_name] = _m\\nos.dup2(_devnull, 2); sys.stderr = open(os.devnull, \\\"w\\\")\\nFrame = _mods[\\\"frame\\\"].Frame\\nNavHelper = _mods[\\\"nav\\\"].NavHelper\\n\\nclass Transition:\\n    \\\"\\\"\\\"One executed action: `action` (name or MOUSE(row=r, col=c)), `before_frame`, `after_frame`, `changed`.\\\"\\\"\\\"\\n    def __init__(self, action, before_frame, after_frame, result=None):\\n        self.action, self.before_frame, self.after_frame, self.result = action, before_frame, after_frame, result or {}\\n    @property\\n    def changed(self):\\n        return self.before_frame.ascii != self.after_frame.ascii\\n    def __repr__(self):\\n        return f\\\"Transition({self.action}, changed={self.changed}, level={self.after_frame.level})\\\"\\n\\nG = {\\\"__name__\\\": \\\"__main__\\\", \\\"Frame\\\": Frame, \\\"NavHelper\\\": NavHelper, \\\"Transition\\\": Transition}\\n\\ndef _refresh(state):\\n    G[\\\"current_frame\\\"] = Frame.from_payload(state[\\\"frame\\\"]) if state.get(\\\"frame\\\") else None\\n    G[\\\"valid_actions\\\"] = list(state.get(\\\"valid_actions\\\") or [])\\n    G[\\\"level\\\"] = int(state.get(\\\"level\\\", 1)); G[\\\"levels_total\\\"] = int(state.get(\\\"levels_total\\\", 0))\\n    trans = []\\n    for t in state.get(\\\"transitions\\\") or []:\\n        trans.append(Transition(t[\\\"action\\\"], Frame.from_payload(t[\\\"before\\\"]), Frame.from_payload(t[\\\"after\\\"]), t.get(\\\"result\\\")))\\n    G[\\\"transitions\\\"] = trans\\n    G[\\\"history\\\"] = trans  # alias\\n    G[\\\"last_action_result\\\"] = state.get(\\\"last_action_result\\\")\\n    G[\\\"level_recaps\\\"] = list(state.get(\\\"level_recaps\\\") or [])\\n    G[\\\"rules_text\\\"] = state.get(\\\"rules_text\\\") or \\\"\\\"\\n    G[\\\"goal_hypotheses\\\"] = list(state.get(\\\"goal_hypotheses\\\") or [])\\n    if \\\"notes\\\" not in G or not G[\\\"notes\\\"]:\\n        G[\\\"notes\\\"] = state.get(\\\"notes\\\") or \\\"\\\"\\n    if not isinstance(G.get(\\\"checklist\\\"), dict) or not G[\\\"checklist\\\"]:\\n        G[\\\"checklist\\\"] = dict(state.get(\\\"checklist\\\") or {\\\"goal\\\": \\\"\\\", \\\"roles\\\": {}, \\\"plan\\\": \\\"\\\", \\\"tried\\\": []})\\n    try:\\n        cur = [t for t in trans if t.before_frame.level == t.after_frame.level == G[\\\"level\\\"]]\\n        G[\\\"level_transitions\\\"] = cur\\n        G[\\\"nav\\\"] = NavHelper(cur, G[\\\"current_frame\\\"]) if G[\\\"current_frame\\\"] is not None else None\\n        G[\\\"nav_error\\\"] = None\\n    except Exception as e:  # pragma: no cover\\n        G[\\\"nav\\\"], G[\\\"nav_error\\\"] = None, repr(e)\\n\\ndef _normalize(actions):\\n    if isinstance(actions, (str, dict)):\\n        actions = [actions]\\n    if not isinstance(actions, (list, tuple)) or not actions:\\n        raise ValueError(\\\"action(actions) expects an action or a non-empty list of actions\\\")\\n    out = []\\n    for a in actions:\\n        if isinstance(a, str):\\n            if a.strip().upper() in (\\\"MOUSE\\\", \\\"CLICK\\\", \\\"ACTION6\\\"):\\n                raise ValueError(\\\"MOUSE needs coordinates: use {'action': 'MOUSE', 'row': r, 'col': c}\\\")\\n            out.append({\\\"action\\\": a.strip().upper()})\\n        elif isinstance(a, dict):\\n            d = {k: v for k, v in a.items()}; d[\\\"action\\\"] = str(d.get(\\\"action\\\", \\\"\\\")).strip().upper()\\n            if d[\\\"action\\\"] in (\\\"MOUSE\\\", \\\"CLICK\\\", \\\"ACTION6\\\"):\\n                d[\\\"action\\\"] = \\\"MOUSE\\\"\\n                if \\\"row\\\" not in d or \\\"col\\\" not in d:\\n                    raise ValueError(\\\"MOUSE needs row and col (0-63)\\\")\\n                d[\\\"row\\\"], d[\\\"col\\\"] = int(d[\\\"row\\\"]), int(d[\\\"col\\\"])\\n                if not (0 <= d[\\\"row\\\"] <= 63 and 0 <= d[\\\"col\\\"] <= 63):\\n                    raise ValueError(f\\\"MOUSE coordinates must be within 0..63 (got row={d['row']}, col={d['col']})\\\")\\n            out.append(d)\\n        else:\\n            raise ValueError(f\\\"unsupported action {a!r}\\\")\\n    return out\\n\\ndef action(actions):\\n    \\\"\\\"\\\"Execute one or more actions on the game and refresh all runtime variables.\\n    Returns {'executed_count', 'board_changed', 'level_completed', 'game_over', 'stopped_reason', 'results'}.\\\"\\\"\\\"\\n    acts = _normalize(actions)\\n    _send({\\\"type\\\": \\\"action\\\", \\\"actions\\\": acts})\\n    reply = _recv()\\n    if reply.get(\\\"type\\\") != \\\"action_result\\\":\\n        raise RuntimeError(\\\"protocol error\\\")\\n    _refresh(reply[\\\"state\\\"])\\n    return reply[\\\"result\\\"]\\n\\ndef propose_solver(code, verify_last=12):\\n    \\\"\\\"\\\"Store a persistent solver. `code` must define `def solve():` returning the next actions (list) or [].\\n    Optional `def predict(before_frame, action_name)` -> ascii string is checked against recent transitions.\\\"\\\"\\\"\\n    if not isinstance(code, str):\\n        raise TypeError(\\\"propose_solver(code) expects a string of python code\\\")\\n    report = {\\\"ok\\\": False}\\n    try:\\n        tree = ast.parse(code)\\n    except SyntaxError as e:\\n        report[\\\"reason\\\"] = f\\\"syntax error: {e}\\\"; _send({\\\"type\\\": \\\"solver\\\", \\\"code\\\": code, \\\"report\\\": report}); return _recv().get(\\\"result\\\", report)\\n    names = {n.name for n in tree.body if isinstance(n, ast.FunctionDef)}\\n    if \\\"solve\\\" not in names:\\n        report[\\\"reason\\\"] = \\\"code must define `def solve():`\\\"; _send({\\\"type\\\": \\\"solver\\\", \\\"code\\\": code, \\\"report\\\": report}); return _recv().get(\\\"result\\\", report)\\n    if \\\"propose_solver\\\" in names or \\\"action\\\" in names:\\n        report[\\\"reason\\\"] = \\\"do not redefine the built-ins `propose_solver` / `action`\\\"; _send({\\\"type\\\": \\\"solver\\\", \\\"code\\\": code, \\\"report\\\": report}); return _recv().get(\\\"result\\\", report)\\n    ns = dict(G)\\n    try:\\n        exec(compile(code, \\\"solver.py\\\", \\\"exec\\\"), ns)\\n        dry = ns[\\\"solve\\\"]()\\n        dry = list(dry or [])\\n        _normalize(dry) if dry else None\\n        report[\\\"dry_run_actions\\\"] = [a if isinstance(a, str) else dict(a) for a in dry[:8]]\\n    except Exception as e:\\n        report[\\\"reason\\\"] = f\\\"solve() raised on dry run: {type(e).__name__}: {e}\\\"\\n        _send({\\\"type\\\": \\\"solver\\\", \\\"code\\\": code, \\\"report\\\": report}); return _recv().get(\\\"result\\\", report)\\n    if \\\"predict\\\" in names and G.get(\\\"transitions\\\"):\\n        checked = G[\\\"transitions\\\"][-verify_last:]; hits = 0; mism = []\\n        for t in checked:\\n            try:\\n                p = ns[\\\"predict\\\"](t.before_frame, str(t.action))\\n            except Exception as e:\\n                p = None; mism.append(f\\\"predict raised {type(e).__name__}: {e}\\\")\\n            if p is None:\\n                continue\\n            if str(p).strip() == t.after_frame.ascii.strip():\\n                hits += 1\\n            else:\\n                exp, got = t.after_frame.ascii.splitlines(), str(p).strip().splitlines()\\n                bad = [i for i in range(min(len(exp), len(got))) if exp[i] != got[i]][:3]\\n                mism.append(f\\\"{t.action}: rows differ at {bad}\\\")\\n        report[\\\"predict_accuracy\\\"] = hits / max(1, len(checked)); report[\\\"predict_mismatches\\\"] = mism[:5]\\n        if report[\\\"predict_accuracy\\\"] < 0.5:\\n            report[\\\"reason\\\"] = f\\\"predict() accuracy {report['predict_accuracy']:.2f} < 0.5 \\u2014 fix the world model first\\\"\\n            _send({\\\"type\\\": \\\"solver\\\", \\\"code\\\": code, \\\"report\\\": report}); return _recv().get(\\\"result\\\", report)\\n    report[\\\"ok\\\"] = True\\n    report[\\\"verified\\\"] = bool(\\\"predict\\\" in names and report.get(\\\"predict_accuracy\\\", 0) >= 0.8 and len(G.get(\\\"transitions\\\") or []) >= 6)\\n    exec(compile(code, \\\"solver.py\\\", \\\"exec\\\"), G)   # accepted: solve()/predict() live in the runtime namespace\\n    _send({\\\"type\\\": \\\"solver\\\", \\\"code\\\": code, \\\"report\\\": report})\\n    return _recv().get(\\\"result\\\", report)\\n\\nG[\\\"action\\\"] = action; G[\\\"propose_solver\\\"] = propose_solver\\n_refresh(_init[\\\"state\\\"])\\n\\ndef _run(code):\\n    tree = ast.parse(code)\\n    if tree.body and isinstance(tree.body[-1], ast.Expr):\\n        last = ast.Expression(tree.body.pop().value)\\n        exec(compile(tree, \\\"<tool>\\\", \\\"exec\\\"), G)\\n        v = eval(compile(last, \\\"<tool>\\\", \\\"eval\\\"), G)\\n        if v is not None:\\n            print(repr(v), file=_cap)\\n    else:\\n        exec(compile(tree, \\\"<tool>\\\", \\\"exec\\\"), G)\\n\\nimport io\\n_cap = io.StringIO()\\nresource.setrlimit(resource.RLIMIT_AS, (4 << 30, 4 << 30))\\nwhile True:\\n    msg = _recv()\\n    if msg.get(\\\"type\\\") == \\\"state\\\":\\n        _refresh(msg[\\\"state\\\"]); continue\\n    if msg.get(\\\"type\\\") != \\\"run\\\":\\n        break\\n    _cap = io.StringIO(); sys.stdout = sys.stderr = _cap\\n    signal.signal(signal.SIGALRM, lambda *_: (_ for _ in ()).throw(TimeoutError(\\\"tool timeout\\\")))\\n    signal.alarm(int(msg.get(\\\"timeout\\\", 30)))\\n    err = None\\n    try:\\n        _run(msg[\\\"code\\\"])\\n    except SystemExit:\\n        pass\\n    except BaseException as e:\\n        tb = traceback.format_exception_only(type(e), e)\\n        err = \\\"\\\".join(tb).strip()\\n        if isinstance(e, TimeoutError):\\n            err = \\\"TimeoutError: tool timeout\\\"\\n    finally:\\n        signal.alarm(0)\\n    sys.stdout = sys.stderr = open(os.devnull, \\\"w\\\")\\n    try:\\n        _ck = json.loads(json.dumps(G.get(\\\"checklist\\\"), default=str))[:1] if isinstance(G.get(\\\"checklist\\\"), list) else json.loads(json.dumps(G.get(\\\"checklist\\\"), default=str))\\n    except Exception:\\n        _ck = {}\\n    _send({\\\"type\\\": \\\"done\\\", \\\"stdout\\\": _cap.getvalue(), \\\"error\\\": err, \\\"notes\\\": str(G.get(\\\"notes\\\") or \\\"\\\")[:3000], \\\"checklist\\\": _ck if isinstance(_ck, dict) else {}})\\n'''\\n\\n\\ndef _sources() -> dict[str, str]:\\n    return {name[:-3]: (_PKG / name).read_text() for name in _SANDBOX_SRC_FILES}\\n\\n\\nclass Sandbox:\\n    \\\"\\\"\\\"A persistent child interpreter for one game. `action_handler(actions) -> (result, state)`;\\n    `solver_handler(code, report) -> result`; `state_provider() -> state payload`.\\\"\\\"\\\"\\n\\n    def __init__(self, *, action_handler: Callable[[list[dict]], tuple[dict, dict]],\\n                 solver_handler: Callable[[str, dict], dict], state_provider: Callable[[], dict],\\n                 timeout: int = 30, max_output_chars: int = 6000):\\n        self.action_handler, self.solver_handler, self.state_provider = action_handler, solver_handler, state_provider\\n        self.timeout, self.max_output_chars = timeout, max_output_chars\\n        self.proc: Optional[subprocess.Popen] = None\\n\\n    def _start(self) -> None:\\n        env = {\\\"PATH\\\": os.environ.get(\\\"PATH\\\", \\\"\\\"), \\\"PYTHONHASHSEED\\\": \\\"0\\\", \\\"HOME\\\": os.environ.get(\\\"HOME\\\", \\\"/tmp\\\")}\\n        self.proc = subprocess.Popen([sys.executable, \\\"-I\\\", \\\"-c\\\", CHILD_PROGRAM], stdin=subprocess.PIPE, stdout=subprocess.PIPE,\\n                                     stderr=subprocess.DEVNULL, text=True, env=env, start_new_session=True)\\n        self._send({\\\"sources\\\": _sources(), \\\"state\\\": self.state_provider()})\\n\\n    def _send(self, obj: dict) -> None:\\n        assert self.proc and self.proc.stdin\\n        self.proc.stdin.write(json.dumps(obj) + \\\"\\\\n\\\"); self.proc.stdin.flush()\\n\\n    def _recv(self) -> dict:\\n        assert self.proc and self.proc.stdout\\n        line = self.proc.stdout.readline()\\n        if not line:\\n            raise RuntimeError(\\\"sandbox died\\\")\\n        return json.loads(line)\\n\\n    def close(self) -> None:\\n        if self.proc:\\n            try:\\n                os.killpg(self.proc.pid, 9)\\n            except Exception:\\n                pass\\n            self.proc = None\\n\\n    def run(self, code: str, *, timeout: Optional[int] = None) -> dict:\\n        \\\"\\\"\\\"Run tool code; returns {'stdout', 'error', 'actions_executed', 'proposals': [...]}\\\"\\\"\\\"\\n        if self.proc is None or self.proc.poll() is not None:\\n            self.close(); self._start()\\n        else:\\n            self._send({\\\"type\\\": \\\"state\\\", \\\"state\\\": self.state_provider()})\\n        self._send({\\\"type\\\": \\\"run\\\", \\\"code\\\": code, \\\"timeout\\\": timeout or self.timeout})\\n        executed, proposals = 0, []\\n        deadline = time.time() + (timeout or self.timeout) + 15\\n        while True:\\n            try:\\n                msg = self._recv()\\n            except RuntimeError:\\n                self.close()\\n                return {\\\"stdout\\\": \\\"\\\", \\\"error\\\": \\\"RuntimeError: python tool process died (memory/time limit?)\\\", \\\"actions_executed\\\": executed, \\\"proposals\\\": proposals}\\n            t = msg.get(\\\"type\\\")\\n            if t == \\\"action\\\":\\n                result, state = self.action_handler(msg[\\\"actions\\\"])\\n                executed += int(result.get(\\\"executed_count\\\", 0))\\n                self._send({\\\"type\\\": \\\"action_result\\\", \\\"result\\\": result, \\\"state\\\": state})\\n            elif t == \\\"solver\\\":\\n                res = self.solver_handler(msg[\\\"code\\\"], msg[\\\"report\\\"]); proposals.append(res)\\n                self._send({\\\"type\\\": \\\"solver_result\\\", \\\"result\\\": res})\\n            elif t == \\\"done\\\":\\n                out = msg.get(\\\"stdout\\\", \\\"\\\")\\n                if len(out) > self.max_output_chars:\\n                    out = out[: self.max_output_chars // 2] + \\\"\\\\n...[truncated]...\\\\n\\\" + out[-self.max_output_chars // 2:]\\n                return {\\\"stdout\\\": out, \\\"error\\\": msg.get(\\\"error\\\"), \\\"actions_executed\\\": executed, \\\"proposals\\\": proposals, \\\"notes\\\": msg.get(\\\"notes\\\", \\\"\\\"),\\n                        \\\"checklist\\\": msg.get(\\\"checklist\\\") or {}}\\n            if time.time() > deadline:\\n                self.close()\\n                return {\\\"stdout\\\": \\\"\\\", \\\"error\\\": \\\"TimeoutError: tool exceeded its time limit\\\", \\\"actions_executed\\\": executed, \\\"proposals\\\": proposals}\\n\", \"solver.py\": \"\\\"\\\"\\\"Persistent-solver policy: when the model has stored a `solve()`, the host\\nruns it turn after turn without the model, and stops on clear failure signals.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport json\\nfrom typing import Optional\\n\\nMAX_ACTIONS_PER_TURN = 12\\nNOOP_TURN_LIMIT = 2          # consecutive solver turns without a board change\\nNO_PROGRESS_ACTIONS = 80     # solver actions without a level advance\\nCYCLE_WINDOW = 6             # identical board seen this many times -> cycling\\nDEMAND_EVERY_TURNS: Optional[int] = None   # periodic proposal demand disabled (hurt free play in v012e/f)\\n\\nRUN_SNIPPET = \\\"\\\"\\\"\\nexec(compile(__solver_code, \\\"solver.py\\\", \\\"exec\\\"), globals())\\n__acts = list(solve() or [])\\nif not __acts:\\n    print(\\\"SOLVER_EMPTY\\\")\\nelse:\\n    __r = action(__acts[:__solver_budget])\\n    print(\\\"SOLVER_RAN\\\", len(__acts), __r.get(\\\"executed_count\\\"), __r.get(\\\"board_changed\\\"), __r.get(\\\"level_completed\\\"), __r.get(\\\"game_over\\\"))\\n\\\"\\\"\\\"\\n\\n\\ndef run_snippet(code: str, budget: int = MAX_ACTIONS_PER_TURN) -> str:\\n    import json as _j\\n    return f\\\"__solver_code = {_j.dumps(code)}\\\\n__solver_budget = {int(budget)}\\\\n\\\" + RUN_SNIPPET\\n\\n\\nVERIFY_TO_AUTORUN = True   # iter2: only solvers whose predict() scores >= 0.8 on >= 6 transitions run without the model\\n\\n\\ndef new_solver(code: str, report: dict) -> dict:\\n    status = \\\"active\\\" if (report.get(\\\"verified\\\") or not VERIFY_TO_AUTORUN) else \\\"draft\\\"\\n    return {\\\"code\\\": code, \\\"report\\\": report, \\\"status\\\": status, \\\"turns\\\": 0, \\\"actions_run\\\": 0, \\\"noop_turns\\\": 0,\\n            \\\"no_progress_actions\\\": 0, \\\"reason\\\": None, \\\"failure_shown\\\": 0, \\\"board_seen\\\": {}}\\n\\n\\ndef status_lines(solver: Optional[dict], *, model_turns: int, level_just_completed: bool) -> list[str]:\\n    if not solver:\\n        base = \\\"Solver: none stored yet.\\\"\\n        if level_just_completed:\\n            return [base + \\\" You just completed a level, so you know the rules: in THIS turn call propose_solver(code) with a solve() that \\\"\\n                    \\\"reproduces what worked (see the templates in the system prompt) and let the harness play the new level.\\\"]\\n        if DEMAND_EVERY_TURNS is not None and model_turns >= DEMAND_EVERY_TURNS:\\n            return [base + f\\\" {model_turns} thinking turns spent: encode your current plan in propose_solver(code) now; the harness reports failures.\\\"]\\n        return [base + \\\" When the rules are clear, call propose_solver(code) so the harness can play on without you.\\\"]\\n    st = solver.get(\\\"status\\\")\\n    if st == \\\"draft\\\":\\n        return [f\\\"Solver: a DRAFT is stored (not verified, so the harness does not run it by itself). Its suggestion for the current board: \\\"\\n                f\\\"{solver.get('suggestion', '?')}. To let it run automatically, add `def predict(before_frame, action_name)` that returns the \\\"\\n                \\\"expected next board (ascii) and re-propose: accuracy >= 0.8 on the recorded transitions makes it verified. Or execute the suggestion yourself.\\\"]\\n    if st == \\\"failed\\\":\\n        shown = solver.get(\\\"failure_shown\\\", 0); solver[\\\"failure_shown\\\"] = shown + 1\\n        head = (f\\\"Solver: your stored solver FAILED ({solver.get('reason')}) after {solver.get('actions_run', 0)} actions. \\\"\\n                \\\"Act manually for a few turns to learn what was wrong; when you know, repair the code and call propose_solver(code) again.\\\")\\n        return [head, f\\\"Failed solver code:\\\\n{solver.get('code', '')[:1500]}\\\"] if shown == 0 else [head]\\n    if st == \\\"rejected\\\":\\n        return [f\\\"Solver: your last proposal was rejected: {solver.get('reason')}. Report: {json.dumps(solver.get('report', {}))[:600]}\\\"]\\n    return []\\n\"}")
os.makedirs(f'{WORK}/arcnav', exist_ok=True)
for name, body in SOURCES.items():
    open(f'{WORK}/arcnav/{name}', 'w').write(body)
sys.path.insert(0, WORK)
import arcnav, arc_agi
print('arcnav', arcnav.__version__, '| arc_agi ok |', f'{time.time()-T0:.0f}s')


In [ ]:
# In-notebook vLLM OpenAI server from the attached wheelhouse (installed into a private target dir so the
# notebook kernel's own packages stay untouched) serving the attached HF snapshot.
import glob, os, shutil, subprocess, sys, time, urllib.request, json
WORK = '/kaggle/working'
def _find(ref):
    owner, slug = ref.split('/', 1)
    for p in (f'/kaggle/input/{slug}', f'/kaggle/input/datasets/{owner}/{slug}'):
        if os.path.exists(p):
            return p
    raise FileNotFoundError(ref)
WHEELHOUSE = _find('driessmit1/arc3-vllm-h100-wheelhouse-v3')
_mp = glob.glob('/kaggle/input/models/**/config.json', recursive=True)
MODEL_PATH = os.path.dirname(_mp[0]) if (False and _mp) else _find('driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot')
cfgs = glob.glob(MODEL_PATH + '/**/config.json', recursive=True)
MODEL_PATH = os.path.dirname(cfgs[0]) if cfgs else MODEL_PATH
SITE = '/tmp/vllm-site-packages'   # outside /kaggle/working so the kernel output stays small
print('wheelhouse:', WHEELHOUSE, '| model:', MODEL_PATH)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())
if not os.path.exists(SITE + '/vllm'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', WHEELHOUSE, '--requirement', WHEELHOUSE + '/requirements.lock',
                    '--target', SITE, '--upgrade', '--ignore-installed', '--only-binary', ':all:', '--no-compile', '--disable-pip-version-check',
                    '--no-warn-conflicts', '-q'], check=True)
# FlashInfer JIT-compiles sm120 kernels and links -lcuda: the driver stub lives in /usr/local/nvidia/lib64 on Kaggle
env = dict(os.environ, PYTHONPATH=SITE, USE_TF='0', TRANSFORMERS_NO_TF='1', TRANSFORMERS_NO_TORCHVISION='1', VLLM_NO_USAGE_STATS='1', **{},
           LIBRARY_PATH='/usr/local/nvidia/lib64:' + os.environ.get('LIBRARY_PATH', ''),
           LD_LIBRARY_PATH='/usr/local/nvidia/lib64:' + os.environ.get('LD_LIBRARY_PATH', ''))
cmd = [sys.executable, '-m', 'vllm.entrypoints.openai.api_server', '--model', MODEL_PATH, '--served-model-name', 'local-model',
       '--host', '127.0.0.1', '--port', '1234', '--max-model-len', '65536', '--gpu-memory-utilization', '0.92',
       '--enable-auto-tool-choice', '--enable-prefix-caching', '--generation-config', 'vllm'] + ['--tool-call-parser', 'qwen3_coder', '--reasoning-parser', 'qwen3', '--default-chat-template-kwargs', '{"preserve_thinking": true}']
if env.get('TIKTOKEN_RS_CACHE_DIR') and not os.path.isdir(env['TIKTOKEN_RS_CACHE_DIR']):
    _alt = '/kaggle/input/' + env['TIKTOKEN_RS_CACHE_DIR'].rstrip('/').split('/')[-1]
    env['TIKTOKEN_RS_CACHE_DIR'] = _alt if os.path.isdir(_alt) else env['TIKTOKEN_RS_CACHE_DIR']
    print('tiktoken cache dir:', env['TIKTOKEN_RS_CACHE_DIR'], os.listdir(env['TIKTOKEN_RS_CACHE_DIR']) if os.path.isdir(env['TIKTOKEN_RS_CACHE_DIR']) else 'MISSING')
log = open(f'{WORK}/vllm-server.log', 'w')
VLLM = subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT)
t0 = time.time()
while True:
    if VLLM.poll() is not None:
        _log = open(f'{WORK}/vllm-server.log').read()
        _err = [l for l in _log.splitlines() if any(k in l for k in ('FAILED', 'error:', 'Error', 'cannot find', 'RuntimeError', 'assert'))]
        raise RuntimeError('vLLM exited. Error lines:\n' + '\n'.join(_err[:40]) + '\n--- tail ---\n' + _log[-3000:])
    try:
        urllib.request.urlopen('http://127.0.0.1:1234/v1/models', timeout=5).read(); break
    except Exception:
        if time.time() - t0 > 1500:
            raise TimeoutError(open(f'{WORK}/vllm-server.log').read()[-4000:])
        time.sleep(5)
print(f'vLLM ready after {time.time()-t0:.0f}s')
req = urllib.request.Request('http://127.0.0.1:1234/v1/chat/completions', data=json.dumps({'model': 'local-model', 'max_tokens': 400,
      'messages': [{'role': 'user', 'content': 'Say hello in five words.'}], **{'chat_template_kwargs': {'enable_thinking': False}}}).encode(),
      headers={'Content-Type': 'application/json'})
try:
    _m = json.loads(urllib.request.urlopen(req, timeout=300).read())['choices'][0]['message']
    print('smoke:', repr((_m.get('content') or '')[:200]), '| reasoning:', repr((_m.get('reasoning_content') or _m.get('reasoning') or '')[:120]))
except Exception as _e:   # the smoke chat is informational; the game run below is the real test
    print('smoke chat failed:', repr(_e)[:300])


In [ ]:
import math, os, sys, time, urllib.request, logging
from pathlib import Path
sys.path.insert(0, '/kaggle/working')
logging.basicConfig(level=logging.WARNING)
from arcnav.runner import run, make_arcade, DEFAULT_CONFIG
cfg = dict(DEFAULT_CONFIG, model='local-model', base_url='http://127.0.0.1:1234/v1', verbose=True,
           extra_body={'chat_template_kwargs': {'enable_thinking': False}}, max_tokens=4096, **{'think_first_turns': 3})
out_dir = Path('/kaggle/working/arcnav-results')
if RERUN:
    os.environ['ARC_API_KEY'] = 'test-key-123'
    deadline = time.time() + 60
    while True:  # wait for the gateway sidecar
        try:
            urllib.request.urlopen('http://gateway:8001/api/games', timeout=10).read(); break
        except Exception as e:
            if time.time() > deadline:
                raise RuntimeError(f'gateway not ready: {e!r}')
            time.sleep(5)
    arc = make_arcade(competition=True, base_url='http://gateway:8001/')
    games = [e.game_id for e in arc.get_environments()]
    waves = max(1, math.ceil(len(games) / 12))
    total_left = 470 - (time.time() - T0) / 60
    cfg.update(jobs=12, max_minutes=max(20, min(150, total_left / waves)), deadline=T0 + 470 * 60)
    print(f'rerun: {len(games)} games, {cfg["jobs"]} concurrent, {cfg["max_minutes"]:.0f} min/game, waves={waves}')
    run(games, cfg, out_dir=out_dir, tag='kaggle-rerun', arc=arc)   # the gateway records actions and emits submission.parquet
else:
    arc = make_arcade(environments_dir='/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files')
    cfg.update(jobs=2, max_minutes=15)
    run(['ls20', 'vc33'], cfg, out_dir=out_dir, tag='kaggle-smoke', arc=arc)
    import pandas as pd   # commit mode: dummy submission so the commit succeeds
    pd.DataFrame(data=[['1_0', '1', True, 1]], columns=['row_id', 'game_id', 'end_of_game', 'score']).to_parquet('/kaggle/working/submission.parquet', index=False)
try:
    VLLM.terminate()
except Exception:
    pass
print(f'done in {(time.time()-T0)/60:.1f} min')
